In [1]:
# ── Cài đặt thư viện cần thiết ────────────────────────────────────────────────
# tree-sitter / tree-sitter-python : parse cây cú pháp (AST) cho Python
# gitpython                        : checkout commit của repo
# rank_bm25                        : tìm kiếm lexical (BM25)
# sentence-transformers            : sinh embedding (vector ngữ nghĩa)
# unidiff                          : đọc patch (unified diff) để lấy đáp án vàng
# datasets                         : tải benchmark SWE-bench
# orjson                           : đọc graph JSON nhanh hơn json chuẩn nhiều lần
!pip install tree-sitter tree-sitter-python gitpython igraph tqdm rank_bm25 sentence-transformers unidiff datasets tiktoken orjson -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.1/668.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.5/231.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 2.8 MB/s eta 0:00:00


## 1. Constants & Function Definitions
All constants and function/class definitions. Run this section once — no I/O, no side effects.

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║ CẤU HÌNH CHẠY — đổi các hằng số ở đây rồi Run All để chạy lại             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os

# ── Thư mục làm việc: repo, graph, kết quả và file của judge đều nằm ở đây ─────────
# Kaggle chỉ hiển thị và lưu (khi Save Version) thư mục /kaggle/working; Colab dùng /content.
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
# Repo clone ra ngoài /kaggle/working (không được lưu, không tốn dung lượng output); graph và kết quả vẫn ở WORK_DIR
REPO_DIR = "/tmp/repos" if WORK_DIR == "/kaggle/working" else WORK_DIR
os.makedirs(REPO_DIR, exist_ok=True)

# ── Repositories ─────────────────────────────────────────────────────────────
# REPOS: các repo GitHub đưa vào thí nghiệm | repo_path_of(): nơi clone (trong REPO_DIR)
# Số instance trong SWE-bench Lite: flask 3, pytest 17, astropy 6, sphinx 16,
# scikit-learn 23, matplotlib 23, django 114 → tổng 202. Django có doc viết tay phong phú (file .txt,
# directive .. class:: / .. method::), khác kiểu autodoc của các repo còn lại; build lâu nhất.
# Còn có thể thêm: pydata/xarray 5, pylint-dev/pylint 6, psf/requests 6, mwaskom/seaborn 4.
# Lần chạy này so 3 kiểu doc: Django (viết tay), sphinx (pha trộn), sympy (autodoc) → 114 + 16 + 77 = 207.
REPOS = [
    # "pallets/flask",
    # "pytest-dev/pytest",
    # "astropy/astropy",
    # "scikit-learn/scikit-learn",
    # "matplotlib/matplotlib",
    "django/django",
    "sphinx-doc/sphinx",
    "sympy/sympy",
]
def repo_path_of(repo):
    return f"{REPO_DIR}/{repo.split('/')[1]}"

# Số instance tối đa mỗi repo; lấy mẫu cố định theo sha1(instance_id) → mỗi lần chạy ra cùng một tập.
# None = lấy hết; repo không có trong MAX_PER_REPO lấy tối đa MAX_PER_REPO_DEFAULT.
MAX_PER_REPO         = {"django/django": None, "sphinx-doc/sphinx": None, "sympy/sympy": None}
MAX_PER_REPO_DEFAULT = 10

# ── Section 5 (SWE-bench đầy đủ) ──────────────────────────────────────────────────────
# Section 5: SWE-bench đầy đủ, 1 graph cho mỗi khoảng SNAPSHOT_BIN_DAYS ngày trong mỗi version
# Toàn bộ 12 repo của SWE-bench đầy đủ: 2.294 instance (kết quả lượt trước: scale_rows_v2.jsonl, nhãn judge ở judge/)
SCALE_REPOS_ALL   = ["django/django", "sympy/sympy", "scikit-learn/scikit-learn", "sphinx-doc/sphinx",
                     "matplotlib/matplotlib", "pytest-dev/pytest", "pydata/xarray", "astropy/astropy",
                     "pylint-dev/pylint", "psf/requests", "mwaskom/seaborn", "pallets/flask"]
# 12 repo của tập test là một thực nghiệm, không chia đợt hay chia tập (quyết định 58 của research_context.md)
SCALE_REPOS       = SCALE_REPOS_ALL
SNAPSHOT_BIN_DAYS = 60

# Tương thích ngược cho code dùng 1 repo (mặc định = repo đầu danh sách)
REPO      = REPOS[0]
REPO_PATH = repo_path_of(REPO)

# ── Embedding model ───────────────────────────────────────────────────────────
# Model embedding nhỏ, nhanh; dùng để vector hóa node code và doc-chunk
_EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"

# Cross-encoder reranker: chấm trực tiếp cặp (query, node) — thay RRF ở tầng
# xếp hạng hop (graph = sinh ứng viên, CE = xếp hạng)
_CE_MODEL_ID = "BAAI/bge-reranker-base"

# ── SWE-bench instance selector ───────────────────────────────────────────────
# Thay đổi INSTANCE_IDX rồi Run All để chạy lại cho instance khác.
# None → dùng HEAD hiện tại (không checkout).
INSTANCE_IDX = 0

# ── Embedding: tốc độ build graph ─────────────────────────────────────────────
# EMBED_FP16: trên GPU, embed bằng float16 (nhanh ~2 lần). Embedding lệch rất nhỏ so với
# float32, nên thứ hạng có thể đổi nhẹ. Đặt False để tái lập embedding float32 như trước.
EMBED_FP16  = True
EMBED_BATCH = 128   # batch lớn hơn tận dụng GPU tốt hơn; giảm về 64 nếu thiếu bộ nhớ

# Số graph giữ index trong RAM cùng lúc. Mỗi graph lớn chiếm vài trăm MB khi đã dựng index;
# không giới hạn thì chạy hết các instance sẽ làm Colab hết RAM.
INDEX_CACHE_MAX = 3

# In lại cấu hình để kiểm tra nhanh trước khi chạy phần sau
print(f"WORK_DIR       : {WORK_DIR}")
print(f"REPOS          : {', '.join(REPOS)}")
print(f"MAX_PER_REPO   : {MAX_PER_REPO} | mặc định: {MAX_PER_REPO_DEFAULT}")
print(f"EMBED_MODEL    : {_EMBED_MODEL_ID}")
print(f"CE_RERANKER    : {_CE_MODEL_ID}")
print(f"INSTANCE_IDX   : {INSTANCE_IDX}")
print(f"EMBED_FP16     : {EMBED_FP16} | EMBED_BATCH: {EMBED_BATCH}")
print(f"Section 5      : {', '.join(SCALE_REPOS)}, snapshot {SNAPSHOT_BIN_DAYS} ngày")

WORK_DIR       : /kaggle/working
REPOS          : django/django, sphinx-doc/sphinx, sympy/sympy
MAX_PER_REPO   : {'django/django': None, 'sphinx-doc/sphinx': None, 'sympy/sympy': None} | mặc định: 10
EMBED_MODEL    : BAAI/bge-small-en-v1.5
CE_RERANKER    : BAAI/bge-reranker-base
INSTANCE_IDX   : 0
EMBED_FP16     : True | EMBED_BATCH: 128
Section 5      : django/django, sympy/sympy, scikit-learn/scikit-learn, sphinx-doc/sphinx, matplotlib/matplotlib, pytest-dev/pytest, pydata/xarray, astropy/astropy, pylint-dev/pylint, psf/requests, mwaskom/seaborn, pallets/flask, snapshot 60 ngày


### 1.1 Source File Discovery
List Python files using `git ls-files`, respecting `.gitignore`.

In [3]:
import subprocess

# ── Bộ lọc: bỏ test/example/docs và file cấu hình → graph chỉ chứa code "thật" ──
_SKIP_DIRS = {"tests", "test", "examples", "example", "benchmarks", "docs"}
_SKIP_FILES = {"conftest.py", "setup.py"}

def get_python_files(repo_path: str) -> list[str]:
    """List tracked .py files, excluding test/example directories and config files."""
    # Ưu tiên hỏi git (nhanh, đúng tập file được quản lý). Lỗi thì fallback rglob.
    try:
        out = subprocess.check_output(
            ["git", "ls-files", "--cached", "--others", "--exclude-standard"],
            cwd=repo_path, text=True, stderr=subprocess.DEVNULL,
        )
        paths = [l for l in out.splitlines() if l.endswith(".py")]
    except Exception:
        # Fallback: quét tay toàn bộ .py, bỏ thư mục .git
        paths = [
            str(p.relative_to(repo_path)).replace("\\", "/")
            for p in Path(repo_path).rglob("*.py")
            if ".git" not in p.parts
        ]

    # ── Áp bộ lọc thư mục/file lên danh sách thu được ──────────────────────────
    result = []
    for p in paths:
        parts = p.replace("\\", "/").split("/")
        # skip if any directory component is in _SKIP_DIRS
        if any(part in _SKIP_DIRS for part in parts[:-1]):
            continue
        # skip specific root-level filenames
        if parts[-1] in _SKIP_FILES:
            continue
        result.append(p)
    return result


### 1.2 Graph Schema & Data Model
Defines `ParseContext`, `SchemaPlugin`, and `SimpleSchema` — the data model for Modules, Classes, Functions and their relationships (Defines / Calls / Imports / Inherits).

In [4]:
from __future__ import annotations
import os
from abc import ABC, abstractmethod
from dataclasses import dataclass, field, replace as dc_replace
from typing import Callable


# ── ParseContext: "túi" trạng thái khi parse 1 file ──────────────────────────
# Giữ id module/scope cha, metadata commit/file, và các list output:
# nodes_out/edges_out (kết quả chắc chắn) + *_refs_out (ref chờ resolve cross-file).
@dataclass
class ParseContext:
    module_id:    str
    parent_id:    str
    commit:       str
    file_:        str
    directory:    str
    is_module_scope: bool
    current_class_id: str | None
    nodes_out:    list = field(default_factory=list)
    edges_out:    list = field(default_factory=list)
    call_refs_out:     list = field(default_factory=list)
    inherit_refs_out:  list = field(default_factory=list)
    instance_attr_types_out: list = field(default_factory=list)
    text_of: Callable = field(default=None, repr=False)

    @property
    def in_class(self) -> bool:
        return self.current_class_id is not None

    # Gom 1 node vào output (label + id + properties)
    def emit_node(self, label, nid, props):
        self.nodes_out.append({"labels": [label], "id": nid, "properties": props})

    # Gom 1 cạnh vào output (type + source→target, kèm field phụ tùy ý)
    def emit_edge(self, etype, src, tgt, **extra):
        entry = {"type": etype, "source": src, "target": tgt}
        entry.update(extra)
        self.edges_out.append(entry)


# ── SchemaPlugin: interface trừu tượng cho schema graph ──────────────────────
# Cho phép thay schema khác (định nghĩa node/edge nào) mà không sửa AST engine.
class SchemaPlugin(ABC):
    @abstractmethod
    def on_class(self, node, ctx): ...
    @abstractmethod
    def on_function(self, node, ctx): ...
    def on_import(self, target_module_id, ctx): pass
    @property
    @abstractmethod
    def call_edge_type(self): ...
    @property
    @abstractmethod
    def inherit_edge_type(self): ...


# ── SimpleSchema: schema thực dùng trong notebook ─────────────────────────────
# Node: Module / Class / Function | Edge: Defines / Calls / Imports / Inherits
class SimpleSchema(SchemaPlugin):
    """Module / Class / Function  +  Defines / Calls / Imports / Inherits"""

    @property
    def call_edge_type(self): return "Calls"
    @property
    def inherit_edge_type(self): return "Inherits"

    # Bắt gặp 1 định nghĩa class → tạo node Class + cạnh Defines (cha → class)
    def on_class(self, node, ctx):
        name = ctx.text_of(node.child_by_field_name("name"))
        if not name:
            return None
        cid = f"{ctx.parent_id}:{name}"   # parent_id, not module_id → correct nesting
        ctx.emit_node("Class", cid, {
            "name":       name,
            "commit":     ctx.commit,
            "file":       ctx.file_,
            "directory":  ctx.directory,
            "start_line": node.start_point[0] + 1,
            "end_line":   node.end_point[0] + 1,
            "source":     ctx.text_of(node)[:8000],   # cắt source để JSON không phình
        })
        ctx.emit_edge("Defines", ctx.parent_id, cid)
        return cid

    # Bắt gặp 1 định nghĩa function/method → tạo node Function + cạnh Defines
    def on_function(self, node, ctx):
        name = ctx.text_of(node.child_by_field_name("name"))
        if not name:
            return None
        fid = f"{ctx.parent_id}:{name}"
        ctx.emit_node("Function", fid, {
            "name":       name,
            "commit":     ctx.commit,
            "file":       ctx.file_,
            "directory":  ctx.directory,
            "start_line": node.start_point[0] + 1,
            "end_line":   node.end_point[0] + 1,
            "source":     ctx.text_of(node)[:8000],
        })
        ctx.emit_edge("Defines", ctx.parent_id, fid)
        return fid

    # Bắt gặp import đã resolve được → tạo cạnh Imports (module hiện tại → module đích)
    def on_import(self, target_module_id, ctx):
        ctx.emit_edge("Imports", ctx.module_id, target_module_id)


SCHEMA = SimpleSchema()


### 1.3 Language Extractor
Configures `LanguageExtractor` for tree-sitter-python — node type mappings and field names for Python AST traversal.

In [5]:
from dataclasses import dataclass as _dc
from tree_sitter import Language, Parser
import tree_sitter_python as tspython

# ── Khởi tạo parser tree-sitter cho Python ───────────────────────────────────
# try/except để tương thích cả API tree-sitter mới (>=0.22) lẫn cũ
try:
    _PY_LANG = Language(tspython.language())
    _parser  = Parser(_PY_LANG)
except TypeError:
    _PY_LANG = Language(tspython.language(), "python")
    _parser  = Parser()
    _parser.set_language(_PY_LANG)


# ── LanguageExtractor: "bảng tra" tên node-type/field của grammar Python ──────
# Trừu tượng hóa toàn bộ chi tiết grammar vào 1 chỗ → muốn hỗ trợ ngôn ngữ khác
# chỉ cần tạo extractor mới, không sửa ASTEngine.
@_dc(frozen=True)
class LanguageExtractor:
    parser: object
    language: object
    builtins: frozenset
    function_types: tuple
    class_types: tuple
    call_type: str
    import_type: str
    from_import_type: str
    assignment_types: tuple
    import_name_types: tuple
    aliased_import_type: str
    import_as_names_type: str
    wildcard_import_type: str
    import_keyword_type: str
    identifier_type: str
    name_field: str
    params_field: str
    superclasses_field: str
    module_name_field: str
    alias_name_field: str
    alias_alias_field: str
    call_function_field: str
    attribute_type: str
    attribute_object_field: str
    attribute_attr_field: str
    attr_name_subfield: str
    self_keywords: frozenset
    source_extensions: tuple = (".py",)
    index_file_name: str = "__init__"


# ── Cấu hình cụ thể cho Python ───────────────────────────────────────────────
EXTRACTOR = LanguageExtractor(
    parser=_parser,
    language=_PY_LANG,
    # builtins: bỏ qua khi tạo cạnh Calls (print/len/super... không phải hàm của repo)
    builtins=frozenset({
        "print", "len", "range", "str", "int", "float", "bool", "list", "dict",
        "tuple", "set", "type", "isinstance", "issubclass", "hasattr", "getattr",
        "setattr", "delattr", "super", "object", "enumerate", "zip", "map",
        "filter", "sorted", "reversed", "min", "max", "sum", "abs", "round",
        "open", "repr", "iter", "next", "any", "all", "vars", "dir", "id",
        "hash", "callable", "staticmethod", "classmethod", "property",
    }),
    # In tree-sitter-python 0.25+, `async def` is function_definition (no separate node type)
    function_types=("function_definition",),
    class_types=("class_definition",),
    call_type="call",
    import_type="import_statement",
    from_import_type="import_from_statement",
    assignment_types=("assignment", "annotated_assignment", "expression_statement"),
    import_name_types=("dotted_name", "aliased_import"),
    aliased_import_type="aliased_import",
    import_as_names_type="import_as_names",
    wildcard_import_type="wildcard_import",
    import_keyword_type="import",
    identifier_type="identifier",
    name_field="name",
    params_field="parameters",
    superclasses_field="superclasses",
    module_name_field="module_name",
    alias_name_field="name",
    alias_alias_field="alias",
    call_function_field="function",
    attribute_type="attribute",
    attribute_object_field="object",
    attribute_attr_field="attribute",
    attr_name_subfield="",
    # self_keywords: nhận diện gọi method trên chính object (self.x()/cls.x())
    self_keywords=frozenset({"self", "cls"}),
)


### 1.4 Doc Extractor
`DocExtractor` parses `.rst` / `.md` documentation files — creates `Document` + `DocChunk` nodes with `HasChunk` edges, and collects pending `Documents` links (via `.. autofunction::` / `:func:` / `:meth:` / `:class:`) and `References` links (via `:doc:`) for resolution after code parsing.

In [6]:
import re
from pathlib import Path

# ── Tham số bóc tách doc ──────────────────────────────────────────────────────
MAX_CHUNK_CHARS = 1600   # ~400 tokens × 4 chars
_RST_ADORN = set("=-~^*+#<>")          # ký tự gạch chân heading RST (===, ---, ~~~)

_DOC_DIR_CANDIDATES = ("docs", "doc", "documentation", "Documentation")
_DOC_SKIP = {"_build", "_static", "_templates", "_themes", "generated", "auto_examples"}


# ── Liệt kê file tài liệu (.rst/.md/.txt), bỏ thư mục build/generated ────────
def get_doc_files(repo_path):
    repo = Path(repo_path)
    # Lấy thư mục doc đầu tiên tồn tại trong danh sách ứng viên
    docs_dir = next(
        (repo / d for d in _DOC_DIR_CANDIDATES if (repo / d).is_dir()),
        None,
    )
    if docs_dir is None:
        return []
    out = []
    for ext in ("*.rst", "*.md", "*.txt"):      # Django viết doc RST trong file .txt
        for p in sorted(docs_dir.rglob(ext)):
            if any(part in _DOC_SKIP for part in p.parts):
                continue
            if p.suffix == ".txt" and p.name.lower().startswith(("requirements", "robots")):
                continue                            # file .txt không phải doc
            out.append(str(p.relative_to(repo)).replace("\\", "/"))
    return out


# ── DocExtractor: 1 file doc → node Document + nhiều DocChunk + link chờ resolve
class DocExtractor:
    # Regex bắt các directive RST trỏ tới code/doc khác:
    _ROLE_RE = re.compile(r":(?:func|meth|class|attr|exc|obj):`([^`]+)`")   # :func:`x`
    # .. autoclass:: x và directive Python domain viết tay .. class:: x / .. py:method:: x (vd Django).
    # Tên dừng ở "(" → bỏ chữ ký: ".. class:: Model(**kwargs)" → "Model"
    _DIR_RE  = re.compile(r"\.\.\s+(?:py:)?(?:auto)?(function|class|method|exception|attribute|decorator)::\s+([~\w.]+)")
    _DOC_RE  = re.compile(r":doc:`([^`]+)`")                                # :doc:`x` (doc→doc)
    _MOD_RE  = re.compile(r"\.\.\s+(?:current)?module::\s+(\S+)")           # .. module:: (ngữ cảnh module)

    def __init__(self, rel_path, repo_path):
        self.rel_path = rel_path
        self.abs_path = Path(repo_path) / rel_path

    # Đọc file → chia chunk theo heading → bắt link → trả nodes/edges + link chờ resolve
    def extract(self):
        try:
            text = self.abs_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return None

        lines  = text.splitlines()
        doc_id = f"doc:{self.rel_path}"
        ext    = Path(self.rel_path).suffix.lstrip(".")

        # Node Document đại diện cho cả file
        doc_node = {
            "id": doc_id, "label": "Document",
            "file": self.rel_path, "type": ext,
            "title": self._first_heading(lines) or Path(self.rel_path).stem,
        }

        nodes, edges = [doc_node], []
        pending_links, pending_refs = [], []   # link doc→code / doc→doc, resolve ở bước sau
        current_module = None                  # ngữ cảnh .. module:: hiện hành
        current_class  = None                  # class gần nhất (cho .. method:: không có tiền tố class)

        # Duyệt từng (heading, body) → tạo node DocChunk + cạnh HasChunk + bắt link
        for i, (heading, body) in enumerate(self._split(lines)):
            cid = f"chunk:{self.rel_path}:{i}"
            nodes.append({
                "id": cid, "label": "DocChunk",
                "file": self.rel_path, "heading": heading,
                "index": i, "text": body, "embedding": None,
            })
            edges.append({"type": "HasChunk", "source": doc_id, "target": cid})

            combined = f"{heading}\n{body}"
            # Cập nhật module ngữ cảnh nếu gặp directive .. module::
            for m in self._MOD_RE.finditer(combined):
                v = m.group(1)
                current_module = None if v == "None" else v
                current_class = None

            # Gom các link để resolve sau (kèm module ngữ cảnh để định danh đầy đủ)
            # Directive định nghĩa đối tượng; method/attribute không có tiền tố class → gán class gần nhất
            for m in self._DIR_RE.finditer(combined):
                kind, name = m.group(1), m.group(2)
                if kind == "class":
                    current_class = name.lstrip("~.").split(".")[-1]
                elif kind in ("method", "attribute") and "." not in name and current_class:
                    name = f"{current_class}.{name}"
                pending_links.append((cid, name, current_module))
            for m in self._ROLE_RE.finditer(combined):
                pending_links.append((cid, m.group(1), current_module))
            for m in self._DOC_RE.finditer(combined):
                pending_refs.append((cid, m.group(1)))

        return {
            "doc_id": doc_id, "nodes": nodes, "edges": edges,
            "pending_links": pending_links, "pending_refs": pending_refs,
        }

    # Tìm heading đầu tiên của file (dòng có gạch chân RST ngay dưới) làm title
    def _first_heading(self, lines):
        for i in range(1, len(lines)):
            ul = lines[i].strip()
            if ul and all(c == ul[0] for c in ul) and ul[0] in _RST_ADORN:
                prev = lines[i - 1].strip()
                if prev and len(ul) >= len(prev):
                    return prev
        return None

    # Chia file thành các (heading, body) theo ranh giới heading RST
    def _split(self, lines):
        boundaries = []
        for i in range(1, len(lines)):
            ul = lines[i].strip()
            if ul and all(c == ul[0] for c in ul) and ul[0] in _RST_ADORN:
                prev = lines[i - 1].strip()
                if prev and len(ul) >= len(prev):
                    boundaries.append((i - 1, prev))

        # Không có heading nào → cả file là 1 chunk
        if not boundaries:
            body = "\n".join(lines).strip()
            return [("", body)] if body else []

        # Mỗi heading → body từ sau gạch chân tới heading kế tiếp
        chunks = []
        for idx, (pos, heading) in enumerate(boundaries):
            start = pos + 2
            end   = boundaries[idx + 1][0] if idx + 1 < len(boundaries) else len(lines)
            body  = "\n".join(lines[start:end]).strip()
            if not body:
                continue
            for chunk in self._maybe_split(heading, body):
                chunks.append(chunk)
        return chunks

    # Chunk dài quá MAX_CHUNK_CHARS → cắt nhỏ theo đoạn (giữ nguyên heading)
    def _maybe_split(self, heading, body):
        if len(body) <= MAX_CHUNK_CHARS:
            return [(heading, body)]
        parts, cur, cur_len = [], [], 0
        for para in re.split(r"\n\n+", body):
            plen = len(para)
            if cur and cur_len + plen > MAX_CHUNK_CHARS:
                parts.append((heading, "\n\n".join(cur)))
                cur, cur_len = [para], plen
            else:
                cur.append(para)
                cur_len += plen
        if cur:
            parts.append((heading, "\n\n".join(cur)))
        return parts or [(heading, body)]


# ── Bảng tên → các node code có tên đó, để resolve link RST ─────────────────────────────────
# Mỗi khóa giữ MỌI node khớp (tên trùng nhau không bị ghi đè); resolve_doc_links quyết định khi nhiều node khớp.
from collections import defaultdict

def _file_module(file):
    """Đường dẫn file → tên module dạng chấm, bỏ tiền tố src. và đuôi .__init__ (docs không viết src.)."""
    m = file.replace("/", ".").removesuffix(".py").removesuffix(".__init__")
    return m[4:] if m.startswith("src.") else m

def build_name_index(code_nodes):
    """Build name key -> {node_id} for RST role resolution (multiple keys per node, multiple nodes per key)."""
    idx = defaultdict(set)
    for n in code_nodes:
        if n["label"] not in ("Function", "Class"):
            continue
        nid  = n.get("id", "")
        name = n.get("name", "")
        file = n.get("file", "")
        if not name:
            continue
        # Docs không document test — chặn test file khỏi index kẻo tên bị "cướp"
        # (vd pytest: Config resolve vào testing/test_assertion.py thay vì src)
        _parts = file.split("/")
        if any(p in ("test", "tests", "testing") for p in _parts) or _parts[-1].startswith("test_"):
            continue
        # Đăng ký nhiều khóa: module.name, pkg.name (dạng re-export: flask.url_for), name thuần, Class.name, pkg.Class.name
        file_mod = _file_module(file)
        keys = {f"{file_mod}.{name}", f"{file_mod.split('.')[0]}.{name}", name}
        try:
            parts = nid.split(f":{file}:", 1)[1].split(":")
            if len(parts) >= 2:
                keys |= {f"{parts[-2]}.{name}", f"{file_mod.split('.')[0]}.{parts[-2]}.{name}"}
        except (IndexError, ValueError):
            pass
        for k in keys:
            idx[k].add(nid)
    return idx


# ── Resolve link chờ → cạnh Documents (doc→code) và References (doc→doc) ──────
def resolve_doc_links(doc_results, name_index):
    """Resolve pending RST links -> Documents + References edges (deduped).
    Documents: thử lần lượt tên kèm module ngữ cảnh (.. module::), tên như doc viết, Class.method, tên cuối.
    Khóa đầu tiên có ứng viên quyết định: 1 node → nối; nhiều node → chỉ giữ node nằm trong module ngữ cảnh;
    vẫn mơ hồ → không nối (không chọn bừa một node trùng tên)."""
    doc_ids = {r["doc_id"] for r in doc_results}
    edges, seen = [], set()

    for r in doc_results:
        # pending_links → cạnh Documents (module ngữ cảnh trước; nhiều node khớp thì lọc theo module, mơ hồ thì bỏ)
        for cid, raw, cur_mod in r["pending_links"]:
            name = raw.strip()
            m = re.search(r"<([^<>]+)>$", name)          # role có tiêu đề riêng: `text <target>`
            if m:
                name = m.group(1)
            name = name.lstrip("~.").removesuffix("()")
            tries = ([f"{cur_mod}.{name}"] if cur_mod else []) + [
                name, ".".join(name.split(".")[-2:]), name.split(".")[-1]]   # Class.method, tên cuối
            target = None
            for k in dict.fromkeys(tries):
                cand = name_index.get(k)
                if not cand:
                    continue
                if len(cand) == 1:
                    target = next(iter(cand))
                elif cur_mod:
                    narrow = [t for t in cand if _file_module(t.split(":", 2)[1]).startswith(cur_mod)]
                    target = narrow[0] if len(narrow) == 1 else None
                break
            if target:
                key = ("Documents", cid, target)
                if key not in seen:
                    edges.append({"type": "Documents", "source": cid, "target": target})
                    seen.add(key)

        # pending_refs → cạnh References (thử đuôi .rst/.txt/.md, kèm và không kèm tiền tố thư mục doc)
        for cid, ref in r["pending_refs"]:
            m = re.search(r"<([^<>]+)>$", ref.strip())   # :doc:`text <target>`
            ref = (m.group(1) if m else ref).strip().lstrip("/")
            base = re.sub(r"\.(rst|txt|md)$", "", ref)
            tgt_candidates = []
            for e in (".rst", ".txt", ".md"):
                tgt_candidates.append(f"doc:{base}{e}")
                for doc_dir in _DOC_DIR_CANDIDATES:
                    if not base.startswith(f"{doc_dir}/"):
                        tgt_candidates.append(f"doc:{doc_dir}/{base}{e}")
            for tgt in tgt_candidates:
                if tgt in doc_ids:
                    key = ("References", cid, tgt)
                    if key not in seen:
                        edges.append({"type": "References", "source": cid, "target": tgt})
                        seen.add(key)
                    break

    return edges

### 1.5 AST Engine
`ASTEngine` parses one `.py` file in 4 passes (definitions → imports → calls → self-assigns) and maps imports for cross-file resolution.

In [7]:
import logging
import os
from dataclasses import replace as dc_replace
from tree_sitter import Query, QueryCursor

logger = logging.getLogger(__name__)


# ── Helper: chạy 1 Query và trả {tên_capture: [Node]} (chuẩn hóa API tree-sitter)
def _caps(query, node):
    """Execute a Query and return {capture_name: [Node]} (tree-sitter 0.24+ API)."""
    result = QueryCursor(query).captures(node)
    if isinstance(result, dict):
        return result
    # Pre-0.24 fallback: list of (Node, str) tuples
    d = {}
    for n, name in result:
        d.setdefault(name, []).append(n)
    return d


class ASTEngine:
    """Parse một file .py → nodes + refs cho cross-file resolve."""

    # Khởi tạo: tính id module, chuẩn bị sẵn 4 Query (defs/imports/calls/assigns)
    def __init__(self, file_path, repo_path, base_commit, schema, extractor):
        self.extractor   = extractor
        self.file_path   = file_path
        self.repo_path   = repo_path
        self.base_commit = base_commit
        self.schema      = schema
        try:
            self.rel_path = os.path.relpath(file_path, repo_path).replace("\\", "/")
        except ValueError:
            self.rel_path = file_path.replace("\\", "/")
        self.module_id   = f"{self.base_commit}:{self.rel_path}"
        self._source: bytes = b""
        self.import_map: dict[str, str] = {}            # alias → module/symbol id
        self.from_import_map: dict[str, str | None] = {} # tên local → "module_id:tên_gốc"

        ext  = extractor
        lang = extractor.language
        self._q_defs = Query(lang,
            "\n".join(f"({t}) @def" for t in (*ext.class_types, *ext.function_types))
        )
        self._q_imports = Query(lang,
            f"({ext.import_type}) @stmt\n({ext.from_import_type}) @stmt"
        )
        self._q_calls   = Query(lang, f"({ext.call_type}) @call")
        self._q_assigns = Query(lang, "(assignment) @assign")

    # ── Public ─────────────────────────────────────────────────────────────────

    # Điểm vào: đọc file → parse → chạy 4 pass → trả dict kết quả cho resolve sau
    def parse(self):
        try:
            with open(self.file_path, "rb") as f:
                self._source = f.read()
        except Exception:
            return None
        try:
            tree = self.extractor.parser.parse(self._source)
        except Exception:
            return None

        commit, _, file_ = self.module_id.partition(":")
        ctx = ParseContext(
            module_id=self.module_id, parent_id=self.module_id,
            commit=commit, file_=file_,
            directory=os.path.dirname(file_) or ".",
            is_module_scope=True, current_class_id=None,
            text_of=self._text,
        )
        ctx.emit_node("Module", self.module_id, {"name": self.rel_path, "commit": self.base_commit})

        # 4 pass bóc tách trên cùng cây cú pháp
        self._extract_definitions(tree, ctx)
        self._extract_imports(tree, ctx)
        self._extract_calls(tree, ctx)
        self._extract_self_assigns(tree, ctx)

        # local_scope/class_scopes phục vụ resolve call cross-file
        local_scope, class_scopes = self._build_scopes(ctx.nodes_out)
        return {
            "module_id":    self.module_id,
            "nodes":        ctx.nodes_out,
            "definite_edges":      ctx.edges_out,
            "call_refs":           ctx.call_refs_out,
            "inherit_refs":        ctx.inherit_refs_out,
            "instance_attr_types": ctx.instance_attr_types_out,
            "import_map":          self.import_map,
            "from_import_map":     self.from_import_map,
            "local_scope":         local_scope,
            "class_scopes":        class_scopes,
        }

    # ── Scope helpers ──────────────────────────────────────────────────────────

    # Leo node.parent → id scope CHỨA node (vd "commit:file:Class:method")
    def _scope_of(self, node):
        """ID of the scope that CONTAINS node (walks up node.parent chain)."""
        ext, parts, cur = self.extractor, [], node.parent
        while cur is not None:
            if cur.type in (*ext.class_types, *ext.function_types):
                n = cur.child_by_field_name(ext.name_field)
                if n:
                    parts.append(self._text(n))
            cur = cur.parent
        return ":".join([self.module_id] + list(reversed(parts))) if parts else self.module_id

    # Leo node.parent → id class bao quanh gần nhất (hoặc None)
    def _class_of(self, node):
        """ID of the innermost class enclosing node, or None."""
        ext, cur = self.extractor, node.parent
        while cur is not None:
            if cur.type in ext.class_types:
                n = cur.child_by_field_name(ext.name_field)
                if n:
                    return f"{self._scope_of(cur)}:{self._text(n)}"
            cur = cur.parent
        return None

    # ── Pass 1: class + function definitions ──────────────────────────────────

    # Tìm mọi class/function (sắp theo vị trí) → tạo node; với class còn lấy superclass
    def _extract_definitions(self, tree, ctx):
        ext = self.extractor
        def_nodes = sorted(_caps(self._q_defs, tree.root_node).get("def", []),
                           key=lambda n: n.start_byte)
        for node in def_nodes:
            tmp = dc_replace(ctx, parent_id=self._scope_of(node),
                             current_class_id=self._class_of(node))
            if node.type in ext.class_types:
                cid = self.schema.on_class(node, tmp)
                if cid:
                    # Bóc danh sách lớp cha → inherit_refs (chờ resolve thành cạnh Inherits)
                    supers = node.child_by_field_name(ext.superclasses_field)
                    if supers:
                        for arg in supers.children:
                            base = self._extract_base_name(arg)
                            if base:
                                ctx.inherit_refs_out.append({"class_id": cid, "base_name": base})
            else:
                self.schema.on_function(node, tmp)

    # ── Pass 2: imports ────────────────────────────────────────────────────────

    # Xử lý `import x` và `from a import b` → build import_map/from_import_map + cạnh Imports
    def _extract_imports(self, tree, ctx):
        ext = self.extractor
        for node in _caps(self._q_imports, tree.root_node).get("stmt", []):
            if node.type == ext.import_type:
                # import a.b as c → tách module + alias, resolve sang file thật
                for sub in node.children:
                    if sub.type in ext.import_name_types:
                        raw = self._text(sub)
                        mod_str, _, alias = raw.partition(" as ")
                        mod_str = mod_str.strip()
                        alias   = alias.strip() if alias.strip() else mod_str.split(".")[-1]
                        tid = self._resolve_module(mod_str)
                        if tid:
                            self.schema.on_import(tid, ctx)
                            self.import_map[alias] = tid
                            if "." in mod_str:
                                self.import_map[mod_str] = tid
            elif node.type == ext.from_import_type:
                # from mod import a, b → cố resolve từng symbol; không được thì ghi nhớ để resolve sau
                mod_node = node.child_by_field_name(ext.module_name_field)
                if not mod_node:
                    continue
                mod_str    = self._text(mod_node)
                module_tid = self._resolve_module(mod_str)
                if module_tid:
                    self.schema.on_import(module_tid, ctx)
                for local_name, orig_name in self._get_from_imports(node):
                    sub_tid = self._resolve_module(f"{mod_str}.{orig_name}")
                    if sub_tid:
                        self.import_map[local_name] = sub_tid
                    elif module_tid:
                        self.from_import_map[local_name] = f"{module_tid}:{orig_name}"
                    else:
                        self.from_import_map[local_name] = None

    # Lấy danh sách (tên_local, tên_gốc) trong mệnh đề `from ... import ...`
    def _get_from_imports(self, from_stmt):
        ext, results, past_import = self.extractor, [], False
        for child in from_stmt.children:
            if child.type == ext.import_keyword_type:
                past_import = True; continue
            if not past_import:
                continue
            if child.type == ext.wildcard_import_type:   # from x import *  → bỏ qua
                break
            if child.type == ext.identifier_type:
                n = self._text(child); results.append((n, n))
            elif child.type == ext.import_as_names_type:
                for sub in child.children:
                    if sub.type == ext.identifier_type:
                        n = self._text(sub); results.append((n, n))
                    elif sub.type == ext.aliased_import_type:
                        r = self._resolve_aliased(sub)
                        if r: results.append(r)
            elif child.type == ext.aliased_import_type:
                r = self._resolve_aliased(child)
                if r: results.append(r)
        return results

    # `orig as alias` → (alias, orig)
    def _resolve_aliased(self, node):
        ext   = self.extractor
        orig  = node.child_by_field_name(ext.alias_name_field)
        alias = node.child_by_field_name(ext.alias_alias_field)
        if orig:
            o = self._text(orig)
            return (self._text(alias) if alias else o, o)
        return None

    # ── Pass 3: call references ────────────────────────────────────────────────

    # Phân loại mọi lời gọi thành kind (simple/attr/self_method/self_attr_method) → call_refs
    def _extract_calls(self, tree, ctx):
        ext = self.extractor
        for call_node in _caps(self._q_calls, tree.root_node).get("call", []):
            func = call_node.child_by_field_name(ext.call_function_field)
            if func is None:
                continue
            base = {
                "caller_id": self._scope_of(call_node),   # ai gọi
                "line":      call_node.start_point[0] + 1,
                "file":      self.rel_path,
            }
            if func.type == ext.identifier_type:
                # foo()  → simple (bỏ qua builtins)
                name = self._text(func)
                if name not in ext.builtins:
                    ctx.call_refs_out.append({**base, "kind": "simple", "name": name})

            elif func.type == ext.attribute_type:
                # obj.attr() → phân biệt self.x() / self.attr.x() / obj.x()
                obj  = func.child_by_field_name(ext.attribute_object_field)
                attr = func.child_by_field_name(ext.attribute_attr_field)
                if not (obj and attr):
                    continue
                attr_name = self._text(attr)
                if attr_name in ext.builtins:
                    continue
                obj_text = self._text(obj)
                if obj_text in ext.self_keywords:
                    # self.method() → resolve trong class hiện tại
                    ctx.call_refs_out.append({
                        **base, "kind": "self_method", "name": attr_name,
                        "class_id": self._class_of(call_node),
                    })
                elif obj.type == ext.attribute_type:
                    inner_obj  = obj.child_by_field_name(ext.attribute_object_field)
                    inner_attr = obj.child_by_field_name(ext.attribute_attr_field)
                    if inner_obj and inner_attr and self._text(inner_obj) in ext.self_keywords:
                        # self.attr.method() → cần biết kiểu của self.attr (xem Pass 4)
                        ctx.call_refs_out.append({
                            **base, "kind": "self_attr_method", "name": attr_name,
                            "attr": self._text(inner_attr),
                            "class_id": self._class_of(call_node),
                        })
                    else:
                        ctx.call_refs_out.append({**base, "kind": "attr", "name": attr_name, "obj": obj_text})
                else:
                    # obj.method() → resolve qua import_map của obj
                    ctx.call_refs_out.append({**base, "kind": "attr", "name": attr_name, "obj": obj_text})

    # ── Pass 4: self.attr = Type() assignments ────────────────────────────────

    # Bắt `self.attr = Type()` → ghi kiểu của thuộc tính, để resolve self.attr.method()
    def _extract_self_assigns(self, tree, ctx):
        ext = self.extractor
        for node in _caps(self._q_assigns, tree.root_node).get("assign", []):
            left  = node.child_by_field_name("left")
            right = node.child_by_field_name("right")
            if not (left and right and left.type == ext.attribute_type and right.type == ext.call_type):
                continue
            obj_node  = left.child_by_field_name(ext.attribute_object_field)
            attr_node = left.child_by_field_name(ext.attribute_attr_field)
            if not (obj_node and attr_node):
                continue
            if self._text(obj_node) not in ext.self_keywords:
                continue
            # Lấy tên kiểu ở vế phải (Type() hoặc mod.Type())
            func = right.child_by_field_name(ext.call_function_field)
            if not func:
                continue
            if func.type == ext.identifier_type:
                type_name = self._text(func)
            elif func.type == ext.attribute_type:
                a = func.child_by_field_name(ext.attribute_attr_field)
                type_name = self._text(a) if a else None
            else:
                continue
            if not type_name or type_name in ext.builtins:
                continue
            class_id = self._class_of(obj_node)
            if class_id:
                ctx.instance_attr_types_out.append({
                    "class_id": class_id,
                    "attr":     self._text(attr_node),
                    "type_name": type_name,
                })

    # ── Module path resolver ───────────────────────────────────────────────────

    # Tên module (kể cả relative `..pkg`) → đường dẫn file thật trong repo, hoặc None
    def _resolve_module(self, module):
        if module.startswith("."):
            # relative import: số dấu chấm = số cấp đi lên
            dots     = len(module) - len(module.lstrip("."))
            rel_part = module.lstrip(".")
            base_dir = os.path.dirname(self.rel_path)
            for _ in range(dots - 1):
                base_dir = os.path.dirname(base_dir)
            parts = (
                os.path.join(base_dir, rel_part.replace(".", os.sep)).replace("\\", "/")
                if rel_part else base_dir
            )
        else:
            parts = module.replace(".", "/")
        # Thử cả file.py lẫn package/__init__.py
        init_name = self.extractor.index_file_name
        for src_ext in self.extractor.source_extensions:
            for candidate in (f"{parts}{src_ext}", f"{parts}/{init_name}{src_ext}"):
                if os.path.exists(os.path.join(self.repo_path, candidate)):
                    return f"{self.base_commit}:{os.path.relpath(os.path.join(self.repo_path, candidate), self.repo_path).replace(chr(92), '/')}"
        return None

    # ── Scope builder ──────────────────────────────────────────────────────────

    # Từ node của file → 2 bảng: local_scope (tên top-level) & class_scopes (method theo class)
    def _build_scopes(self, nodes):
        local_scope, class_scopes = {}, {}
        for node in nodes:
            label   = node["labels"][0]
            if label == "Module":
                continue
            node_id = node["id"]
            name    = node["properties"].get("name", "")
            parts   = node_id[len(self.module_id):].lstrip(":").split(":")
            if len(parts) == 1:
                local_scope[name] = node_id
            elif len(parts) == 2 and label == "Function":
                class_id = f"{self.module_id}:{parts[0]}"
                class_scopes.setdefault(class_id, {})[name] = node_id
        return local_scope, class_scopes

    # ── Helpers ────────────────────────────────────────────────────────────────

    # Lấy text gốc của 1 node từ byte buffer
    def _text(self, node):
        return self._source[node.start_byte:node.end_byte].decode("utf-8", errors="replace")

    # Lấy tên lớp cơ sở từ node superclass (xử lý identifier / attribute / subscript)
    def _extract_base_name(self, node):
        ext = self.extractor
        if node.type == ext.identifier_type:
            return self._text(node)
        if node.type == ext.attribute_type:
            a = node.child_by_field_name(ext.attribute_attr_field)
            return self._text(a) if a else ""
        if node.type in ("subscript", "type"):
            child = node.child_by_field_name("value") or (node.children[0] if node.child_count else None)
            return self._extract_base_name(child) if child else ""
        return ""

    # ── Cross-file resolution — returns ONLY semantic (Calls, Inherits) edges ──

    # Sau khi parse hết file: gộp scope/import toàn repo → resolve thành cạnh Calls/Inherits
    @staticmethod
    def resolve_cross_file(all_results, schema):
        from collections import defaultdict
        # Tập id node toàn cục + các bảng tra theo module
        all_node_ids = set()
        all_node_ids.update(n["id"] for r in all_results for n in r["nodes"])
        fmap   = {r["module_id"]: r["from_import_map"] for r in all_results}
        lscope = {r["module_id"]: r["local_scope"]     for r in all_results}
        cscope = {r["module_id"]: r["class_scopes"]    for r in all_results}
        imap   = {r["module_id"]: r["import_map"]      for r in all_results}
        iat    = {r["module_id"]: r["instance_attr_types"] for r in all_results}

        # Build global method map: lowercase_method_name -> list of candidate node IDs
        global_method_map = defaultdict(list)
        for r in all_results:
            for node in r["nodes"]:
                if node["labels"][0] in ("Function", "Class"):
                    name = node["properties"].get("name", "")
                    if name:
                        global_method_map[name.lower()].append(node["id"])

        semantic_edges = []   # Calls + Inherits only

        # Inherit refs → cạnh Inherits (thử local scope → import → from-import)
        for r in all_results:
            mid   = r["module_id"]
            ls, im, fim = lscope.get(mid, {}), imap.get(mid, {}), fmap.get(mid, {})
            for ref in r["inherit_refs"]:
                cid, base = ref["class_id"], ref["base_name"]
                target = (
                    ls.get(base)
                    or im.get(base)
                    or (fim.get(base) if fim.get(base) in all_node_ids else None)
                )
                if target and target in all_node_ids:
                    semantic_edges.append({"type": schema.inherit_edge_type, "source": cid, "target": target})

        # Call refs → cạnh Calls (mỗi ref gọi _resolve_call; target có thể là list)
        for r in all_results:
            mid  = r["module_id"]
            ls   = lscope.get(mid, {})
            im   = imap.get(mid, {})
            fim  = fmap.get(mid, {})
            cs   = cscope.get(mid, {})
            iat_ = {e["class_id"]: e for e in iat.get(mid, [])}

            for ref in r["call_refs"]:
                target = ASTEngine._resolve_call(
                    ref, all_node_ids, ls, im, fim, cs, iat_, cs, lscope, imap, global_method_map
                )
                if target:
                    if isinstance(target, list):
                        for t in target:
                            semantic_edges.append({
                                "type": schema.call_edge_type,
                                "source": ref["caller_id"],
                                "target": t,
                                "line": ref.get("line"),
                            })
                    else:
                        semantic_edges.append({
                            "type": schema.call_edge_type,
                            "source": ref["caller_id"],
                            "target": target,
                            "line": ref.get("line"),
                        })
        return semantic_edges

    # Resolve 1 lời gọi → id node đích, theo từng kind; cuối cùng có fallback heuristic
    @staticmethod
    def _resolve_call(ref, all_ids, ls, im, fim, cs, iat, cscope, lscope, imap, global_method_map=None):
        kind = ref.get("kind", "simple")
        name = ref.get("name", "")

        if kind == "simple":
            # foo() → local scope / import / from-import
            resolved = ASTEngine._find_first(name, all_ids, ls, im, fim)
            if resolved:
                return resolved

        if kind == "self_method":
            # self.foo() → method cùng class
            class_id = ref.get("class_id")
            if class_id:
                target = cs.get(class_id, {}).get(name)
                if target and target in all_ids:
                    return target

        if kind == "self_attr_method":
            # self.attr.foo() → tra kiểu của self.attr (Pass 4) rồi tìm method trong class kiểu đó
            class_id = ref.get("class_id")
            attr     = ref.get("attr", "")
            if class_id and attr in iat:
                type_name    = iat[attr]["type_name"]
                target_class = ASTEngine._find_first(type_name, all_ids, ls, im, fim)
                if target_class:
                    t = cscope.get(target_class, {}).get(name)
                    if t and t in all_ids:
                        return t

        if kind == "attr":
            # obj.foo() → obj là module đã import → tìm trong scope module đó
            obj = ref.get("obj", "")
            module_tid = im.get(obj) or fim.get(obj)
            if module_tid:
                remote_ls = lscope.get(module_tid, {})
                t = remote_ls.get(name)
                if t and t in all_ids:
                    return t
                candidate = imap.get(module_tid, {}).get(name)
                if candidate and candidate in all_ids:
                    return candidate

        # Heuristic-based fallback: tên method hiếm (≤3 ứng viên, không phổ biến) → nối luôn
        if global_method_map and name:
            common_methods = {
                "get", "set", "run", "update", "delete", "clear", "close", "save", "load",
                "__init__", "__str__", "__repr__", "__call__", "setup", "teardown", "execute",
                "pop", "append", "extend", "insert", "remove", "add", "format", "keys", "values", "items"
            }
            if name not in common_methods:
                candidates = global_method_map.get(name.lower(), [])
                if 0 < len(candidates) <= 3:
                    valid_candidates = [c for c in candidates if c in all_ids]
                    if valid_candidates:
                        return valid_candidates

        return None

    # Tra tên lần lượt qua (local scope, import_map, from_import_map) → id hợp lệ đầu tiên
    @staticmethod
    def _find_first(name, all_ids, ls, im, fim):
        for source in (ls, im, fim):
            t = source.get(name)
            if t and t in all_ids:
                return t
        return None


### 1.6 Build & Search Pipeline
Three functions that form the core pipeline:
- `build_graph(commit)` — checkout, parse, embed (BGE-small), export JSON (disk-cached)
- `load_graph_indices(path)` — load JSON, build BM25 + vector + call-graph indices (memory-cached)
- `get_oracle(patch, nodes)` — map changed lines of the gold patch to the innermost Function/Class (ground truth)

In [8]:
import json
import numpy as np
from collections import deque, defaultdict
from pathlib import Path
from datetime import datetime, timezone
from git import Repo as _Repo
from rank_bm25 import BM25Okapi
import hashlib

# Đọc/ghi graph JSON: orjson nhanh hơn json chuẩn nhiều lần với file lớn; thiếu orjson thì dùng json.
# Không dùng json.dump khi ghi: json.dump chạy bộ mã hóa Python thuần, chậm với hàng triệu số embedding.
# Ghi vào <file>.tmp rồi mới thay file cũ, nên lần ghi bị dừng giữa chừng không để lại file cụt.
try:
    import orjson
    def read_graph_json(path):
        with open(path, "rb") as f:
            return orjson.loads(f.read())
    def write_graph_json(path, obj):
        with open(f"{path}.tmp", "wb") as f:
            f.write(orjson.dumps(obj))
        Path(f"{path}.tmp").replace(path)
except ImportError:
    def read_graph_json(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    def write_graph_json(path, obj):
        with open(f"{path}.tmp", "w", encoding="utf-8") as f:
            f.write(json.dumps(obj, ensure_ascii=False))   # json.dumps dùng bộ mã hóa C
        Path(f"{path}.tmp").replace(path)

# ── Cache trong phiên: model / index đã build / embedding theo nội dung ───────
_MODEL_CACHE = {}
_INDEX_CACHE = {}
_EMB_CACHE   = {}

# Load model embedding 1 lần duy nhất cho mỗi model_id
def _get_model(model_id=_EMBED_MODEL_ID):
    if model_id not in _MODEL_CACHE:
        import torch
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(model_id)
        if EMBED_FP16 and torch.cuda.is_available():
            m = m.half()                                  # float16 trên GPU: nhanh ~2 lần
        if not torch.cuda.is_available():
            print("⚠ Không có GPU: bước embed khi build graph sẽ chậm hơn nhiều (đổi runtime sang GPU)")
        _MODEL_CACHE[model_id] = m
    return _MODEL_CACHE[model_id]

# Load cross-encoder reranker 1 lần (tự chọn GPU nếu có; max 512 token/cặp)
def _get_ce(model_id=_CE_MODEL_ID):
    if model_id not in _MODEL_CACHE:
        from sentence_transformers import CrossEncoder
        _MODEL_CACHE[model_id] = CrossEncoder(model_id, max_length=512)
    return _MODEL_CACHE[model_id]

# Text dùng để embed 1 node code: 2000 ký tự đầu của source (fallback: label+name)
def _sig_doc_fn(node):
    src = node.get("source", "")
    return src[:2000] if src else f"{node.get('label','')} {node.get('name','')}"

# Embed có cache theo sha1(text): source giống nhau giữa các commit chỉ encode 1 lần
def encode_cached(texts, model, batch_size=64):
    """Embed texts with an in-session cache keyed by sha1(text).
    Source giống hệt giữa các commit chỉ encode 1 lần (~80-90% reuse trên cùng repo)."""
    if not texts:
        return np.zeros((0, model.get_sentence_embedding_dimension()), dtype=np.float32)
    keys = [hashlib.sha1(t.encode("utf-8")).hexdigest() for t in texts]
    # Tách phần đã có trong cache (out) và phần thiếu (miss) cần encode
    out, miss_i, miss_t = [None] * len(texts), [], []
    for i, k in enumerate(keys):
        v = _EMB_CACHE.get(k)
        if v is None:
            miss_i.append(i); miss_t.append(texts[i])
        else:
            out[i] = v
    # Chỉ encode phần thiếu rồi nạp ngược vào cache + vị trí tương ứng
    if miss_t:
        vecs = model.encode(miss_t, batch_size=batch_size,
                            normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
        for j, i in enumerate(miss_i):
            _EMB_CACHE[keys[i]] = vecs[j]
            out[i] = vecs[j]
    return np.vstack(out)

In [9]:
# ── build_graph(): orchestrator — build full graph cho 1 commit ──────────────
# Luồng 7 bước: cache → checkout → parse AST → resolve → flatten → doc → embed → export
import re
import time
from tqdm.auto import tqdm

def build_graph(commit, repo_path=REPO_PATH, output_dir=WORK_DIR, force=False,
                embed_model_id=_EMBED_MODEL_ID):
    """
    Checkout commit → parse AST + docs → embed → export JSON.
    Always builds the full graph (code nodes + Document/DocChunk nodes).
    """
    repo_name   = Path(repo_path).name
    short       = commit[:8]
    output_file = f"{output_dir}/{repo_name}_graph_{short}.json"

    # 0. Disk cache: đã có JSON đúng schema "simple+doc2" thì trả luôn, khỏi build lại
    #    (simple+doc2: lớp doc đọc thêm .txt và directive Python domain viết tay)
    if not force and Path(output_file).exists():
        try:
            # "meta" là khóa đầu tiên của JSON → chỉ đọc 4 KB đầu, không parse cả file
            with open(output_file, encoding="utf-8") as _f:
                _m = re.search(r'"schema":\s*"([^"]*)"', _f.read(4096))
            _schema = _m.group(1) if _m else ""
            if _schema == "simple+doc2":
                print(f"[{short}] disk cache hit → {output_file}", flush=True)
                return output_file
            print(f"[{short}] cached schema={_schema!r} — rebuilding with doc layer", flush=True)
        except Exception:
            pass

    # 1. Checkout
    _t0 = time.perf_counter()
    _repo = _Repo(repo_path)
    _repo.git.checkout(commit, force=True)   # force: /kaggle/working không giữ quyền file, git coi là thay đổi
    _t1 = time.perf_counter()

    # 2. Source files
    src_files = get_python_files(repo_path)

    # 3. AST parse — File-level Cache (Bỏ qua Tree-sitter cho 99% file không đổi)
    global _FILE_AST_CACHE
    if "_FILE_AST_CACHE" not in globals():
        _FILE_AST_CACHE = {}
    import hashlib

    results, n_cache, n_parse, n_fail = [], 0, 0, 0
    for rel in tqdm(src_files, desc=f"[{short}] AST", unit="file", leave=False):
        try:
            fpath = Path(repo_path) / rel
            content = fpath.read_bytes()
            fhash = (rel, hashlib.sha1(content).hexdigest())   # khóa có đường dẫn: file trùng nội dung ở chỗ khác không nhận nhầm đường dẫn

            if fhash in _FILE_AST_CACHE:
                old_short, cached_r = _FILE_AST_CACHE[fhash]
                if old_short == short:
                    results.append(cached_r)
                else:
                    # Đổi tên commit trên ID node bằng cách serialize siêu tốc
                    # (không import json tại đây — import trong hàm biến 'json'
                    #  thành local toàn hàm → UnboundLocalError ở chỗ dùng khác)
                    r_str = json.dumps(cached_r)
                    r_str = r_str.replace(f"{old_short}:", f"{short}:").replace(f'"{old_short}"', f'"{short}"')
                    results.append(json.loads(r_str))
                n_cache += 1
                continue

            eng = ASTEngine(str(fpath), repo_path, short, SCHEMA, EXTRACTOR)
            r = eng.parse()
            if r:
                _FILE_AST_CACHE[fhash] = (short, r)
                results.append(r)
                n_parse += 1
        except Exception:
            n_fail += 1
    # failed > 0 → file bị bỏ khỏi graph → oracle/recall thiếu mà không có dấu vết
    print(f"[{short}] AST: {len(results)}/{len(src_files)} files "
          f"(cache={n_cache} parsed={n_parse} failed={n_fail})", flush=True)

    # 4. Resolve cross-file edges (sem = Calls/Inherits, def = Defines/Imports)
    sem_edges = ASTEngine.resolve_cross_file(results, SCHEMA)
    def_edges = [e for r in results for e in r["definite_edges"]]

    # 5. Flatten code nodes / edges (dedupe node theo id)
    all_nodes = list({
        node["id"]: {
            "id":         node["id"],
            "label":      node["labels"][0],
            "name":       node["properties"].get("name", ""),
            "file":       node["properties"].get("file", ""),
            "start_line": node["properties"].get("start_line"),
            "end_line":   node["properties"].get("end_line"),
            "source":     node["properties"].get("source", ""),
            "community":  node["properties"].get("community"),
        }
        for r in results for node in r["nodes"]
    }.values())
    all_edges = [
        {"type": e["type"], "source": e["source"], "target": e["target"], "line": e.get("line")}
        for e in (def_edges + sem_edges)
    ]

    _t2 = time.perf_counter()

    # 5b. Doc parse → thêm node Document/DocChunk + File-level Cache
    global _FILE_DOC_CACHE
    if "_FILE_DOC_CACHE" not in globals():
        _FILE_DOC_CACHE = {}

    doc_files = get_doc_files(repo_path)
    doc_results = []
    for rel in tqdm(doc_files, desc=f"[{short}] docs", unit="file", leave=False):
        try:
            fpath = Path(repo_path) / rel
            content = fpath.read_bytes()
            fhash = (rel, hashlib.sha1(content).hexdigest())   # khóa có đường dẫn: file trùng nội dung ở chỗ khác không nhận nhầm đường dẫn
            if fhash in _FILE_DOC_CACHE:
                doc_results.append(_FILE_DOC_CACHE[fhash])
                continue

            r = DocExtractor(rel, repo_path).extract()
            if r:
                _FILE_DOC_CACHE[fhash] = r
                doc_results.append(r)
        except Exception:
            pass
    doc_nodes   = [n for r in doc_results for n in r["nodes"]]
    doc_edges   = [e for r in doc_results for e in r["edges"]]
    cross_edges = resolve_doc_links(doc_results, build_name_index(all_nodes))
    n_has_chunk = sum(1 for e in doc_edges   if e["type"] == "HasChunk")
    n_documents = sum(1 for e in cross_edges if e["type"] == "Documents")
    n_refs      = sum(1 for e in cross_edges if e["type"] == "References")
    n_pending   = sum(len(r["pending_links"]) for r in doc_results)
    # % resolved = sức khỏe cầu doc→code (đo trực tiếp hiệu quả build_name_index)
    print(f"[{short}] docs: {len(doc_nodes)} nodes | HasChunk={n_has_chunk}  "
          f"Documents={n_documents}/{n_pending} ({n_documents/max(1, n_pending):.0%} resolved)  "
          f"References={n_refs}", flush=True)
    all_nodes += doc_nodes
    all_edges += doc_edges + cross_edges

    _t3 = time.perf_counter()

    # 6. Embeddings — Function/Class: sig+docstring; DocChunk: heading+text
    _model    = _get_model(embed_model_id)
    vec_cands = [n for n in all_nodes if n["label"] in ("Function", "Class", "DocChunk")]
    embed_texts = [
        _sig_doc_fn(n) if n["label"] in ("Function", "Class")
        else (f"{n['heading']}\n{n['text']}" if n.get('heading') else n.get('text', ""))
        for n in vec_cands
    ]
    vecs = encode_cached(embed_texts, _model, batch_size=EMBED_BATCH)  # in-session cache
    # Làm tròn 5 chữ số: JSON nhỏ đi nhiều → ghi/đọc nhanh hơn; sai lệch cosine ~1e-5
    vecs_r = np.round(vecs.astype(np.float64), 5)   # float64: tolist() ra số ngắn (0.01235)
    id2emb = {vec_cands[i]["id"]: vecs_r[i].tolist() for i in range(len(vec_cands))}
    for node in all_nodes:
        node["embedding"] = id2emb.get(node["id"])

    _t4 = time.perf_counter()

    # 7. Export ra JSON (meta + nodes + edges) + dọn index cache cũ
    try:
        remote_url = _repo.remotes.origin.url
    except Exception:
        remote_url = repo_path

    graph = {
        "meta": {
            "repo": remote_url, "commit": short, "schema": "simple+doc2", "lang": "python",
            "doc_links": "v2",                   # cạnh Documents nối theo cách mới (xem resolve_doc_links)
            "built_at": datetime.now(timezone.utc).isoformat(),
            "node_count": len(all_nodes), "edge_count": len(all_edges),
            "embed_model": embed_model_id, "embed_dim": int(vecs.shape[1]),
        },
        "nodes": all_nodes, "edges": all_edges,
    }
    write_graph_json(output_file, graph)
    write_graph_summary(output_file, all_nodes, all_edges)   # file nhỏ cho oracle/thống kê

    _INDEX_CACHE.pop(output_file, None)

    size_mb = Path(output_file).stat().st_size / 1024 / 1024
    print(f"[{short}] → {output_file}  ({len(all_nodes)} nodes, {len(all_edges)} edges, {size_mb:.1f} MB) | "
          f"checkout={_t1-_t0:.1f}s parse={_t2-_t1:.1f}s doc={_t3-_t2:.1f}s "
          f"embed={_t4-_t3:.1f}s export={time.perf_counter()-_t4:.1f}s", flush=True)
    return output_file

In [10]:
import re
import time
from collections import Counter
from tqdm.auto import tqdm
_CAMEL_RE = re.compile(r'([A-Z])')
_ALPHANUM_RE = re.compile(r'[a-zA-Z0-9]+')
def _tokenize_code(txt):
    # Dùng regex đã pre-compile để tăng tốc tối đa
    return _ALPHANUM_RE.findall(_CAMEL_RE.sub(r' \1', txt).lower())

# ── _embed_query(): làm sạch query trước khi encode ──────────────────────────
# bge-small cắt input ở 512 token; issue dài kèm code block chiếm hết cửa sổ →
# bỏ code block, giữ ~2000 chars đầu (≈512 token) phần mô tả tự nhiên.
_MD_CODEBLOCK_RE = re.compile(r"```.*?```", re.S)
def _embed_query(q):
    return _MD_CODEBLOCK_RE.sub(" ", q)[:2000]

# ── File tóm tắt cạnh mỗi graph: node rút gọn (không embedding, không source) + thống kê ──
# Oracle và cell thống kê chỉ cần vài trường của node → đọc file nhỏ này thay vì parse cả graph JSON.
_LITE_KEYS = ("id", "label", "name", "file", "start_line", "end_line")

def graph_summary_path(output_file):
    return output_file.removesuffix(".json") + ".summary.json"

def write_graph_summary(output_file, nodes, edges):
    fl = [n["end_line"] - n["start_line"] + 1 for n in nodes
          if n["label"] == "Function" and n.get("start_line") and n.get("end_line")]
    summary = {
        "nodes": [{k: n[k] for k in _LITE_KEYS if k in n} for n in nodes],
        "stats": {"nodes": len(nodes), "edges": len(edges),
                  "labels": dict(Counter(n["label"] for n in nodes)),
                  "edge_types": dict(Counter(e["type"] for e in edges)),
                  "avg_func_loc": round(sum(fl) / len(fl), 1) if fl else 0},
    }
    write_graph_json(graph_summary_path(output_file), summary)

def load_graph_summary(output_file):
    """Đọc file tóm tắt; graph build trước khi có tính năng này thì tạo file tóm tắt 1 lần."""
    p = graph_summary_path(output_file)
    if not Path(p).exists():
        g = read_graph_json(output_file)
        write_graph_summary(output_file, g["nodes"], g["edges"])
    return read_graph_json(p)

# ── load_graph_nodes(): loader nhẹ — chỉ nodes rút gọn + map id, không build index ───
# Dùng để trích oracle mà không tốn chi phí parse cả graph và dựng index tìm kiếm.
def load_graph_nodes(output_file):
    """Lightweight loader — summary nodes + node_by_id only, no index built."""
    nodes = load_graph_summary(output_file)["nodes"]
    return {"nodes": nodes, "node_by_id": {n["id"]: n for n in nodes}}


# ── load_graph_indices(): nạp graph + dựng toàn bộ index truy hồi ────────────
def load_graph_indices(output_file):
    """
    Load graph JSON, build retrieval indices.

    Two search indices:
      search()      — BM25 + vector on AST nodes only (Function/Class/Module)
      search_full() — BM25 + vector on AST nodes + DocChunk nodes

    Two coverage traversals (pure BFS, no decay, no rerank, return ALL reachable nodes):
      hop_coverage_ast()  — seeds via search()      + BFS via AST edges only
      hop_coverage_full() — coverage AST ∪ node có doc_votes() (doc-as-signal)
    """
    # Memory cache: đã dựng index cho file này thì trả luôn (im lặng — bị gọi lặp)
    if output_file in _INDEX_CACHE:
        return _INDEX_CACHE[output_file]

    _stem = Path(output_file).stem
    _t0 = time.perf_counter()
    gdata = read_graph_json(output_file)
    _t_load = time.perf_counter()

    nodes          = gdata["nodes"]
    embed_model_id = gdata["meta"].get("embed_model", _EMBED_MODEL_ID)
    node_by_id     = {n["id"]: n for n in nodes}
    _model         = _get_model(embed_model_id)

    # ── AST-only search index ──────────────────────────────────────────────────
    # Corpus BM25 chỉ trên node code; thiếu source thì dùng label+name+file
    bm25_nodes = [n for n in nodes if n["label"] not in ("Document", "DocChunk")]
    corpus_ast = []
    for n in tqdm(bm25_nodes, desc=f"[{_stem}] tokenize AST", unit="node", leave=False):
        text = n.get("source", "")
        if not text or len(text.strip()) < 5:
            text = f"{n.get('label','')} {n.get('name','')} {n.get('file','')}"
        corpus_ast.append(_tokenize_code(text))
    _t_tok_ast = time.perf_counter()
    bm25_ast = BM25Okapi(corpus_ast)
    _t_bm25_ast = time.perf_counter()

    # Ma trận vector + map id→hàng cho cả index BM25 và vector
    vec_nodes  = [n for n in bm25_nodes if n.get("embedding") is not None]
    vec_matrix = np.array([n["embedding"] for n in vec_nodes])
    vi_of      = {nd["id"]: i for i, nd in enumerate(vec_nodes)}
    bi_of      = {nd["id"]: i for i, nd in enumerate(bm25_nodes)}
    _t_vec = time.perf_counter()

    # ── Full search index (AST nodes + DocChunk) ───────────────────────────────
    # Thay vì tách từ lại toàn bộ AST (rất chậm), ta tận dụng lại corpus_ast
    doc_nodes = [n for n in nodes if n["label"] == "DocChunk"]
    corpus_doc = []
    for n in tqdm(doc_nodes, desc=f"[{_stem}] tokenize doc", unit="chunk", leave=False):
        text = f"{n.get('heading', '')} {n.get('text', '')}".strip()
        if not text or len(text.strip()) < 5:
            text = f"{n.get('label','')} {n.get('name','')}"
        corpus_doc.append(_tokenize_code(text))

    full_nodes = bm25_nodes + doc_nodes
    corpus_full = corpus_ast + corpus_doc
    _t_tok_doc = time.perf_counter()
    bm25_full = BM25Okapi(corpus_full)
    _t_bm25_full = time.perf_counter()

    full_vec_nodes  = [n for n in full_nodes if n.get("embedding") is not None]
    full_vec_matrix = np.array([n["embedding"] for n in full_vec_nodes])
    fvi_of          = {nd["id"]: i for i, nd in enumerate(full_vec_nodes)}
    fbi_of          = {nd["id"]: i for i, nd in enumerate(full_nodes)}
    _t_fvec = time.perf_counter()

    # ── Adjacency lists (danh sách kề có trọng số) ─────────────────────────────
    _AST_TYPES     = {"Calls", "Inherits", "Defines", "Imports"}
    _BRIDGE_LABELS = {"DocChunk"}
    # Trọng số theo loại cạnh: ngữ nghĩa code cao, cấu trúc/hub thấp, cầu doc trung bình.
    # _hop_coverage bỏ qua trọng số (giữ tương thích); _hop_ranked dùng để cho điểm.
    _EDGE_W = {"Calls": 1.0, "Inherits": 0.9, "Defines": 0.5, "Imports": 0.4,
               "Documents": 0.8, "HasChunk": 0.5, "References": 0.5}
    # fwd = xuôi (source→target), bwd = ngược — BFS chỉ đi cạnh code (AST);
    # cạnh Documents gom vào doc2code làm bảng chỉ đường chunk → code node.
    fwd_ast,  bwd_ast  = defaultdict(list), defaultdict(list)
    doc2code = defaultdict(list)
    for edge in gdata["edges"]:
        w = _EDGE_W.get(edge["type"], 0.5)
        if edge["type"] in _AST_TYPES:
            fwd_ast[edge["source"]].append((edge["target"], w))
            bwd_ast[edge["target"]].append((edge["source"], w))
        elif edge["type"] == "Documents":
            doc2code[edge["source"]].append(edge["target"])
    _t_adj = time.perf_counter()

    # Tên hiển thị của node (Function/Class dùng name, DocChunk dùng heading)
    def _node_name(nd):
        return nd.get("name") or nd.get("heading", "")

    # Lấy chỉ số top-k điểm cao nhất (argpartition cho nhanh khi k nhỏ)
    def _topn(scores, k):
        k = min(k, len(scores))
        if k == len(scores): return np.argsort(scores)[::-1]
        idx = np.argpartition(scores, -k)[-k:]
        return idx[np.argsort(scores[idx])[::-1]]

    # ── _rrf(): Reciprocal Rank Fusion — gộp 2 bảng xếp hạng BM25 + Vector ──────
    # Node có mặt ở CẢ HAI list được cộng dồn hai số hạng → ưu tiên node vừa khớp
    # lexical (BM25) vừa liên quan ngữ nghĩa (Vector). Bất biến theo thang điểm.
    RRF_C = 60
    def _rrf(bm_order, bm_id, v_order, v_id, keep_label=None):
        bm_rank = {bm_id(i): r for r, i in enumerate(bm_order)}
        v_rank  = {v_id(i):  r for r, i in enumerate(v_order)}
        cand = set(bm_rank) | set(v_rank)
        if keep_label is not None:
            cand = {nid for nid in cand if node_by_id.get(nid, {}).get("label") == keep_label}
        scored = []
        for nid in cand:
            s = 0.0
            if nid in bm_rank: s += 1.0 / (RRF_C + bm_rank[nid])
            if nid in v_rank:  s += 1.0 / (RRF_C + v_rank[nid])
            scored.append((nid, s))
        scored.sort(key=lambda x: (-x[1], x[0]))   # hòa điểm xếp theo id: kết quả không phụ thuộc thứ tự duyệt set
        return scored

    # ── search(): hybrid trên node code (BM25 + vector) ────────────────────────
    def search(query, n=5, method="hybrid", pool=50):
        """Hybrid search on AST nodes only."""
        q_emb, bm_sc, _ = _q_sig(query)          # encode + BM25 chỉ 1 lần / query (memo)
        v_sim = vec_matrix @ q_emb

        # method="bm25"/"vector": chỉ 1 tín hiệu
        if method == "bm25":
            return [
                {"id": bm25_nodes[i]["id"], "name": _node_name(bm25_nodes[i]),
                 "label": bm25_nodes[i]["label"], "score": round(float(bm_sc[i]), 4)}
                for i in _topn(bm_sc, n) if bm_sc[i] > 0
            ]
        if method == "vector":
            return [
                {"id": vec_nodes[i]["id"], "name": _node_name(vec_nodes[i]),
                 "label": vec_nodes[i]["label"], "score": round(float(v_sim[i]), 4)}
                for i in _topn(v_sim, n)
            ]

        # hybrid: gộp 2 bảng xếp hạng BM25 + Vector bằng RRF (ưu tiên node vào cả hai)
        k = min(pool, len(bm25_nodes))
        bm25_top_idx = _topn(bm_sc, k)
        vec_top_idx  = _topn(v_sim, min(k, len(vec_nodes)))
        ranked = _rrf(bm25_top_idx, lambda i: bm25_nodes[i]["id"],
                      vec_top_idx,  lambda i: vec_nodes[i]["id"])
        return [
            {"id": nid, "name": _node_name(node_by_id[nid]),
             "label": node_by_id[nid]["label"], "score": round(sc, 4)}
            for nid, sc in ranked[:n] if nid in node_by_id
        ]

    # ── search_full(): hybrid trên node code + DocChunk ────────────────────────
    def search_full(query, n=5):
        """Hybrid search on AST nodes + DocChunk nodes."""
        q_emb, _, bm_sc = _q_sig(query)          # dùng chung memo với search()/search_doc()
        v_sim = full_vec_matrix @ q_emb

        k = min(50, len(full_nodes))
        bm25_top_idx = _topn(bm_sc, k)
        vec_top_idx  = _topn(v_sim, min(k, len(full_vec_nodes)))
        ranked = _rrf(bm25_top_idx, lambda i: full_nodes[i]["id"],
                      vec_top_idx,  lambda i: full_vec_nodes[i]["id"])
        return [
            {"id": nid, "name": _node_name(node_by_id[nid]),
             "label": node_by_id[nid]["label"], "score": round(sc, 4)}
            for nid, sc in ranked[:n] if nid in node_by_id
        ]

    # ── search_doc(): xếp hạng TRỰC TIẾP trong tập DocChunk có cầu Documents ──
    # Chunk cạnh tranh với nhau (không với code) — hết nghẽn top-50 làm đói seed.
    _chunk_ids     = [n["id"] for n in doc_nodes if n["id"] in doc2code]
    _all_chunk_ids = [n["id"] for n in doc_nodes]
    _chunk_bi  = {n["id"]: len(bm25_nodes) + i for i, n in enumerate(doc_nodes)}

    def _search_chunks(query, ids, n):
        """Hybrid RRF (BM25 + vector) xếp hạng nội bộ trong tập chunk ids. Mỗi kết quả kèm điểm gốc của hai tín hiệu:
        bm25 = điểm BM25, cos = cosine giữa embedding của query và của chunk (0 nếu chunk không có embedding)."""
        if not ids:
            return []
        q_emb, _, bm_full = _q_sig(query)
        bm = [float(bm_full[_chunk_bi[cid]]) for cid in ids]
        vs = [float(full_vec_matrix[fvi_of[cid]] @ q_emb) if cid in fvi_of else 0.0
              for cid in ids]
        idxs = range(len(ids))
        r_bm = {i: r for r, i in enumerate(sorted(idxs, key=lambda i: -bm[i]))}
        r_vs = {i: r for r, i in enumerate(sorted(idxs, key=lambda i: -vs[i]))}
        scored = sorted(
            ((i, cid, 1.0 / (RRF_C + r_bm[i]) + 1.0 / (RRF_C + r_vs[i]))
             for i, cid in enumerate(ids)),
            key=lambda x: -x[2])
        return [{"id": cid, "name": _node_name(node_by_id[cid]),
                 "label": "DocChunk", "score": round(sc, 4), "bm25": bm[i], "cos": vs[i]}
                for i, cid, sc in scored[:n]]

    def search_doc(query, n=5):
        """Hybrid RRF xếp hạng nội bộ các DocChunk có cạnh Documents."""
        return _search_chunks(query, _chunk_ids, n)

    def search_doc_all(query, n=10):
        """Như search_doc nhưng trên MỌI DocChunk, kể cả chunk không có cạnh Documents."""
        return _search_chunks(query, _all_chunk_ids, n)

    # ── doc_votes(): chunk liên quan BỎ PHIẾU cho code node nó document ───────
    # Doc evidence vào RANKING (bảng doc-rank của RRF) thay vì lái traversal →
    # hub API không kéo cả vùng lân cận vào coverage như cơ chế doc-seed cũ.
    def doc_votes(query, n_doc=5, spill=0.5, max_fanout=20):
        votes = {}
        for ch in search_doc(query, n=n_doc):
            for tgt in doc2code.get(ch["id"], []):
                nd = node_by_id.get(tgt)
                if not nd or nd["label"] not in ("Function", "Class"):
                    continue
                v = ch["score"] * _EDGE_W["Documents"]
                if v > votes.get(tgt, 0.0):
                    votes[tgt] = v
                # lan 1 bước Calls (API được document → helper nó gọi), né hub
                nbrs = fwd_ast.get(tgt, [])
                if len(nbrs) <= max_fanout:
                    for nxt, _w in nbrs:
                        nd2 = node_by_id.get(nxt)
                        if nd2 and nd2["label"] in ("Function", "Class"):
                            nv = v * spill
                            if nv > votes.get(nxt, 0.0):
                                votes[nxt] = nv
        return votes

    # ── _hop_coverage(): BFS thuần từ seed, trả MỌI node Function/Class chạm tới ─
    def _hop_coverage(seeds_fn, fwd, bwd, n_seeds, depth):
        """Pure BFS from seeds — no decay, no rerank, returns ALL reachable Function/Class nodes."""
        seeds = seeds_fn(n_seeds)
        if not seeds: return []
        visited = {s["id"]: 0 for s in seeds}
        queue = deque((s["id"], 0) for s in seeds)
        # Lan tỏa theo cả cạnh xuôi và ngược tới độ sâu `depth`
        while queue:
            nid, d = queue.popleft()
            if d >= depth: continue
            for nxt, _ in fwd.get(nid, []) + bwd.get(nid, []):
                if nxt not in visited:
                    visited[nxt] = d + 1
                    queue.append((nxt, d + 1))
        # Bỏ Module/Document/DocChunk, chỉ trả Function/Class kèm số hop
        out = []
        for nid, hop in visited.items():
            nd = node_by_id.get(nid)
            if not nd or nd["label"] in ("Module", "Document", "DocChunk"): continue
            out.append({"id": nid, "name": _node_name(nd), "label": nd["label"],
                        "file": nd.get("file", ""), "hop": hop, "score": 0.0})
        return out

    # Coverage chỉ qua cạnh code (seed = search hybrid)
    def hop_coverage_ast_fn(query, n_seeds=5, depth=2):
        return _hop_coverage(lambda n: search(query, n=n, method="hybrid"),
                             fwd_ast, bwd_ast, n_seeds, depth)

    # Coverage Full = coverage AST ∪ node có doc-vote (vote KHÔNG kéo hàng xóm);
    # node chỉ-có-vote gán hop=0 (mức seed, không qua traversal).
    def hop_coverage_full_fn(query, n_seeds=5, depth=2, n_doc=5):
        cov = _hop_coverage(lambda n: search(query, n=n, method="hybrid"),
                            fwd_ast, bwd_ast, n_seeds, depth)
        seen = {r["id"] for r in cov}
        for nid in doc_votes(query, n_doc=n_doc):
            if nid not in seen:
                nd = node_by_id[nid]
                cov.append({"id": nid, "name": _node_name(nd), "label": nd["label"],
                            "file": nd.get("file", ""), "hop": 0, "score": 0.0})
        return cov

    # ── _q_sig(): memo tín hiệu query — encode + BM25 scan chỉ 1 lần / query ──
    _q_cache = {}
    def _q_sig(query):
        sig = _q_cache.get(query)
        if sig is None:
            q_emb = _model.encode([_embed_query(query)], normalize_embeddings=True,
                                  convert_to_numpy=True)[0]
            toks = _tokenize_code(query)
            sig = (q_emb, bm25_ast.get_scores(toks), bm25_full.get_scores(toks))
            _q_cache[query] = sig
        return sig

    # ── _ce_scores(): cross-encoder chấm (query, node) — memo theo query+node ─
    # Coverage giữa các config (depth/n_seeds) trùng nhau nhiều → mỗi node chỉ
    # chấm 1 lần/query dù matrix gọi hop ranking 12 lần/instance.
    _ce_q_cache = {}
    def _ce_scores(query, cand):
        cache = _ce_q_cache.setdefault(query, {})
        todo = [nid for nid in cand if nid not in cache]
        if todo:
            q = _embed_query(query)
            pairs = []
            for nid in todo:
                nd = node_by_id.get(nid, {})
                text = (nd.get("source")
                        or f"{nd.get('label','')} {nd.get('name','')} {nd.get('file','')}")[:4000]
                pairs.append((q, text))
            scores = _get_ce().predict(pairs, batch_size=64, show_progress_bar=False)
            for nid, s in zip(todo, scores):
                cache[nid] = float(s)
        return {nid: cache[nid] for nid in cand}

    # ── _hop_ranked(): graph SINH ứng viên, cross-encoder XẾP HẠNG ─────────────
    # CE chấm trực tiếp từng cặp trên MỘT thang điểm nên không còn gì để RRF
    # trộn — RRF chỉ còn trong baseline Hybrid và search_doc (sinh ứng viên doc).
    def _hop_ranked(seeds_fn, fwd, bwd, n_seeds, depth, query, votes=None):
        seeds = seeds_fn(n_seeds)
        if not seeds:
            return []
        # BFS 2 chiều thuần — chỉ để xác định tập reachable + số hop
        hopd = {s["id"]: 0 for s in seeds}
        queue = deque((s["id"], 0) for s in seeds)
        while queue:
            nid, d = queue.popleft()
            if d >= depth:
                continue
            for nxt, _w in fwd.get(nid, []) + bwd.get(nid, []):
                if nxt not in hopd:
                    hopd[nxt] = d + 1
                    queue.append((nxt, d + 1))
        # Candidates = Function/Class trong coverage ∪ node có doc-vote
        # (vote đi vào một mình, không kéo hàng xóm như seed BFS)
        votes = votes or {}
        cand = {nid for nid in hopd
                if node_by_id.get(nid, {}).get("label") in ("Function", "Class")}
        cand |= set(votes)
        cand = list(cand)
        if not cand:
            return []
        ce = _ce_scores(query, cand)
        out = [{"id": nid, "name": _node_name(node_by_id[nid]),
                "label": node_by_id[nid]["label"],
                "file": node_by_id[nid].get("file", ""),
                "hop": hopd.get(nid, 0), "score": round(ce[nid], 4)}
               for nid in cand]
        out.sort(key=lambda r: -r["score"])
        return out

    # AST-hop có xếp hạng: seed = search hybrid, đi qua cạnh code
    def hop_ast_ranked_fn(query, n_seeds=5, depth=2):
        return _hop_ranked(lambda n: search(query, n=n, method="hybrid"),
                           fwd_ast, bwd_ast, n_seeds, depth, query)

    # Full-hop = AST-hop (cùng seed, cùng BFS) + doc_votes MỞ RỘNG tập ứng viên —
    # chênh lệch recall so với AST-hop = đóng góp của lớp doc ở khâu sinh ứng viên.
    def hop_full_ranked_fn(query, n_seeds=5, depth=2, n_doc=5):
        return _hop_ranked(lambda n: search(query, n=n, method="hybrid"),
                           fwd_ast, bwd_ast, n_seeds, depth, query,
                           votes=doc_votes(query, n_doc=n_doc))

    # BFS đơn giản trả khoảng cách hop từ tập seed (dùng cho callers/callees)
    def _bfs(seeds, adj, depth):
        dist = {s: 0 for s in seeds}
        q = deque((s, 0) for s in seeds)
        while q:
            cur, d = q.popleft()
            if d >= depth: continue
            for nxt, _ in adj.get(cur, []):
                if nxt not in dist:
                    dist[nxt] = d + 1
                    q.append((nxt, d + 1))
        return dist

    # Người gọi (đi ngược) / bị gọi (đi xuôi) của 1 node trong đồ thị code
    def callers_fn(node_id, depth=2):
        return _bfs([node_id], bwd_ast, depth)

    def callees_fn(node_id, depth=2):
        return _bfs([node_id], fwd_ast, depth)

    # bridged=0 → Full-hop ≡ AST-hop, biết trước khi nhìn bảng matrix
    n_bridged = sum(1 for n in doc_nodes if n["id"] in doc2code)
    print(f"[{_stem}] indices: ast={len(bm25_nodes)} chunks={len(doc_nodes)} "
          f"(bridged={n_bridged}) vec={vec_matrix.shape} | {time.perf_counter()-_t0:.1f}s", flush=True)
    print(f"[{_stem}] index timing: load={_t_load-_t0:.1f}s "
          f"tok_ast={_t_tok_ast-_t_load:.1f}s bm25_ast={_t_bm25_ast-_t_tok_ast:.1f}s "
          f"vec={_t_vec-_t_bm25_ast:.1f}s tok_doc={_t_tok_doc-_t_vec:.1f}s "
          f"bm25_full={_t_bm25_full-_t_tok_doc:.1f}s vec_full={_t_fvec-_t_bm25_full:.1f}s "
          f"adj={_t_adj-_t_fvec:.1f}s", flush=True)

    # Gói toàn bộ hàm + dữ liệu vào dict kết quả, cache lại rồi trả về
    result = {
        "nodes": nodes, "node_by_id": node_by_id,
        "search": search, "search_full": search_full, "search_doc": search_doc, "search_doc_all": search_doc_all,
        "doc_votes": doc_votes,
        "doc2code": doc2code, "fwd_ast": fwd_ast,    # cạnh đã dựng sẵn, graph_edges() dùng lại
        "documented": {t for ts in doc2code.values() for t in ts},
                "hop_coverage_ast": hop_coverage_ast_fn,
        "hop_coverage_full": hop_coverage_full_fn,
        "hop_ast_ranked": hop_ast_ranked_fn,
        "hop_full_ranked": hop_full_ranked_fn,
        "callers": callers_fn, "callees": callees_fn,
    }
    while len(_INDEX_CACHE) >= INDEX_CACHE_MAX:          # giữ RAM ổn định khi chạy nhiều instance
        _INDEX_CACHE.pop(next(iter(_INDEX_CACHE)))
    _INDEX_CACHE[output_file] = result
    return result

In [11]:
import unidiff
from collections import defaultdict

# ── _changed_lines(): dòng ở bản GỐC bị đổi, theo full path, bỏ dòng context ─────────
# Dòng xóa → lấy đúng số dòng gốc. Dòng thêm → neo vào dòng gốc ngay trước vị trí chèn.
# File mới tạo không có ở commit gốc → tách riêng, không tính là gold.
# File đổi tên: unidiff `path` trả tên MỚI, nhưng dòng gốc thuộc file ở tên CŨ → dùng tên cũ (_src_path).
def _src_path(pf):
    s = pf.source_file or ""
    if s.startswith("a/"):
        s = s[2:]
    return pf.path if (pf.is_added_file or s in ("", "/dev/null")) else s

def _changed_lines(patch_text):
    out, added_files = {}, set()
    for pf in unidiff.PatchSet(patch_text):
        if pf.is_added_file:
            added_files.add(pf.path)
            continue
        lines = out.setdefault(_src_path(pf), set())
        for h in pf:
            last = None
            for ln in h:
                if ln.is_removed:
                    lines.add(ln.source_line_no)
                    last = ln.source_line_no
                elif ln.is_context:
                    last = ln.source_line_no
                elif ln.is_added:
                    lines.add(last if last is not None else h.source_start)
    return out, added_files

# ── _patch_changes(): như _changed_lines nhưng giữ riêng dòng xóa và các khối dòng thêm ─────────
# Khối = các dòng thêm liền nhau, cùng điểm chèn (dòng gốc ngay trước, dòng gốc ngay sau).
# Mỗi dòng trong khối = (độ thụt lề, nội dung); dòng trống có độ thụt lề None.
# Hunk chèn thuần "-L,0" nghĩa là chèn sau dòng L. renames = {tên cũ: tên mới} của file đổi tên.
def _patch_changes(patch_text):
    out, renames, added_files = {}, {}, set()
    for pf in unidiff.PatchSet(patch_text):
        if pf.is_added_file:
            added_files.add(pf.path)
            continue
        src = _src_path(pf)
        if src != pf.path:
            renames[src] = pf.path
        rec = out.setdefault(src, {"removed": set(), "inserts": []})
        for h in pf:
            hl, last, blk = list(h), None, None
            for i, ln in enumerate(hl):
                if not ln.is_added:
                    blk = None
                if ln.is_removed:
                    rec["removed"].add(ln.source_line_no)
                    last = ln.source_line_no
                elif ln.is_context:
                    last = ln.source_line_no
                elif ln.is_added:
                    if blk is None:
                        nxt = next((x.source_line_no for x in hl[i + 1:] if not x.is_added), None)
                        if h.source_length == 0:
                            prev = last if last is not None else h.source_start
                        else:
                            prev = last if last is not None else h.source_start - 1
                        blk = (prev, nxt if nxt is not None else prev + 1, [])
                        rec["inserts"].append(blk)
                    text = ln.value.rstrip("\n")
                    blk[2].append((len(text) - len(text.lstrip(" \t")) if text.strip() else None, text))
    return out, renames, added_files

# ── get_oracle(): mỗi dòng đổi → node Function/Class HẸP NHẤT chứa nó ───────────────
# Class chỉ là gold khi có dòng đổi nằm trong class nhưng ngoài mọi method.
# File chỉ đổi ở cấp module (import, hằng số), không dòng nào nằm trong Function/Class
# → gold là node Module của file đó (vd pytest-5227 chỉ sửa hằng số DEFAULT_LOG_FORMAT).
# Khoảng dòng của node tính từ decorator đầu tiên (deco_start, do _file_nodes_at gắn).
# Dòng thêm gán theo điểm chèn của khối: (1) khối nằm giữa thân node; (2) khối nối vào cuối thân node và mọi dòng
# từ đầu khối tới dòng đó đều thụt sâu hơn dòng def (gặp dòng thụt nông hơn = bắt đầu code mới); (3) decorator mới
# ngay trên dòng def, khi mọi dòng sau nó trong khối cũng là decorator. Dòng trống chỉ tính khi nằm trong thân một
# Function. Node không có thông tin thụt lề (graph cũ) → gán theo dòng trước như cũ.
# Kết quả có thêm "renames" = {tên cũ: tên mới} để nơi cần (cạnh history) đổi id sang tên mới.
def get_oracle(patch_text, graph_nodes):
    changed, renames, added_files = _patch_changes(patch_text)
    by_file = defaultdict(list)
    module_of = {n["name"]: n["id"] for n in graph_nodes if n["label"] == "Module"}   # name = path file
    for n in graph_nodes:
        if n["label"] in ("Function", "Class") and n.get("start_line") and n.get("end_line"):
            by_file[n["file"]].append(n)
    start = lambda n: n.get("deco_start") or n["start_line"]
    narrow = lambda c: min(c, key=lambda n: (n["end_line"] - start(n), n["id"]))
    gold, module_only = set(), set()
    for f, rec in changed.items():
        if not f.endswith(".py"):
            continue
        nodes, hit = by_file[f], False
        for L in rec["removed"]:
            inner = [n for n in nodes if start(n) <= L <= n["end_line"]]
            if inner:
                gold.add(narrow(inner)["id"])
                hit = True
        for a, b, blk in rec["inserts"]:
            if not (nodes and all("indent" in n for n in nodes)):
                cand = [n for n in nodes if start(n) <= a <= n["end_line"]] or \
                       [n for n in nodes if start(n) <= b <= n["end_line"]]
                if cand:
                    gold.add(narrow(cand)["id"])
                    hit = True
                continue
            inside = [n for n in nodes if start(n) <= a and b <= n["end_line"]]
            tail = [n for n in nodes if n["end_line"] == a]
            head = [n for n in nodes if start(n) == b]
            low = None                                   # thụt lề nhỏ nhất từ đầu khối tới dòng đang xét
            for i, (d, text) in enumerate(blk):
                if d is None:
                    cand = [n for n in inside if n["label"] == "Function"]
                else:
                    low = d if low is None else min(low, d)
                    cand = inside + [n for n in tail if low > n["indent"]]
                    if text.lstrip().startswith("@") and all(t.lstrip().startswith("@") for x, t in blk[i:] if x is not None):
                        cand += [n for n in head if d == n["indent"]]
                if cand:
                    gold.add(narrow(cand)["id"])
                    hit = True
        if (rec["removed"] or rec["inserts"]) and not hit:
            module_only.add(f)
            if f in module_of:
                gold.add(module_of[f])
    files_code  = {f for f in changed if f.endswith(".py")}
    graph_files = {n["file"] for n in graph_nodes if n.get("file")}
    return {"nodes": gold,
            "files_code": files_code,
            "added_files": {f for f in added_files if f.endswith(".py")},
            "unreachable": files_code - graph_files,
            "module_only": module_only,
            "renames": renames}

# File của 1 node: node Module không có trường "file", đường dẫn nằm ở "name"
def node_file(nd):
    return nd.get("file") or (nd.get("name", "") if nd.get("label") == "Module" else "")

def get_oracle_nodes(patch_text, graph_nodes):
    return get_oracle(patch_text, graph_nodes)["nodes"]

def get_oracle_files(patch_text):
    changed, _ = _changed_lines(patch_text)
    return {f for f in changed if f.endswith(".py")}

print("get_oracle() / get_oracle_nodes() / get_oracle_files() ready: line-level oracle")

get_oracle() / get_oracle_nodes() / get_oracle_files() ready: line-level oracle


In [12]:
# ── Xóa cache index để dùng ĐÚNG code search/index hiện tại ─────────────────────
# load_graph_indices() cache kết quả KÈM closure (search, doc_votes, ...) theo _INDEX_CACHE.
# Sau khi sửa cell load_graph_indices, chạy cell này để index được dựng lại từ code mới.
# (Không đụng _EMB_CACHE/_MODEL_CACHE → embedding không phải tính lại.)
_INDEX_CACHE.clear()
print(f"_INDEX_CACHE cleared ({len(_INDEX_CACHE)} entries) — indices sẽ build lại từ code hiện tại.")

_INDEX_CACHE cleared (0 entries) — indices sẽ build lại từ code hiện tại.


In [13]:
# ── Xóa file kết quả không đọc được, để cell tạo ra file đó tạo lại nó ─────────────────────────────────────────
# Lần chạy bị dừng giữa lúc ghi có thể để lại file cụt. Đã gặp với file graph, scale_rows_v4.jsonl và file hunk.
# Các cell sau dùng lại file đã có, nên file cụt làm chúng lỗi. Cell này đọc thử mọi file .json và .jsonl mà notebook
# ghi: graph và file tóm tắt ở WORK_DIR, và file trong history, v4 và judge_v4. File nào không đọc được thì bị xóa.
# Cell tạo ra file đó sẽ tạo lại nó như khi file chưa có.
_files = sorted(Path(WORK_DIR).glob("*.json"))                         # graph và file tóm tắt
for _d in ("history", "v4", "judge_v4"):
    _files += sorted(p for p in (Path(WORK_DIR) / _d).rglob("*") if p.suffix in (".json", ".jsonl"))
_broken = []
for p in tqdm(_files, desc="đọc thử file", unit="file"):
    try:
        if p.suffix == ".jsonl":
            with open(p, encoding="utf-8") as f:
                for l in f:
                    if l.strip():
                        json.loads(l)
        elif p.parent == Path(WORK_DIR):
            read_graph_json(p)                                         # đọc như các cell sau đọc graph
        else:
            with open(p, encoding="utf-8") as f:
                json.load(f)
    except ValueError:                                                 # file cụt
        p.unlink()
        _broken.append(p.name)
print(f"Đã đọc thử {len(_files)} file | xóa {len(_broken)} file không đọc được: {_broken}")

đọc thử file:   0%|          | 0/694 [00:00<?, ?file/s]

Đã đọc thử 694 file | xóa 0 file không đọc được: []


## 2. Build Test Data (SWE-bench)
Load SWE-bench Lite, keep the instances of the repos in `REPOS` (at most `MAX_PER_REPO` per repo), and clone each repo.

In [14]:
from datasets import load_dataset

# ── Nạp SWE-bench Lite, lọc các instance thuộc REPO đang xét ──────────────────
_swe_lite = load_dataset("princeton-nlp/SWE-bench_lite", split="test")
SWE_INSTANCES = [
    {
        "id":     row["instance_id"],
        "repo":   row["repo"],
        "commit": row["base_commit"],                    # commit để checkout
        "title":  row["instance_id"].split("__", 1)[-1],
        "text":   row["problem_statement"],              # dùng làm query truy hồi
        "patch":  row["patch"],                          # dùng để trích đáp án vàng
    }
    for row in _swe_lite
    if row["repo"] in REPOS
]

# Giới hạn số instance mỗi repo (MAX_PER_REPO): sắp theo sha1(instance_id) rồi lấy k đầu → mẫu cố định
import hashlib
_keep = set()
for _repo in REPOS:
    _ids = sorted((i["id"] for i in SWE_INSTANCES if i["repo"] == _repo),
                  key=lambda x: hashlib.sha1(x.encode()).hexdigest())
    _cap = MAX_PER_REPO.get(_repo, MAX_PER_REPO_DEFAULT)
    _keep.update(_ids if _cap is None else _ids[:_cap])
SWE_INSTANCES = [i for i in SWE_INSTANCES if i["id"] in _keep]      # giữ thứ tự của SWE-bench

# Chọn 1 instance theo INSTANCE_IDX (None → dùng HEAD, không checkout)
TARGET_COMMIT = SWE_INSTANCES[INSTANCE_IDX]["commit"] if INSTANCE_IDX is not None else None
TARGET_ID     = SWE_INSTANCES[INSTANCE_IDX]["id"]     if INSTANCE_IDX is not None else "HEAD"

from collections import Counter as _Counter
_by_repo = _Counter(i["repo"] for i in SWE_INSTANCES)
print(f"Total instances : {len(SWE_INSTANCES)}  ({dict(_by_repo)})")
print(f"Instance        : {TARGET_ID}")
print(f"Commit          : {TARGET_COMMIT[:8] if TARGET_COMMIT else 'HEAD'}")

README.md:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120kB            

data/dev-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.12MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating dev split:   0%|          | 0/23 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Total instances : 207  ({'django/django': 114, 'sphinx-doc/sphinx': 16, 'sympy/sympy': 77})
Instance        : django__django-10914
Commit          : e7fd69d0


In [15]:
import os
from pathlib import Path
from git import Repo, RemoteProgress
from tqdm.auto import tqdm

# Thanh tiến độ cho git clone (mỗi pha: counting → compressing → receiving → resolving)
class _CloneProgress(RemoteProgress):
    def __init__(self, name):
        super().__init__()
        self.bar = tqdm(desc=f"clone {name}", unit="obj")
    def update(self, op_code, cur_count, max_count=None, message=""):
        self.bar.total = int(max_count) if max_count else self.bar.total
        self.bar.n = int(cur_count)
        self.bar.refresh()

# ── Clone tất cả repo trong REPOS nếu chưa có trong WORK_DIR ──────────────────
for _r in REPOS:
    _p = repo_path_of(_r)
    if not Path(_p).exists():
        _prog = _CloneProgress(_r)
        Repo.clone_from(f"https://github.com/{_r}.git", _p, progress=_prog)
        _prog.bar.close()
print(f"{len(REPOS)} repo sẵn sàng: " + ", ".join(repo_path_of(r) for r in REPOS))

# Checkout commit của instance đã chọn (trên đúng repo của nó)
if INSTANCE_IDX is not None:
    _inst = SWE_INSTANCES[INSTANCE_IDX]
    repo = Repo(repo_path_of(_inst["repo"]))
    repo.git.checkout(_inst["commit"], force=True)   # force: thư mục làm việc không giữ quyền file
    BASE_COMMIT = repo.head.commit.hexsha[:8]
    OUTPUT_FILE = f"{_inst['repo'].split('/')[1]}_graph_{BASE_COMMIT}.json"
    print(f"Checked out : {BASE_COMMIT}  ({_inst['id']})")
    print(f"Output      : {OUTPUT_FILE}")

clone django/django: 0obj [00:00, ?obj/s]

clone sphinx-doc/sphinx: 0obj [00:00, ?obj/s]

clone sympy/sympy: 0obj [00:00, ?obj/s]

3 repo sẵn sàng: /tmp/repos/django, /tmp/repos/sphinx, /tmp/repos/sympy
Checked out : e7fd69d0  (django__django-10914)
Output      : django_graph_e7fd69d0.json


## 4. Truy xuất có doc: các hàm dùng chung

Section này chỉ định nghĩa hàm. `doc_set()` tạo tập node do doc đưa vào theo nguồn `vote`, tức `doc_votes`. Hậu tố `_ast` giữ cùng tập node nhưng sắp theo hạng AST sâu. `scale_row` ở Section 5 dùng các hàm này.

In [16]:
# ── Hàm dùng chung cho truy xuất có doc: nguồn doc vote, cạnh lấy từ index ───────────────────────────
import hashlib
import json
import random
import numpy as np
import pandas as pd
from collections import defaultdict

# Nguồn doc của doc_set(): vote (doc_votes). Hậu tố _ast: sắp theo hạng AST sâu.
EXP_N_DOC    = 5                 # số chunk doc, giống doc_votes(n_doc=5)
EXP_DEEP     = 500               # độ sâu xếp hạng AST dùng để sắp node doc cho nguồn *_ast
_FC = ("Function", "Class")

# Cạnh lấy từ index đã dựng (không đọc lại JSON): d2c = chunk → [code] (cạnh Documents),
# fwd = node → [node] theo cạnh AST (fwd_ast bỏ trọng số). Trả bản sao để không sửa index.
def graph_edges(output_file):
    idx = load_graph_indices(output_file)
    d2c = defaultdict(list, idx["doc2code"])
    fwd = defaultdict(list, {s: [t for t, _w in lst] for s, lst in idx["fwd_ast"].items()})
    return d2c, fwd

# Tập node doc của 1 nguồn + các chunk nó đã dùng. Nhớ theo (graph, nguồn, issue); scale_row xóa bộ nhớ đầu mỗi instance.
_DOC_MEMO = {}
def doc_set(idx, q, d2c, fwd, src, seed):
    key = (id(idx), src, q, seed)
    if key not in _DOC_MEMO:
        if len(_DOC_MEMO) > 64:
            _DOC_MEMO.clear()
        if src.removesuffix("_ast") != "vote":
            raise ValueError(src)
        chunks = [r["id"] for r in idx["search_doc"](q, n=EXP_N_DOC)]           # = các chunk doc_votes dùng
        votes = idx["doc_votes"](q, n_doc=EXP_N_DOC)                            # {node_id: điểm}
        doc = [nid for nid, _ in sorted(votes.items(), key=lambda x: (-x[1], x[0]))]
        if src.endswith("_ast"):
            deep = {r["id"]: i for i, r in enumerate(idx["search"](q, n=EXP_DEEP, method="hybrid", pool=EXP_DEEP))}
            doc = sorted(doc, key=lambda nid: deep.get(nid, len(deep)))   # ổn định: ngoài bảng giữ thứ tự cũ
        _DOC_MEMO[key] = (chunks, doc)
    return _DOC_MEMO[key]

def instance_seed(iid):
    return int(hashlib.sha1(iid.encode()).hexdigest()[:8], 16)

print(f"Hàm truy xuất có doc sẵn sàng | deep={EXP_DEEP} n_doc={EXP_N_DOC}")

Hàm truy xuất có doc sẵn sàng | deep=500 n_doc=5


In [17]:
# ── Quy tắc nêu tên gold (scale_row dùng cho cột issue_names_gold) ─────────────────────────────────────
# Tên khớp theo ranh giới từ. Tên trần ngắn hoặc phổ biến có thể khớp nhầm.
import re

def _mention_re(key):
    return re.compile(r"(?<!\w)" + re.escape(key) + r"(?!\w)")

print("Quy tắc nêu tên gold sẵn sàng")

Quy tắc nêu tên gold sẵn sàng


## 5. Mở rộng lên SWE-bench đầy đủ (graph theo snapshot)

Section này nạp SWE-bench đầy đủ cho các repo trong `SCALE_REPOS`.

Build một graph cho mỗi commit là quá tốn kém. Vì vậy, instance được chia theo version, rồi theo các khoảng `SNAPSHOT_BIN_DAYS` ngày. Mỗi khoảng dùng chung một graph, dựng tại base commit sớm nhất của khoảng. Do đó, graph không bao giờ chứa bản sửa của instance dùng nó. Gold được tính trên đúng base commit của instance, rồi đối chiếu sang graph của snapshot theo đường dẫn file và tên đầy đủ. Kiểm chứng trên máy cho thấy 99% gold đối chiếu được, và trên 114 instance Django Lite, kết quả xếp hạng BM25 trùng với graph theo commit ở 113/113 instance.

In [18]:
# ── Section 5: nạp SWE-bench đầy đủ, clone repo, gán snapshot cho từng instance ───────────
import subprocess
from datasets import load_dataset
from git import Repo

_lite_ids = {row["instance_id"] for row in load_dataset("princeton-nlp/SWE-bench_Lite", split="test")}
SCALE_INSTANCES = [
    {"id": row["instance_id"], "repo": row["repo"], "commit": row["base_commit"], "version": row["version"],
     "text": row["problem_statement"], "patch": row["patch"], "lite": row["instance_id"] in _lite_ids}
    for row in load_dataset("princeton-nlp/SWE-bench", split="test") if row["repo"] in SCALE_REPOS
]

# Clone repo còn thiếu (cần đủ lịch sử: gold được tính trên base commit bằng git show)
for _r in SCALE_REPOS:
    _p = repo_path_of(_r)
    if not Path(_p).exists():
        _prog = _CloneProgress(_r)
        Repo.clone_from(f"https://github.com/{_r}.git", _p, progress=_prog)
        _prog.bar.close()

def _commit_times(repo_path, commits):
    """Thời điểm commit (Unix time) của nhiều commit; chia đợt để lệnh git không quá dài."""
    out = {}
    for i in range(0, len(commits), 200):
        res = subprocess.run(["git", "-C", repo_path, "show", "-s", "--format=%H %ct", *commits[i:i + 200]],
                             capture_output=True, text=True, check=True).stdout
        for line in res.splitlines():
            h, t = line.split()
            out[h] = int(t)
    return out

# Snapshot: trong mỗi (repo, version), chia instance theo thời gian thành các khoảng SNAPSHOT_BIN_DAYS ngày.
# Snapshot của một khoảng = base commit sớm nhất trong khoảng → không chứa bản sửa của instance nào dùng nó.
for _r in SCALE_REPOS:
    _insts = [i for i in SCALE_INSTANCES if i["repo"] == _r]
    _ct = _commit_times(repo_path_of(_r), sorted({i["commit"] for i in _insts}))
    for i in _insts:
        i["ct"] = _ct[i["commit"]]
    for v in {i["version"] for i in _insts}:
        start = snap = None
        for i in sorted((i for i in _insts if i["version"] == v), key=lambda i: (i["ct"], i["id"])):
            if start is None or i["ct"] - start > SNAPSHOT_BIN_DAYS * 86400:
                start, snap = i["ct"], i["commit"]
            i["snapshot"], i["snap_ct"] = snap, start

# Chạy theo thứ tự snapshot → mỗi graph chỉ build và dựng index 1 lần
SCALE_INSTANCES.sort(key=lambda i: (i["repo"], i["snap_ct"], i["snapshot"], i["id"]))

README.md:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.38MB            

data/dev-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.1MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (10 outputs left out)

clone pylint-dev/pylint: 0obj [00:00, ?obj/s]

clone psf/requests: 0obj [00:00, ?obj/s]

clone mwaskom/seaborn: 0obj [00:00, ?obj/s]

clone pallets/flask: 0obj [00:00, ?obj/s]

### 5b. Cạnh Documents mới và lớp history

Cell `doc-links-v2` cập nhật cạnh `Documents` của các graph đã build theo cách nối mới. Cách nối mới thử tên kèm module của đoạn doc trước. Khi nhiều node cùng khớp, nó chỉ giữ node nằm trong module đó. Nếu vẫn mơ hồ, nó không nối. Cell chỉ thay cạnh, không parse lại code và không embed lại. Chunk doc trích lại phải trùng với graph cũ, nếu không cell sẽ dừng.

Cell `hist-build` thêm lớp history vào graph. Mỗi node `Commit` là một commit trên nhánh chính (`git log --first-parent`), so với cha thứ nhất. Cạnh `Modifies` nối commit với các hàm và lớp mà nó sửa, tính bằng đúng hàm tính gold. Dòng nằm ngoài mọi hàm và lớp được ghi thành cạnh tới `Module`, và cạnh này chỉ được lưu lại.

In [19]:
# ── Section 5b: cập nhật cạnh Documents của graph đã build theo cách nối mới (không build lại graph) ──────────
# Graph build trước khi có cách nối mới vẫn mang cạnh cũ. Cell này trích lại doc tại commit snapshot, nối lại cạnh
# bằng resolve_doc_links (cell PqDgRg_1mNuK), rồi chỉ thay cạnh Documents và References trong file graph.
# Node (kể cả chunk doc và embedding) giữ nguyên; chunk trích lại phải trùng id và text với graph cũ. Nếu không trùng
# (graph build bằng bộ nhớ đệm cũ, khóa chỉ là nội dung file, nên file doc đổi chỗ giữa các snapshot mang đường dẫn cũ),
# graph đó được build lại với force=True.
from tqdm.auto import tqdm

DOC_LINKS_VERSION = "v2"

def update_doc_links(snapshot, repo):
    rp = repo_path_of(repo)
    of = build_graph(snapshot, repo_path=rp)                  # graph đã có thì chỉ trả đường dẫn
    g = read_graph_json(of)
    if g["meta"].get("doc_links") == DOC_LINKS_VERSION:
        return of, None
    _Repo(rp).git.checkout(snapshot, force=True)
    docs = [r for r in (DocExtractor(rel, rp).extract() for rel in get_doc_files(rp)) if r]
    old_ch = {n["id"]: (n.get("heading"), n.get("text")) for n in g["nodes"] if n["label"] == "DocChunk"}
    new_ch = {n["id"]: (n.get("heading"), n.get("text")) for r in docs for n in r["nodes"] if n["label"] == "DocChunk"}
    if old_ch != new_ch:
        return build_graph(snapshot, repo_path=rp, force=True), "build lại"
    new_edges = resolve_doc_links(docs, build_name_index([n for n in g["nodes"] if n["label"] in ("Function", "Class")]))
    n_old = sum(e["type"] == "Documents" for e in g["edges"])
    g["edges"] = [e for e in g["edges"] if e["type"] not in ("Documents", "References")] + new_edges
    g["meta"]["doc_links"] = DOC_LINKS_VERSION
    g["meta"]["edge_count"] = len(g["edges"])
    write_graph_json(of, g)
    write_graph_summary(of, g["nodes"], g["edges"])
    _INDEX_CACHE.pop(of, None)
    return of, (n_old, sum(e["type"] == "Documents" for e in new_edges))

_dl = []
for _repo, _snap in tqdm(sorted({(i["repo"], i["snapshot"]) for i in SCALE_INSTANCES}), desc="cạnh Documents v2",
                         unit="graph"):
    _of, _st = update_doc_links(_snap, _repo)
    _up = isinstance(_st, tuple)
    _dl.append({"Repo": _repo, "Snapshot": _snap[:8], "Cập nhật": _up, "Build lại": _st == "build lại",
                "Cạnh Documents cũ": _st[0] if _up else None, "Cạnh Documents mới": _st[1] if _up else None})
_dl = pd.DataFrame(_dl)
print(f"{len(_dl)} graph: thay cạnh {_dl['Cập nhật'].sum()} | build lại {_dl['Build lại'].sum()} "
      f"(chunk cũ mang sai đường dẫn) | đã là {DOC_LINKS_VERSION} từ trước {(~_dl['Cập nhật'] & ~_dl['Build lại']).sum()}")
if _dl["Build lại"].any():
    print("Build lại:", ", ".join(f"{r.Repo} {r.Snapshot}" for r in _dl[_dl["Build lại"]].itertuples()))
if _dl["Cập nhật"].any():
    display(_dl[_dl["Cập nhật"]].groupby("Repo")[["Cạnh Documents cũ", "Cạnh Documents mới"]].sum())

cạnh Documents v2:   0%|          | 0/279 [00:00<?, ?graph/s]

[0e1d299f] disk cache hit → /kaggle/working/astropy_graph_0e1d299f.json
[264d9677] disk cache hit → /kaggle/working/astropy_graph_264d9677.json
[38322105] disk cache hit → /kaggle/working/astropy_graph_38322105.json
[3a0cd2d8] disk cache hit → /kaggle/working/astropy_graph_3a0cd2d8.json
[43ee5806] disk cache hit → /kaggle/working/astropy_graph_43ee5806.json
[4fc9f31a] disk cache hit → /kaggle/working/astropy_graph_4fc9f31a.json
[5aa2d0be] disk cache hit → /kaggle/working/astropy_graph_5aa2d0be.json
[5e5764ed] disk cache hit → /kaggle/working/astropy_graph_5e5764ed.json
[69e2fabd] disk cache hit → /kaggle/working/astropy_graph_69e2fabd.json
[848c8fa2] disk cache hit → /kaggle/working/astropy_graph_848c8fa2.json
[9626265d] disk cache hit → /kaggle/working/astropy_graph_9626265d.json
[a3f4ae6c] disk cache hit → /kaggle/working/astropy_graph_a3f4ae6c.json
[a6c71237] disk cache hit → /kaggle/working/astropy_graph_a6c71237.json
[a7141cd9] disk cache hit → /kaggle/working/astropy_graph_a7141c

In [20]:
# ── Section 5: gold trên base commit → graph snapshot; 1 dòng kết quả cho mỗi instance ─────
import subprocess
import tempfile

SCALE_SRCS = ("vote_ast",)                   # nguồn doc: doc_votes, tập node sắp theo hạng AST
SCALE_TOPK = 100                             # số node đầu của xếp hạng AST được lưu id (v4-run đặt lại bằng V4_TOPK)
_DOC_COV   = {}                              # snapshot → doc_cov (tỷ lệ Function/Class có cạnh Documents)

def _key(nid):
    """Id node bỏ tiền tố snapshot: 'abcd1234:django/db/x.py:Cls:m' → 'django/db/x.py:Cls:m'."""
    return nid.split(":", 1)[1]

def _skip_rel(rel):
    """Cùng bộ lọc với get_python_files: file test/docs/example và file cấu hình không có trong graph."""
    parts = rel.split("/")
    return any(p in _SKIP_DIRS for p in parts[:-1]) or parts[-1] in _SKIP_FILES

def _file_nodes_at(repo_path, commit, rel):
    """Node (dạng phẳng như build_graph) của 1 file tại 1 commit; id bỏ tiền tố commit."""
    data = subprocess.run(["git", "-C", repo_path, "show", f"{commit}:{rel}"], capture_output=True).stdout
    if not data:
        return []
    with tempfile.TemporaryDirectory() as td:
        p = Path(td) / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(data)
        r = ASTEngine(str(p), td, "X", SCHEMA, EXTRACTOR).parse()
    deco, stack = {}, [_parser.parse(data).root_node]       # dòng def/class → dòng decorator đầu tiên
    while stack:
        t = stack.pop()
        if t.type == "decorated_definition" and t.child_by_field_name("definition") is not None:
            deco[t.child_by_field_name("definition").start_point[0] + 1] = t.start_point[0] + 1
        stack.extend(t.children)
    lines = data.decode("utf-8", errors="replace").split("\n")
    out = []
    for n in (r or {}).get("nodes", []):
        pr = n["properties"]
        nd = {"id": n["id"].split(":", 1)[1], "label": n["labels"][0], "name": pr.get("name", ""),
              "file": pr.get("file", ""), "start_line": pr.get("start_line"), "end_line": pr.get("end_line")}
        s = nd["start_line"]
        if nd["label"] in ("Function", "Class") and s and s <= len(lines):
            nd["deco_start"] = deco.get(s, s)
            nd["indent"] = len(lines[s - 1]) - len(lines[s - 1].lstrip(" \t"))
        out.append(nd)
    return out

def get_oracle_snapshot(inst, idx):
    """Như get_oracle(), nhưng gold tính trên đúng base commit (parse riêng các file bị sửa), rồi đổi sang
    id của graph snapshot. Gold không có trong snapshot (hàm thêm sau snapshot) bị bỏ, đếm ở 'dropped'."""
    changed, _ = _changed_lines(inst["patch"])
    rp = repo_path_of(inst["repo"])
    nodes = [n for rel in changed if rel.endswith(".py") and not _skip_rel(rel)
             for n in _file_nodes_at(rp, inst["commit"], rel)]
    orc = get_oracle(inst["patch"], nodes)
    ids = set(idx["node_by_id"])
    mapped = {f"{inst['snapshot'][:8]}:{k}" for k in orc["nodes"]}
    orc["dropped"] = len(mapped - ids)
    orc["nodes"] = mapped & ids
    return orc

def scale_row(c):
    """Dữ liệu của 1 instance, id bỏ tiền tố snapshot: ast_ids = SCALE_TOPK node đầu của xếp hạng AST; gold_ids;
    issue_names_gold = issue nêu sẵn tên một gold; doc_ids_<src>, doc_score_<src> = tập doc theo thứ tự và điểm
    doc_votes; doc_r_<src> = hạng AST của từng node doc (0 = ngoài xếp hạng AST); doc_gold_<src> = vị trí trong tập doc
    của từng gold (None = không có); doc_cand, doc_cand_all = ứng viên bằng chứng DOC kèm điểm gốc; gold_doc = số gold có
    cạnh Documents; doc_cov = tỷ lệ Function/Class của snapshot có cạnh Documents."""
    idx, nbi, d2c, fwd, q = c["idx"], c["nbi"], c["d2c"], c["fwd"], c["q"]
    seed = instance_seed(c["iid"])
    _DOC_MEMO.clear()
    gfc = sorted(g for g in c["gold"] if nbi[g]["label"] in _FC)
    deep = [r["id"] for r in idx["search"](q, n=EXP_DEEP, method="hybrid", pool=EXP_DEEP) if r["label"] in _FC]
    rk = {x: i + 1 for i, x in enumerate(deep)}
    row = {"issue_names_gold": int(any(_mention_re(nbi[g]["name"]).search(q) for g in gfc)),
           "ast_ids": [_key(x) for x in deep[:SCALE_TOPK]], "gold_ids": [_key(g) for g in gfc]}
    # Biến độ phủ doc (mục 10.2 pipeline_design.md). doc_cov dùng cùng định nghĩa với mục 1 của coverage_report.md và được
    # nhớ theo snapshot.
    snap = c["inst"]["snapshot"]
    if snap not in _DOC_COV:
        fc = [x for x, n in nbi.items() if n["label"] in _FC]
        _DOC_COV[snap] = sum(x in idx["documented"] for x in fc) / max(1, len(fc))
    row["gold_doc"] = sum(g in idx["documented"] for g in gfc)
    row["doc_cov"] = round(_DOC_COV[snap], 6)
    for s in SCALE_SRCS:
        _, doc = doc_set(idx, q, d2c, fwd, s, seed)
        votes = idx["doc_votes"](q, n_doc=EXP_N_DOC)
        pos = {x: j + 1 for j, x in enumerate(doc)}
        row[f"doc_ids_{s}"] = [_key(x) for x in doc]
        row[f"doc_score_{s}"] = [round(votes.get(x, 0.0), 6) for x in doc]
        row[f"doc_r_{s}"] = [rk.get(x, 0) for x in doc]
        row[f"doc_gold_{s}"] = [pos.get(g) for g in gfc]
    # Ứng viên bằng chứng DOC kèm điểm gốc, để đặt ngưỡng ngoại tuyến (mục 14.2 planning.md): V4_EVID_N chunk đầu của
    # search_doc (chunk có cạnh Documents; 5 chunk đầu chính là các chunk mà nguồn vote_ast dùng) và của search_doc_all
    # (mọi chunk). Mỗi phần tử: id, score (RRF), bm25, cos (cosine embedding), targets (đích cạnh Documents, [] nếu không
    # có cạnh).
    for key, fn in (("doc_cand", "search_doc"), ("doc_cand_all", "search_doc_all")):
        row[key] = [{"id": x["id"], "score": x["score"], "bm25": x["bm25"], "cos": x["cos"],
                     "targets": [_key(t) for t in d2c.get(x["id"], []) if nbi.get(t, {}).get("label") in _FC]}
                    for x in idx[fn](q, n=V4_EVID_N)]
    return row

print(f"Section 5 sẵn sàng | nguồn doc {SCALE_SRCS}")

Section 5 sẵn sàng | nguồn doc ('vote_ast',)


In [21]:
# ── Section 5b: lớp history của graph: node Commit, cạnh Modifies ──────────────────────────────────────────
# Mỗi repo build 1 lần → HIST_DIR/<repo>_history.jsonl (1 dòng / commit: sha, cha thứ nhất, thời điểm, message,
# touch = hàm/lớp bị sửa, touch_mod = file có dòng đổi ngoài mọi hàm/lớp → cạnh Modifies tới Module, không bỏ phiếu).
import random
import shutil
import subprocess

HIST_DIR       = Path(WORK_DIR) / "history"
HIST_INPUT     = None        # thư mục có sẵn file <repo>_history.jsonl (vd dataset Kaggle); None = build tại chỗ
HIST_YEARS     = 5           # chỉ dùng commit trong HIST_YEARS năm trước snapshot
HIST_MAX_TOUCH = 50          # bỏ commit sửa quá nhiều hàm/lớp (đổi tên hàng loạt, format lại code)
HIST_DIR.mkdir(parents=True, exist_ok=True)
_YEAR = 365 * 86400
_SNAP_CT = {(i["repo"], i["snapshot"]): i["snap_ct"] for i in SCALE_INSTANCES}

def _git(rp, *args):
    """stdout của git dạng text. Đọc bytes rồi chỉ đổi CRLF thành LF, giữ CR đứng riêng. text=True đổi cả CR đứng riêng
    thành xuống dòng, làm một dòng của diff tách làm hai, và unidiff báo "Hunk diff line expected"."""
    out = subprocess.run(["git", "-C", rp, *args], capture_output=True).stdout
    return out.decode("utf-8", errors="replace").replace("\r\n", "\n")

def _hist_path(repo):
    return HIST_DIR / f"{repo.split('/')[1]}_history_v2.jsonl"   # v2: oracle đã sửa (đổi tên file, decorator, dòng thêm)

def _hist_units(repo):
    """Đơn vị = commit trên nhánh chính, từ snapshot mới nhất lùi tới HIST_YEARS năm trước snapshot cũ nhất."""
    snaps = {s: t for (r, s), t in _SNAP_CT.items() if r == repo}
    tip, since = max(snaps, key=snaps.get), min(snaps.values()) - HIST_YEARS * _YEAR
    # -z: git ngăn các commit bằng NUL. Message có thể chứa 0x1e, nên không dùng 0x1e để ngăn commit.
    log = _git(repo_path_of(repo), "log", "-z", "--first-parent", f"--since=@{since}", "--format=%H%x1f%P%x1f%ct%x1f%B", tip)
    units = []
    for rec in log.split("\0"):
        rec = rec.strip("\n")
        if rec:
            h, parents, ct, msg = rec.split("\x1f", 3)
            if parents:
                units.append((h, parents.split()[0], int(ct), msg.strip(), len(parents.split()) > 1))
    return units

def build_history(repo, units):
    """Ghi 1 dòng cho mỗi đơn vị của units vào _hist_path(repo)."""
    rp = repo_path_of(repo)
    with open(_hist_path(repo), "w", encoding="utf-8") as f:
        for h, p, ct, msg, merge in tqdm(units, desc=f"history {repo}", unit="commit"):
            diff = _git(rp, "diff", "-U0", p, h, "--", "*.py")
            changed, _ = _changed_lines(diff) if diff else ({}, set())
            nodes, mods = [], set()
            for rel, lines in changed.items():
                if not rel.endswith(".py") or _skip_rel(rel):
                    continue
                fn = _file_nodes_at(rp, p, rel)                    # node của file tại commit cha, như khi tính gold
                nodes += fn
                spans = [(n["start_line"], n["end_line"]) for n in fn if n["label"] in _FC and n.get("start_line")]
                if any(not any(a <= L <= b for a, b in spans) for L in lines):
                    mods.add(rel)
            lab = {n["id"]: n["label"] for n in nodes}
            orc = get_oracle(diff, nodes) if nodes else {"nodes": set(), "renames": {}}
            ren = orc["renames"]                               # file đổi tên: id theo tên mới (khớp snapshot sau)
            touch = sorted(ren[k.split(":", 1)[0]] + ":" + k.split(":", 1)[1] if k.split(":", 1)[0] in ren else k
                           for k in orc["nodes"] if lab.get(k) in _FC)
            f.write(json.dumps({"h": h, "p": p, "t": ct, "merge": merge, "msg": msg[:2000], "touch": touch,
                                "touch_mod": sorted(mods)}, ensure_ascii=False) + "\n")

def load_history(repo):
    p = _hist_path(repo)
    if not p.exists() and HIST_INPUT and (Path(HIST_INPUT) / p.name).exists():
        shutil.copy(Path(HIST_INPUT) / p.name, p)
    units = _hist_units(repo)
    # Số dòng khác số commit nghĩa là lần build trước dừng giữa chừng, nên build lại.
    if not p.exists() or sum(1 for l in open(p, encoding="utf-8") if l.strip()) != len(units):
        build_history(repo, units)
    return {"units": [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]}

HIST = {}
for _repo in SCALE_REPOS:
    HIST[_repo] = load_history(_repo)
    _u = HIST[_repo]["units"]
    print(f"{_repo}: {len(_u)} commit trên nhánh chính | sửa ≥ 1 hàm/lớp: {sum(bool(u['touch']) for u in _u)} | "
          f"chỉ đổi ngoài hàm/lớp: {sum(bool(u['touch_mod']) and not u['touch'] for u in _u)}")

django/django: 22111 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 10736 | chỉ đổi ngoài hàm/lớp: 851
sympy/sympy: 8141 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 6224 | chỉ đổi ngoài hàm/lớp: 520
scikit-learn/scikit-learn: 12437 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 6778 | chỉ đổi ngoài hàm/lớp: 521
sphinx-doc/sphinx: 2920 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 1701 | chỉ đổi ngoài hàm/lớp: 168
matplotlib/matplotlib: 10011 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 6387 | chỉ đổi ngoài hàm/lớp: 778
pytest-dev/pytest: 3411 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 1757 | chỉ đổi ngoài hàm/lớp: 129
pydata/xarray: 3116 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 1777 | chỉ đổi ngoài hàm/lớp: 96
astropy/astropy: 7188 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 3904 | chỉ đổi ngoài hàm/lớp: 437
pylint-dev/pylint: 5399 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 2771 | chỉ đổi ngoài hàm/lớp: 539
psf/requests: 2386 commit trên nhánh chính | sửa ≥ 1 hàm/lớp: 895 | chỉ đổi ngoài hàm/lớp: 2

## 6. Cấu hình LLM-as-a-judge

Cell `judge-config` đặt API key, model và mức thinking của judge. Cell `judge-v4` ở Section 8 dùng các giá trị này.

In [22]:
# ── Section 6: cấu hình LLM-as-a-judge (Gemini API) ─────────────────────────────────────────
!pip install -q -U google-genai
import json
import random
import re
import time
from pathlib import Path

GOOGLE_API_KEY = ""                          # dán API key của Google AI Studio vào đây
JUDGE_MODEL    = "gemini-3.1-flash-lite"     # ổn định, có gói miễn phí
THINKING_LEVEL = "MINIMAL"                   # token thinking tính như token đầu ra → mức thấp nhất; prompt không yêu cầu reason
# Prompt, schema và thư mục nhãn của judge nằm ở cell judge-v4.
print(f"Judge: {JUDGE_MODEL} | thinking={THINKING_LEVEL}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 15.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Judge: gemini-3.1-flash-lite | thinking=MINIMAL


## 8. Thực nghiệm v4: nguồn HIST theo phần lời của commit

Phần này chạy thực nghiệm v4 trên 12 repo của tập test SWE-bench. Kết quả nằm trong `V4_DIR`. Thiết kế đầy đủ nằm ở `planning.md` và `pipeline_design.md`.

Lượt 1 trên Kaggle chạy từ đầu notebook tới hết phần này. Các cell chính của lượt 1 như sau.

- `hist-build` dựng lớp history bằng oracle mới, ghi vào `history/<repo>_history_v2.jsonl`.
- `v4-run` chạy lại truy hồi với oracle mới và lưu 500 id AST cho mỗi instance, ghi vào `scale_rows_v4.jsonl`.
- `hist-proc` dựng chỉ mục hunk, rồi xếp hạng hàm/lớp theo Locus và thêm cột HIST vào `scale_rows_v4.jsonl`.
- `hist-msg` xếp hạng commit theo phần lời của commit (message cùng ticket hoặc PR liên kết) và theo liên kết với các node đầu của AST, rồi thêm cột HIST mới vào `scale_rows_v4.jsonl`. Đây là nguồn HIST chính. HIST theo Locus được giữ làm hệ thống phụ.
- `graph-proc` xếp các hàm/lớp đi tới được qua cạnh của graph, từ 10 node đầu của AST và từ chunk doc, theo công thức của KGCompass, rồi thêm cột GRAPH và DOC vào `scale_rows_v4.jsonl`.

LLM chỉ là judge. Không cell nào trong phần này gọi LLM.

In [23]:
# ── Section 8 (v4): cấu hình ───────────────────────────────────────────────────────────────────────────────
# Kết quả của v4 nằm trong V4_DIR. nltk cần cho bước đưa từ về gốc (Porter) của hist-proc. rapidfuzz tính khoảng cách
# Levenshtein cho graph-proc.
!pip install -q nltk rapidfuzz
V4_DIR     = Path(WORK_DIR) / "v4"               # kết quả của mọi repo
V4_RESULTS = V4_DIR / "scale_rows_v4.jsonl"   # 1 dòng / instance: xếp hạng AST, nguồn DOC, HIST theo Locus và ISSUE
V4_TOPK    = 500                              # số id AST lưu cho mỗi instance, bằng EXP_DEEP → MRR và MAP tính tới hạng 500
LIST_K     = V4_TOPK                          # độ dài danh sách sau khi trộn (v4-fusion, verify-v4)
V4_EVID_N  = 10                               # trần số phần tử của mỗi loại bằng chứng (DOC, HIST, ISSUE), bằng kích thước pool
GITHUB_TOKEN = ""                             # token GitHub cho issue-fetch (mọi repo GitHub); để trống thì chỉ 60 lượt gọi mỗi giờ
ISSUE_CUTOFF = "instance"                     # mốc cắt của ISSUE: "instance" = lúc tạo báo cáo của chính instance, "pr" = created_at
# Xếp hạng node sau hop theo KGCompass (arXiv 2503.21710), cell graph-proc, mục 8b của pipeline_design.md. Giá trị lấy từ
# bài và từ code công khai của KGCompass, không chọn trên dữ liệu của đề tài.
KGC_BETA    = 0.6                             # β, hệ số suy giảm theo đường đi (bài, mục 4.3)
KGC_ALPHA   = 0.3                             # α, trọng số của cosine so với Levenshtein (bài, mục 4.3)
KGC_BUDGET  = 4.0                             # tổng trọng số tối đa của đường đi từ issue (code, max_hops=4)
KGC_W       = {"root": 0.25, "Defines": 0.25, "Documents": 0.5, "Calls": 1.0, "Inherits": 1.0}   # trọng số cạnh (code)
KGC_SEEDS   = 10                              # số node đầu của AST làm điểm xuất phát của GRAPH, bằng k lớn nhất của depth-v4
KGC_SRC_MAX = 20000                           # số ký tự source tối đa đưa vào Levenshtein (code)
V4_DIR.mkdir(parents=True, exist_ok=True)
print(f"V4_DIR: {V4_DIR} | V4_RESULTS: {V4_RESULTS} | V4_TOPK: {V4_TOPK}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.6 MB/s eta 0:00:00:00:01
V4_DIR: /kaggle/working/v4 | V4_RESULTS: /kaggle/working/v4/scale_rows_v4.jsonl | V4_TOPK: 500


In [24]:
# ── Section 8 (v4): truy hồi lại với oracle mới, lưu V4_TOPK id AST mỗi instance (việc 1.2) ───────────────────
# Mỗi instance một dòng trong V4_RESULTS. Số id AST được lưu là V4_TOPK = 500, để MRR và MAP tính tới hạng 500. Gold dùng
# oracle mới (cell 1dc937a9). Nguồn DOC là vote_ast. Chạy lại thì tiếp tục từ instance chưa có.
import time
from tqdm.auto import tqdm

SCALE_TOPK = V4_TOPK                                          # scale_row đọc hằng số này khi lưu ast_ids
V4_RANK_RULE = "rrf-tie-id"   # cách xếp hạng AST đang dùng; đổi giá trị này khi sửa cách xếp hạng (search, _rrf ở ec3fc119) hay scale_row
_done = set()
if V4_RESULTS.exists():
    with open(V4_RESULTS, encoding="utf-8") as f:
        _old = [json.loads(l) for l in f if l.strip()]
    if any("skip" not in r and ("doc_cand" not in r or "gold_doc" not in r or r.get("rank_rule") != V4_RANK_RULE)
           for r in _old):
        # File tính theo cách cũ: trước bước A (mục 14.2 planning.md, thiếu doc_cand, doc_cand_all, hist_cand), trước phân tích
        # độ phủ doc (thiếu gold_doc, doc_cov; mục 10.2 pipeline_design.md), hoặc theo cách xếp hạng AST cũ (rank_rule khác).
        # v4-run và hist-proc bỏ qua dòng đã có, nên đổi tên file cũ (giữ để so sánh) và tính lại mọi instance.
        _bak = V4_RESULTS.with_name(f"{V4_RESULTS.stem}_cu_{time.strftime('%Y%m%d_%H%M%S')}.jsonl")
        V4_RESULTS.replace(_bak)
        print(f"{V4_RESULTS.name} tính theo cách cũ (thiếu doc_cand, gold_doc hoặc rank_rule ≠ {V4_RANK_RULE!r}) → đổi tên thành "
              f"{_bak.name}, tính lại mọi instance")
        _old = []
    _done = {r["Instance"] for r in _old}
_todo = [i for i in SCALE_INSTANCES if i["id"] not in _done]
print(f"Đã có {len(_done)} instance trong {V4_RESULTS} | còn {len(_todo)}")

_t0, _cur = time.perf_counter(), None
_pbar = tqdm(_todo, desc="v4-run", unit="inst")
for inst in _pbar:
    iid = inst["id"].split("__", 1)[-1]
    _pbar.set_postfix_str(iid)
    if _cur != (inst["repo"], inst["snapshot"]):              # snapshot mới: build (nếu thiếu) + dựng index 1 lần
        _cur = (inst["repo"], inst["snapshot"])
        of = build_graph(inst["snapshot"], repo_path=repo_path_of(inst["repo"]))
        idx = load_graph_indices(of)
        d2c, fwd = graph_edges(of)
    orc = get_oracle_snapshot(inst, idx)
    row = {"Instance": inst["id"], "Repo": inst["repo"], "version": inst["version"],
           "snapshot": inst["snapshot"][:8], "lite": inst["lite"], "gold_dropped": orc["dropped"], "rank_rule": V4_RANK_RULE}
    if not any(idx["node_by_id"][g]["label"] in _FC for g in orc["nodes"]):
        row["skip"] = "không có gold Function/Class trong graph snapshot"
    else:
        c = {"inst": inst, "iid": iid, "idx": idx, "nbi": idx["node_by_id"], "orc": orc,
             "gold": orc["nodes"], "q": inst["text"], "d2c": d2c, "fwd": fwd}
        row.update(scale_row(c))
    with open(V4_RESULTS, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Xong {len(_todo)} instance trong {time.perf_counter() - _t0:.0f}s → {V4_RESULTS}")

Đã có 2294 instance trong /kaggle/working/v4/scale_rows_v4.jsonl | còn 0


v4-run: 0inst [00:00, ?inst/s]

Xong 0 instance trong 0s → /kaggle/working/v4/scale_rows_v4.jsonl


In [25]:
# ── Section v4: nguồn HIST theo Locus (Wen, Wu, Cheung, ASE 2016), ở mức hàm/lớp ─────────────────────────────────
# Tài liệu = 1 hunk (dòng đổi, 3 dòng ngữ cảnh, commit message) của đơn vị history là tổ tiên của snapshot.
#   score(b, f) = max_{h ∈ H(f)} {NL(b, h) + α·CE(b, h)} + β1·Fix(f)             Locus (7), file s → hàm/lớp f
#   NL, CE      = cosine trong VSM, trọng số w_td = (log f_td + 1)·log(N / n_t)   Locus (1)–(4)
#   α           = min(1, λ·#tên code / #token đã tách của issue), λ = 5          Locus (9)
#   Fix(f)      = Σ 1 / (1 + e^(−12·t_i + 12)) trên commit sửa lỗi đã sửa f      Locus (5), β1 = 0,1
# Chỗ chuyển đổi (planning.md mục 6.4): A1 file → hàm/lớp, H(f) theo oracle mới; A2 commit sửa lỗi = message khớp
# fix|bug (AmaLgam, quy tắc 1); A3 t: 0 = đơn vị cũ nhất của file history, 1 = created_at của instance; A4 tài liệu =
# hunk của đơn vị là tổ tiên của snapshot, IDF riêng từng snapshot; A5 file .py có trong graph, git diff -U3 -w, bỏ hunk
# chỉ đổi comment; A6 query = problem_statement; A7 từ giống code: có _ hoặc ., có chữ hoa sau ký tự đầu, nằm trong
# backtick, hoặc đứng ngay trước "(".
# Cần chạy sau hist-build (HIST, HIST_DIR) và v4-run (V4_RESULTS có ast_ids, gold_ids).
import functools
import keyword
import math
import re
import shutil
import scipy.sparse as sp
import unidiff
from collections import Counter
from itertools import groupby
from datasets import load_dataset
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

LOCUS_LAMBDA = 5          # Locus: λ tốt trong 3..7, bài đặt 5
LOCUS_BETA1  = 0.1        # Locus: β1 tốt trong 0,05..0,15 ở mức file, bài đặt 0,1
LOCUS_TOPN   = 500        # số hàm/lớp đầu được lưu, bằng độ sâu xếp hạng AST (EXP_DEEP)
LOCUS_SRC    = "hist_locus"
_BUGFIX_RE   = re.compile(r"fix|bug", re.I)                                       # AmaLgam, quy tắc 1
_NL_STOP     = set(ENGLISH_STOP_WORDS) | {k.lower() for k in keyword.kwlist}      # stop word + từ khóa Python
_stem        = functools.lru_cache(maxsize=None)(PorterStemmer().stem)
_IDENT_RE    = re.compile(r"`[^`\n]+`|[A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*\(?")
_NAME_RE     = re.compile(r"[A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*")

def _nl_terms(txt):
    """Token NL: tách định danh (camelCase, snake_case), bỏ stop word và từ khóa, đưa về gốc bằng Porter."""
    return [_stem(t) for t in _tokenize_code(txt) if t not in _NL_STOP]

def _code_terms(txt):
    """Từ giống code (A7), giữ nguyên không tách. Từ có dấu chấm cho thêm từng phần: 'QuerySet.filter' →
    'QuerySet.filter', 'QuerySet', 'filter'. Việc giữ từ nào khớp tên trong graph làm ở bước dựng chỉ mục."""
    out = []
    for m in _IDENT_RE.finditer(txt):
        t = m.group(0)
        if t.startswith("`"):
            toks = _NAME_RE.findall(t)
        else:
            call, t = t.endswith("("), t.rstrip("(")
            toks = [t] if call or "_" in t or "." in t or any(c.isupper() for c in t[1:]) else []
        for x in toks:
            out.append(x)
            if "." in x:
                out.extend(x.split("."))
    return out

def _hunk_body(text):
    """Nội dung hunk (dòng ngữ cảnh, dòng xóa, dòng thêm), bỏ dòng tiêu đề @@ và ký tự đầu dòng."""
    return "\n".join(l[1:] for l in text.split("\n")[1:])

def _comment_only(h):
    return all(not t or t.startswith("#") for t in (ln.value.strip() for ln in h if ln.is_added or ln.is_removed))

def _hunks_path(repo):
    return HIST_DIR / f"{repo.split('/')[1]}_hunks_v2.jsonl"

def build_hunks(repo):
    """1 dòng / hunk: u = sha của đơn vị, f = file (tên sau đổi tên), k = thứ tự hunk trong file, ids = hàm/lớp mà
    hunk đổi (oracle mới, id theo tên file mới như touch; rỗng nếu chỉ đổi ở cấp module), text = hunk dạng diff.
    Chỉ duyệt đơn vị có touch khác rỗng: đơn vị khác không đổi hàm/lớp nào trong file có ở graph. Duyệt xong mới ghi file,
    nên lần dựng dừng giữa chừng không để lại file thiếu."""
    rp, out = repo_path_of(repo), []
    units = [u for u in HIST[repo]["units"] if u["touch"]]
    for u in tqdm(units, desc=f"hunk {repo}", unit="commit"):
        diff = _git(rp, "diff", "-U3", "-w", u["p"], u["h"], "--", "*.py")
        if not diff:
            continue
        for pf in unidiff.PatchSet(diff):
            src = _src_path(pf)
            if pf.is_added_file or not src.endswith(".py") or _skip_rel(src):
                continue
            nodes = _file_nodes_at(rp, u["p"], src)          # node của file tại commit cha, như khi tính touch
            if not nodes:
                continue
            lab = {x["id"]: x["label"] for x in nodes}
            for k, h in enumerate(pf):
                if _comment_only(h):
                    continue
                one = f"--- a/{src}\n+++ b/{pf.path}\n{h}"
                ids = sorted(pf.path + x[len(src):] for x in get_oracle(one, nodes)["nodes"] if lab.get(x) in _FC)
                out.append(json.dumps({"u": u["h"], "f": pf.path, "k": k, "ids": ids, "text": str(h)},
                                      ensure_ascii=False) + "\n")
    with open(_hunks_path(repo), "w", encoding="utf-8") as f:
        f.writelines(out)
    return len(out)

def load_hunks(repo):
    p = _hunks_path(repo)
    if not p.exists() and HIST_INPUT and (Path(HIST_INPUT) / p.name).exists():
        shutil.copy(Path(HIST_INPUT) / p.name, p)
    # File hunk cũ hơn file history nghĩa là history vừa được build lại, nên dựng lại hunk.
    if not p.exists() or p.stat().st_mtime < _hist_path(repo).stat().st_mtime:
        build_hunks(repo)
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

def _counts(docs, fn):
    """Ma trận đếm (hàng = tài liệu, cột = từ) và từ điển cột."""
    vocab, r, c, v = {}, [], [], []
    for i, d in enumerate(docs):
        for t, k in Counter(fn(d)).items():
            r.append(i); c.append(vocab.setdefault(t, len(vocab))); v.append(k)
    return sp.csr_matrix((v, (r, c)), shape=(len(docs), len(vocab)), dtype=np.float64), vocab

def locus_corpus(repo):
    """Chỉ mục của 1 repo, dùng chung cho mọi snapshot: đếm NL và CE của từng hunk, ma trận hunk × hàm/lớp (H(f)),
    ma trận đơn vị sửa lỗi × hàm/lớp (cho Fix) cùng thời điểm của chúng."""
    H = HIST[repo]
    msg = {u["h"]: u["msg"] for u in H["units"]}
    hunks = load_hunks(repo)
    docs = [_hunk_body(x["text"]) + "\n" + msg[x["u"]] for x in tqdm(hunks, desc=f"văn bản {repo}", unit="hunk")]
    Xnl, nl_vocab = _counts(docs, _nl_terms)
    Xce, ce_vocab = _counts(docs, _code_terms)
    fn_vocab, r, c = {}, [], []
    for i, x in enumerate(hunks):
        for k in x["ids"]:
            r.append(i); c.append(fn_vocab.setdefault(k, len(fn_vocab)))
    bf = [u for u in H["units"] if u["touch"] and _BUGFIX_RE.search(u["msg"])]
    fr, fc = [], []
    for i, u in enumerate(bf):
        for k in u["touch"]:
            fr.append(i); fc.append(fn_vocab.setdefault(k, len(fn_vocab)))
    nf = len(fn_vocab)
    return {"hunks": hunks, "unit": np.array([x["u"] for x in hunks]), "Xnl": Xnl, "nl_vocab": nl_vocab,
            "Xce": Xce, "ce_vocab": ce_vocab, "fn_vocab": fn_vocab, "fn_keys": sorted(fn_vocab, key=fn_vocab.get),
            "M": sp.csr_matrix((np.ones(len(r)), (r, c)), shape=(len(hunks), nf)),
            "bf_unit": np.array([u["h"] for u in bf]), "bf_t": np.array([u["t"] for u in bf], dtype=np.float64),
            "F": sp.csr_matrix((np.ones(len(fr)), (fr, fc)), shape=(len(bf), nf)),
            "t0": float(min(u["t"] for u in H["units"]))}

def _tfidf(X):
    """Locus (2): w = (log f + 1)·log(N / n_t), N và n_t tính trên đúng các tài liệu đưa vào; hàng chuẩn hóa L2."""
    X = X.tocsr().copy(); X.eliminate_zeros()           # bỏ ô 0 do che cột CE
    N = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1])
    idf = np.zeros(X.shape[1]); nz = df > 0; idf[nz] = np.log(N / df[nz])
    W = X.copy(); W.data = np.log(W.data) + 1.0
    W = (W @ sp.diags(idf)).tocsr()
    norm = np.sqrt(np.asarray(W.power(2).sum(axis=1)).ravel()); norm[norm == 0] = 1.0
    return sp.diags(1.0 / norm) @ W, idf

def _qvec(terms, vocab, idf):
    v = np.zeros(len(idf))
    for t, k in Counter(terms).items():
        j = vocab.get(t)
        if j is not None:
            v[j] = (math.log(k) + 1.0) * idf[j]
    n = np.linalg.norm(v)
    return v / n if n else v

def locus_snapshot(C, repo, snapshot, names):
    """Phần của 1 snapshot (A4): hunk và đơn vị sửa lỗi là tổ tiên của snapshot; IDF tính trên các hunk đó; CE chỉ
    giữ từ khớp tên module, lớp, hàm trong graph của snapshot (names)."""
    anc = set(_git(repo_path_of(repo), "rev-list", snapshot).split())
    rows = np.flatnonzero([u in anc for u in C["unit"]])
    Wnl, idf_nl = _tfidf(C["Xnl"][rows])
    mask = np.zeros(len(C["ce_vocab"])); mask[[j for t, j in C["ce_vocab"].items() if t in names]] = 1.0
    Wce, idf_ce = _tfidf((C["Xce"][rows] @ sp.diags(mask)).tocsr())
    bfr = np.flatnonzero([u in anc for u in C["bf_unit"]])
    return {"rows": rows, "Wnl": Wnl, "idf_nl": idf_nl, "Wce": Wce, "idf_ce": idf_ce, "names": names,
            "M": C["M"][rows].tocsc(), "F": C["F"][bfr], "bf_t": C["bf_t"][bfr]}

def locus_rank(C, S, q, t_issue):
    """Điểm Locus của mọi hàm/lớp cho 1 issue, ở ba cấu hình của phân tích thành phần (Locus, RQ4): chỉ NL, NL + α·CE,
    và đủ NL + α·CE + β1·Fix. Trả (dict cấu hình → điểm theo cột hàm/lớp, điểm từng hunk của NL + α·CE, điểm từng hunk
    của NL, α)."""
    ce_q = [t for t in _code_terms(q) if t in S["names"]]
    alpha = min(1.0, LOCUS_LAMBDA * len(ce_q) / max(1, len(_tokenize_code(q))))
    nl = S["Wnl"] @ _qvec(_nl_terms(q), C["nl_vocab"], S["idf_nl"])
    s = nl + alpha * (S["Wce"] @ _qvec(ce_q, C["ce_vocab"], S["idf_ce"]))
    best = lambda v: (sp.diags(v) @ S["M"]).max(axis=0).toarray().ravel()   # max trên các hunk của từng hàm/lớp
    t = np.clip((S["bf_t"] - C["t0"]) / max(1.0, t_issue - C["t0"]), 0.0, 1.0)
    fix = S["F"].T @ (1.0 / (1.0 + np.exp(-12.0 * t + 12.0)))
    b = best(s)
    return {"nl": best(nl), "nlce": b, "full": b + LOCUS_BETA1 * fix}, s, nl, alpha

def _put(row, src, sc, keep, keys, rk):
    """Ghi danh sách của 1 cấu hình vào dòng kết quả, cùng dạng với nguồn DOC của scale_row. Trả thứ tự cột đã chọn."""
    sc = np.where(keep, sc, 0.0)
    order = [j for j in np.lexsort((keys, -sc)) if sc[j] > 0][:LOCUS_TOPN]
    ids = [keys[j] for j in order]
    pos = {x: i + 1 for i, x in enumerate(ids)}
    row[f"doc_ids_{src}"] = ids
    row[f"doc_score_{src}"] = [round(float(sc[j]), 6) for j in order]
    row[f"doc_r_{src}"] = [rk.get(x, 0) for x in ids]
    row[f"doc_gold_{src}"] = [pos.get(g) for g in row["gold_ids"]]
    return order

def _graph_names(nodes):
    """Tên dùng cho CE: tên lớp và hàm, cùng tên module dạng chấm (django/db/models/query.py → django.db.models.query)."""
    out = {n["name"] for n in nodes if n["label"] in _FC and n.get("name")}
    for n in nodes:
        if n["label"] == "Module" and n.get("name", "").endswith(".py"):
            out.add(n["name"][:-3].replace("/", ".").removesuffix(".__init__"))
    return out

_created = {r["instance_id"]: pd.Timestamp(r["created_at"]).timestamp()
            for r in load_dataset("princeton-nlp/SWE-bench", split="test") if r["repo"] in SCALE_REPOS}
_inst = {i["id"]: i for i in SCALE_INSTANCES}
_rows = [json.loads(l) for l in open(V4_RESULTS, encoding="utf-8") if l.strip()]
_todo = [r for r in _rows if "skip" not in r and (f"doc_ids_{LOCUS_SRC}" not in r
                                                  or r.get("hist_cand_max_touch") != HIST_MAX_TOUCH)]
print(f"{V4_RESULTS}: {len(_rows)} dòng | cần tính HIST (Locus): {len(_todo)}")
_LC = {}
for (_repo, _snap), _grp in groupby(sorted(_todo, key=lambda r: (r["Repo"], _inst[r["Instance"]]["snap_ct"], _inst[r["Instance"]]["snapshot"])),
                                    key=lambda r: (r["Repo"], _inst[r["Instance"]]["snapshot"])):
    if _repo not in _LC:
        _LC[_repo] = locus_corpus(_repo)
    C, s8 = _LC[_repo], _snap[:8]
    _of = build_graph(_snap, repo_path=repo_path_of(_repo))
    _nodes = json.load(open(graph_summary_path(_of), encoding="utf-8"))["nodes"]
    _fc = {n["id"] for n in _nodes if n["label"] in _FC}
    S = locus_snapshot(C, _repo, _snap, _graph_names(_nodes))
    keep = np.array([f"{s8}:{k}" in _fc for k in C["fn_keys"]])
    keys = np.array(C["fn_keys"])
    _ntouch = {u["h"]: len(u["touch"]) for u in HIST[_repo]["units"]}
    for row in _grp:
        inst = _inst[row["Instance"]]
        scs, s, nl, alpha = locus_rank(C, S, inst["text"], _created[row["Instance"]])
        rk = {x: i + 1 for i, x in enumerate(row["ast_ids"])}
        _put(row, f"{LOCUS_SRC}_nl", scs["nl"], keep, keys, rk)          # phân tích thành phần: chỉ NL
        _put(row, f"{LOCUS_SRC}_nlce", scs["nlce"], keep, keys, rk)      # phân tích thành phần: NL + α·CE
        _put(row, LOCUS_SRC, scs["full"], keep, keys, rk)                # nguồn HIST: NL + α·CE + β1·Fix
        row["hist_locus_alpha"] = round(alpha, 4)
        # Ứng viên bằng chứng HIST kèm điểm gốc, để đặt ngưỡng ngoại tuyến (mục 14.2 planning.md): V4_EVID_N hunk của
        # snapshot có s = NL + α·CE cao nhất. Mỗi phần tử: u = sha 8 ký tự, f = file, k = thứ tự hunk trong file,
        # nl = NL, ce = CE (cosine, chưa nhân α; α ở hist_locus_alpha), ids = hàm/lớp mà hunk sửa (theo file hunks).
        # Bỏ hunk của commit sửa hơn HIST_MAX_TOUCH hàm/lớp (cùng ngưỡng với hist-build; ROSE bỏ transaction chạm hơn 30
        # thực thể vì là nhiễu). Các commit này chỉ là 1 đến 2% số commit nhưng chứa 25 đến 39% số hunk, và mỗi hunk kéo theo
        # danh sách hàm/lớp rất dài khi đưa cho judge. Xếp hạng hàm/lớp của HIST ở trên không đổi.
        row["hist_cand"] = []
        for h in np.argsort(-s, kind="stable"):
            if len(row["hist_cand"]) == V4_EVID_N or s[h] <= 0:
                break
            hk = C["hunks"][S["rows"][h]]
            if _ntouch.get(hk["u"], 0) > HIST_MAX_TOUCH:
                continue
            row["hist_cand"].append({"u": hk["u"][:8], "f": hk["f"], "k": hk["k"], "nl": round(float(nl[h]), 6),
                                     "ce": round(float((s[h] - nl[h]) / alpha), 6) if alpha else 0.0,
                                     "ids": hk["ids"]})
        row["hist_cand_max_touch"] = HIST_MAX_TOUCH
_tmp = Path(str(V4_RESULTS) + ".tmp")
with open(_tmp, "w", encoding="utf-8") as f:
    for r in _rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
_tmp.replace(V4_RESULTS)

/kaggle/working/v4/scale_rows_v4.jsonl: 2294 dòng | cần tính HIST (Locus): 0


PosixPath('/kaggle/working/v4/scale_rows_v4.jsonl')

In [26]:
# ── Section 8 (v4): issue-fetch, tải báo cáo lỗi cũ cho nguồn ISSUE (mục 6.5 và 14.3 planning.md) ────────────────────
# Django: ticket Trac có resolution = fixed, tải bằng CSV hàng loạt từ code.djangoproject.com/query, mỗi trang 500 ticket.
# Repo GitHub (SymPy và 10 repo còn lại của tập test): issue và PR đã đóng qua GitHub REST API (/repos/<repo>/issues?state=closed). Endpoint này
# trả cả issue lẫn PR, kèm body. Có GITHUB_TOKEN (v4-config) thì GitHub cho 5.000 lượt gọi mỗi giờ. Không có token thì chỉ
# 60 lượt mỗi giờ: cell tự chờ tới lúc GitHub mở lại, và việc tải mất gần 5 giờ.
# Kết quả: V4_DIR/issues/<repo>_reports.jsonl, 1 dòng / báo cáo: n = số ticket hoặc số issue, pr = mục là PR (chỉ repo GitHub),
# title, body, created = thời điểm tạo (Unix time), type = loại ticket (chỉ Django). Trac không ghi múi giờ nên được coi
# là UTC; issue-index chừa 1 ngày đệm. File đã có thì không tải lại. Tải dở thì phần đã tải nằm ở <file>.part, và lần
# chạy sau đi tiếp từ trang kế tiếp. GitHub chỉ nhận tham số page cho 1.000 mục đầu (trang 11 trở đi trả lỗi 422), nên
# repo GitHub đi theo link "next" trong header Link (phân trang bằng con trỏ after); link kế tiếp được ghi ở <file>.next.
import csv
import io
import re
import time
import urllib.error
import urllib.request
from datetime import datetime, timezone

ISSUE_DIR = V4_DIR / "issues"
ISSUE_DIR.mkdir(parents=True, exist_ok=True)
_TRAC_PAGE = 500
_TRAC_URL  = ("https://code.djangoproject.com/query?status=closed&resolution=fixed&format=csv&order=id"
              f"&max={_TRAC_PAGE}&col=id&col=summary&col=description&col=type&col=time")
_GH_PAGE   = 100
def _gh_url(repo):
    return (f"https://api.github.com/repos/{repo}/issues?state=closed&per_page={_GH_PAGE}"
            "&sort=created&direction=asc")
csv.field_size_limit(10 ** 8)

def _issues_path(repo):
    return ISSUE_DIR / f"{repo.split('/')[1]}_reports.jsonl"

def _http(url, headers=None):
    """GET, thử lại khi lỗi mạng hoặc lỗi máy chủ. Trả (văn bản, header); (None, {}) khi trang không tồn tại."""
    req = urllib.request.Request(url, headers={"User-Agent": "repograph-research", **(headers or {})})
    for attempt in range(8):
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read().decode("utf-8-sig", errors="replace"), dict(r.headers)
        except urllib.error.HTTPError as e:
            if e.code in (400, 404):                  # Trac trả lỗi khi số trang vượt quá số ticket
                return None, {}
            if e.code in (403, 429):                  # GitHub hết lượt gọi: chờ tới lúc mở lại
                ra, reset = e.headers.get("Retry-After"), e.headers.get("X-RateLimit-Reset")
                wait = max(5.0, float(ra) if ra else (int(reset) - time.time() + 5 if reset else 60.0))
                print(f"  GitHub trả {e.code} (hết lượt gọi), chờ {wait / 60:.1f} phút")
                time.sleep(wait)
                continue
            if e.code < 500:
                raise
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            pass
        time.sleep(min(120, 5 * 2 ** attempt))
    raise RuntimeError(f"không tải được {url}")

def _trac_rows(txt):
    """Một trang CSV của Trac → dòng của file báo cáo. Cột Created có dạng 'Jul 13, 2005, 2:03:27 PM'."""
    out = []
    for r in csv.DictReader(io.StringIO(txt)):
        r = {k.lower(): v for k, v in r.items()}
        t = datetime.strptime(r["created"], "%b %d, %Y, %I:%M:%S %p").replace(tzinfo=timezone.utc).timestamp()
        out.append({"n": int(r["id"]), "pr": False, "title": r["summary"], "body": r.get("description") or "",
                    "created": int(t), "type": r.get("type", "")})
    return out

def _gh_rows(txt):
    """Một trang JSON của GitHub → dòng của file báo cáo."""
    return [{"n": it["number"], "pr": "pull_request" in it, "title": it["title"] or "", "body": it.get("body") or "",
             "created": int(datetime.strptime(it["created_at"], "%Y-%m-%dT%H:%M:%SZ")
                            .replace(tzinfo=timezone.utc).timestamp())}
            for it in json.loads(txt)]

def fetch_reports(repo):
    """Tải mọi trang của 1 repo vào <file>.part (mỗi trang ghi một lần), rồi bỏ mục trùng số và đổi tên thành file chính."""
    path = _issues_path(repo)
    part, nxt = Path(str(path) + ".part"), Path(str(path) + ".next")
    django = repo == "django/django"
    size, parse = (_TRAC_PAGE, _trac_rows) if django else (_GH_PAGE, _gh_rows)
    hd = {} if django else {"Accept": "application/vnd.github+json",
                             **({"Authorization": f"Bearer {GITHUB_TOKEN}"} if GITHUB_TOKEN else {})}
    done = [l for l in open(part, encoding="utf-8")] if part.exists() else []
    if django:
        done = done[:len(done) // size * size]       # chỉ giữ các trang đã đủ
        url = f"{_TRAC_URL}&page={len(done) // size + 1}"
    elif done and nxt.exists() and nxt.read_text(encoding="utf-8").strip():
        url = nxt.read_text(encoding="utf-8").strip()     # đi tiếp từ link kế tiếp đã ghi
    else:
        done, url = [], _gh_url(repo)                # không có link kế tiếp (vd file .part của bản cũ): tải lại từ đầu
    with open(part, "w", encoding="utf-8") as f:
        f.writelines(done)
    page, t0 = len(done) // size + 1, time.time()
    while url:
        txt, h = _http(url, hd)
        rows = parse(txt) if txt else []
        with open(part, "a", encoding="utf-8") as f:
            f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)
        if django:
            url = f"{_TRAC_URL}&page={page + 1}" if len(rows) == size else None
        else:
            m = re.search(r'<([^>]+)>;\s*rel="next"', h.get("Link") or h.get("link") or "")
            url = m.group(1) if m and rows else None
            nxt.write_text(url or "", encoding="utf-8")
        if page % 10 == 0 or not url:
            left = h.get("X-RateLimit-Remaining") or h.get("x-ratelimit-remaining")
            print(f"  {repo}: trang {page}, {(page - 1) * size + len(rows)} mục, {time.time() - t0:.0f}s"
                  + (f", còn {left} lượt gọi" if left else ""))
        page += 1
        if url:
            time.sleep(1.0)
    rows = {}
    for l in open(part, encoding="utf-8"):
        r = json.loads(l)
        rows[r["n"]] = r                              # trang lệch do có mục mới đóng trong lúc tải: giữ bản sau cùng
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for _, r in sorted(rows.items()))
    part.unlink()
    nxt.unlink(missing_ok=True)
    return len(rows)

for _repo in SCALE_REPOS:
    if _issues_path(_repo).exists():
        print(f"{_repo}: đã có {_issues_path(_repo)}")
    else:
        if _repo != "django/django" and not GITHUB_TOKEN:
            print(f"{_repo}: chưa đặt GITHUB_TOKEN trong v4-config → 60 lượt gọi mỗi giờ, việc tải có thể mất nhiều giờ")
        print(f"{_repo}: tải xong {fetch_reports(_repo)} báo cáo → {_issues_path(_repo)}")
    _r = [json.loads(l) for l in open(_issues_path(_repo), encoding="utf-8")]
    print(f"  {len(_r)} mục | là PR: {sum(x['pr'] for x in _r)} | tạo từ "
          f"{datetime.fromtimestamp(min(x['created'] for x in _r), timezone.utc).date()} tới "
          f"{datetime.fromtimestamp(max(x['created'] for x in _r), timezone.utc).date()}")

django/django: đã có /kaggle/working/v4/issues/django_reports.jsonl
  19388 mục | là PR: 0 | tạo từ 2005-07-13 tới 2026-10-15
sympy/sympy: đã có /kaggle/working/v4/issues/sympy_reports.jsonl
  24139 mục | là PR: 14532 | tạo từ 2007-01-05 tới 2026-10-20
scikit-learn/scikit-learn: đã có /kaggle/working/v4/issues/scikit-learn_reports.jsonl
  31813 mục | là PR: 21015 | tạo từ 2010-08-31 tới 2026-11-01
sphinx-doc/sphinx: đã có /kaggle/working/v4/issues/sphinx_reports.jsonl
  13034 mục | là PR: 6657 | tạo từ 2015-01-02 tới 2026-10-23
matplotlib/matplotlib: đã có /kaggle/working/v4/issues/matplotlib_reports.jsonl
  30964 mục | là PR: 20569 | tạo từ 2011-02-19 tới 2026-10-31
pytest-dev/pytest: đã có /kaggle/working/v4/issues/pytest_reports.jsonl
  13223 mục | là PR: 7467 | tạo từ 2010-10-10 tới 2026-11-01
pydata/xarray: đã có /kaggle/working/v4/issues/xarray_reports.jsonl
  9453 mục | là PR: 5373 | tạo từ 2013-11-24 tới 2026-10-28
astropy/astropy: đã có /kaggle/working/v4/issues/astropy_report

In [27]:
# ── Section 8 (v4): issue-index, nguồn ISSUE theo BugLocator (Zhou, Zhang, Lo, ICSE 2012), ở mức hàm/lớp ──────────────
# Tài liệu = 1 báo cáo lỗi cũ (issue-fetch): ticket Trac có resolution = fixed (Django), issue GitHub đã đóng (repo GitHub, không
# tính PR). Văn bản = title + body. Liên kết báo cáo → đơn vị history (hist-build); mỗi cặp (báo cáo, đơn vị) có một loại:
#   Django: message có "Fixed #N" (loại fixed) hoặc "Refs #N" (loại refs), kể cả dạng danh sách "Fixed #123, #456". Theo
#           hướng dẫn commit của Django, "Fixed #N" đóng ticket, còn "Refs #N" chỉ nhắc tới ticket. Cặp có cả hai là fixed.
#   GitHub: PR có từ khóa đóng issue của GitHub (close, fix, resolve và các dạng chia) trong body nối tới đơn vị của PR đó:
#           merge commit "Merge pull request #PR", hoặc commit squash có "(#PR)" ở cuối dòng đầu (scikit-learn, xarray,
#           pylint, seaborn merge kiểu này). Đơn vị có từ khóa đóng issue ngay trong message cũng được dùng. Từ khóa chỉ
#           nhận số issue của chính repo ("#N", "<repo>#N" hoặc URL issue của repo). Mọi cặp là fixed.
# Fix(b′) = hợp touch của các đơn vị đã nối và là tổ tiên của snapshot. Hai biến thể: chỉ từ liên kết fixed (cột doc_*_issue,
# issue_cand; hệ thống chính trong lúc chờ chốt) và từ fixed cộng refs (cột doc_*_issue_refs, issue_cand_refs; để so ngoại
# tuyến). Với repo GitHub hai biến thể trùng nhau.
# SimiScore(f) = Σ sim(b, b′) / |Fix(b′)|, tổng trên các báo cáo cũ b′ có f ∈ Fix(b′); sim = cosine TF-IDF giữa issue và b′,
# không có ngưỡng. TF-IDF: tách bằng _tokenize_code, bỏ stop word tiếng Anh, đưa về gốc bằng Porter; TF dạng log
# (sublinear_tf, như rVSM của BugLocator); IDF tính riêng cho từng snapshot, trên các báo cáo tạo trước thời điểm snapshot.
# Lọc rò rỉ (mục 6.5 planning.md): (1) báo cáo tạo trước mốc cắt ít nhất 1 ngày; (2) commit sửa là tổ tiên của snapshot;
# (3) bỏ báo cáo của chính instance: summary hoặc title trùng dòng đầu của problem statement (sau khi chuẩn hóa), và với repo GitHub
# thêm các issue mà PR của instance đóng; (4) báo cáo có Fix(b′) rỗng không đóng góp.
# Mốc cắt ISSUE_CUTOFF (v4-config): "instance" = thời điểm tạo báo cáo của chính instance (báo cáo mới nhất trong các báo cáo
# của instance tạo trước created_at; không nhận ra báo cáo nào thì dùng created_at); "pr" = created_at của SWE-bench.
# Output mỗi dòng: doc_ids/doc_score/doc_r/doc_gold_issue (và _issue_refs) cùng dạng với nguồn DOC, V4_TOPK hàm/lớp đầu có trong
# graph snapshot; issue_cand (và issue_cand_refs) = V4_EVID_N báo cáo cũ có sim cao nhất và Fix(b′) khác rỗng, mỗi phần tử
# gồm n = số báo cáo, cos = sim, fix = Fix(b′), links = loại liên kết đã dùng. Cặp liên kết ghi ở V4_DIR/issues/<repo>_links.jsonl.
# Cần chạy sau hist-proc (dùng _stem, _counts, _put, _created, _inst) và issue-fetch (ISSUE_DIR, _issues_path).
# Mỗi dòng ghi issue_sig, dấu của cách tính: ISSUE_CUTOFF, V4_EVID_N, LOCUS_TOPN và cây cú pháp của cell này (sửa chú thích
# không làm đổi dấu). Dòng chưa có ISSUE hoặc có dấu khác dấu hiện tại được tính lại, nên đổi cấu hình hay sửa cách tính
# thì không còn cột ISSUE cũ.
import ast
import hashlib
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfTransformer

ISSUE_SRC = "issue"
ISSUE_VARIANTS = {ISSUE_SRC: {"fixed"}, f"{ISSUE_SRC}_refs": {"fixed", "refs"}}   # biến thể → loại liên kết được dùng
_DAY = 86400
_DJ_LINK_RE  = re.compile(r"\b(fixed|refs)\s+((?:#\d+(?:\s*(?:,|and)\s*)?)+)", re.I)
_MERGE_RE    = re.compile(r"Merge pull request #(\d+)")
_SQUASH_RE   = re.compile(r"\(#(\d+)\)\s*$")       # dòng đầu của commit squash: "Fix X (#12345)"

def _gh_close_re(repo):
    """Từ khóa đóng issue của GitHub, chỉ nhận số issue của chính repo: "#N", "<repo>#N" hoặc URL issue của repo."""
    r = re.escape(repo)
    return re.compile(r"\b(?:close[sd]?|fix(?:e[sd])?|resolve[sd]?)\b\s*:?\s*"
                      rf"(?:https?://github\.com/{r}/issues/|{r}#|#)(\d+)", re.I)
assert ISSUE_CUTOFF in ("instance", "pr"), ISSUE_CUTOFF

def _issue_terms(txt):
    """Token của báo cáo và của issue: tách định danh, bỏ stop word tiếng Anh, đưa về gốc bằng Porter."""
    return [_stem(t) for t in _tokenize_code(txt) if t not in ENGLISH_STOP_WORDS]

def _norm_title(s):
    return re.sub(r"\s+", " ", s.strip().lower())

def issue_links(repo, reps):
    """{(số báo cáo, chỉ số đơn vị trong HIST[repo]["units"]): loại liên kết}."""
    out = {}
    if repo == "django/django":
        for i, u in enumerate(HIST[repo]["units"]):
            for m in _DJ_LINK_RE.finditer(u["msg"]):
                for n in re.findall(r"#(\d+)", m.group(2)):
                    if out.get((int(n), i)) != "fixed":
                        out[(int(n), i)] = m.group(1).lower()
    else:
        body, close = {r["n"]: r["body"] for r in reps if r["pr"]}, _gh_close_re(repo)
        for i, u in enumerate(HIST[repo]["units"]):
            ns = {int(n) for n in close.findall(u["msg"])}
            m = _MERGE_RE.match(u["msg"]) or _SQUASH_RE.search(u["msg"].split("\n", 1)[0])
            if m:
                ns |= {int(n) for n in close.findall(body.get(int(m.group(1)), ""))}
            for n in ns:
                out[(n, i)] = "fixed"
    return out

def issue_corpus(repo):
    """Phần dùng chung cho mọi snapshot của 1 repo: ma trận đếm từ của báo cáo, liên kết, chỉ mục title và body của PR."""
    reps = [json.loads(l) for l in open(_issues_path(repo), encoding="utf-8") if l.strip()]
    docs = [r for r in reps if not r["pr"]]
    pos = {r["n"]: j for j, r in enumerate(docs)}
    X, vocab = _counts(tqdm([r["title"] + "\n" + r["body"] for r in docs], desc=f"báo cáo {repo}", unit="báo cáo"),
                       _issue_terms)
    links = sorted((pos[n], i, ty) for (n, i), ty in issue_links(repo, reps).items() if n in pos)
    units = HIST[repo]["units"]
    with open(ISSUE_DIR / f"{repo.split('/')[1]}_links.jsonl", "w", encoding="utf-8") as f:
        for j, i, ty in links:
            f.write(json.dumps({"n": docs[j]["n"], "u": units[i]["h"], "type": ty}) + "\n")
    fn_vocab = {}
    for _, i, _ in links:
        for k in units[i]["touch"]:
            fn_vocab.setdefault(k, len(fn_vocab))
    by_title = defaultdict(list)
    for j, r in enumerate(docs):
        by_title[_norm_title(r["title"])].append(j)
    return {"docs": docs, "pos": pos, "X": X, "vocab": vocab, "links": links, "fn_vocab": fn_vocab,
            "fn_keys": sorted(fn_vocab, key=fn_vocab.get), "created": np.array([r["created"] for r in docs], dtype=np.float64),
            "by_title": by_title, "pr_body": {r["n"]: r["body"] for r in reps if r["pr"]}}

def issue_snapshot(C, repo, snapshot, snap_ct):
    """Phần của 1 snapshot: TF-IDF với IDF trên báo cáo tạo trước snapshot, và Fix(b′) của từng biến thể (ma trận báo cáo ×
    hàm/lớp, chỉ từ đơn vị là tổ tiên của snapshot)."""
    anc = set(_git(repo_path_of(repo), "rev-list", snapshot).split())
    units = HIST[repo]["units"]
    before = np.flatnonzero(C["created"] < snap_ct)
    cols = np.flatnonzero(np.asarray((C["X"][before] > 0).sum(axis=0)).ravel() > 0)   # từ điển của snapshot
    tr = TfidfTransformer(sublinear_tf=True).fit(C["X"][before][:, cols])
    S = {"cols": cols, "tr": tr, "W": tr.transform(C["X"][:, cols]), "R": {}, "nfix": {}, "types": {}}
    for var, types in ISSUE_VARIANTS.items():
        r, c, ty_of = [], [], defaultdict(set)
        for j, i, ty in C["links"]:
            if ty in types and units[i]["h"] in anc and units[i]["touch"]:
                ty_of[j].add(ty)
                for k in units[i]["touch"]:
                    r.append(j); c.append(C["fn_vocab"][k])
        R = sp.csr_matrix((np.ones(len(r)), (r, c)), shape=(len(C["docs"]), len(C["fn_vocab"])))
        R.data[:] = 1.0                                   # đơn vị khác nhau sửa cùng hàm/lớp chỉ tính 1 lần
        S["R"][var], S["nfix"][var], S["types"][var] = R, np.asarray(R.sum(axis=1)).ravel(), ty_of
    return S

def issue_own(C, repo, inst):
    """Chỉ số các báo cáo của chính instance (lọc 3)."""
    own = set(C["by_title"].get(_norm_title(inst["text"].split("\n", 1)[0]), []))
    if repo != "django/django":
        pr = int(inst["id"].rsplit("-", 1)[1])
        own |= {C["pos"][int(n)] for n in _gh_close_re(repo).findall(C["pr_body"].get(pr, "")) if int(n) in C["pos"]}
    return own

def issue_cutoff(C, inst, own):
    """Mốc cắt thời gian (lọc 1, trước khi trừ 1 ngày đệm) và nguồn của mốc."""
    t_pr = _created[inst["id"]]
    ts = [C["created"][j] for j in own if C["created"][j] <= t_pr]
    if ISSUE_CUTOFF == "instance" and ts:
        return max(ts), "báo cáo của instance"
    return t_pr, "created_at"

def issue_rank(C, S, inst, own):
    """Cosine của issue với mọi báo cáo, và mặt nạ báo cáo qua lọc 1 và 3. Lọc 2 và 4 nằm trong S["R"] và S["nfix"]."""
    qc = Counter(t for t in _issue_terms(inst["text"]) if t in C["vocab"])
    q = sp.csr_matrix((list(qc.values()), ([0] * len(qc), [C["vocab"][t] for t in qc])), shape=(1, len(C["vocab"])))
    sims = (S["W"] @ S["tr"].transform(q[:, S["cols"]]).T).toarray().ravel()
    cut, _src = issue_cutoff(C, inst, own)
    ok = C["created"] <= cut - _DAY
    ok[list(own)] = False
    return sims, ok

def _issue_sig():
    """Dấu của cách tính. Mã của cell lấy từ In[-1] (IPython ghi mã của cell đang chạy vào In trước khi chạy); không có In
    thì dấu chỉ gồm cấu hình."""
    code = (globals().get("In") or [""])[-1]
    try:
        code = ast.dump(ast.parse(code))
    except SyntaxError:
        pass
    return hashlib.sha1(json.dumps([ISSUE_CUTOFF, V4_EVID_N, LOCUS_TOPN, code]).encode("utf-8")).hexdigest()[:12]

_ISSUE_SIG = _issue_sig()
_rows = [json.loads(l) for l in open(V4_RESULTS, encoding="utf-8") if l.strip()]
_todo = [r for r in _rows if "skip" not in r and r.get("issue_sig") != _ISSUE_SIG]
print(f"{V4_RESULTS}: {len(_rows)} dòng | cần tính ISSUE: {len(_todo)} | ISSUE_CUTOFF = {ISSUE_CUTOFF!r} | "
      f"dấu cách tính {_ISSUE_SIG}")
_IC = {}
for (_repo, _snap), _grp in groupby(sorted(_todo, key=lambda r: (r["Repo"], _inst[r["Instance"]]["snap_ct"], _inst[r["Instance"]]["snapshot"])),
                                    key=lambda r: (r["Repo"], _inst[r["Instance"]]["snapshot"])):
    _grp = list(_grp)
    if _repo not in _IC:
        _IC[_repo] = issue_corpus(_repo)
    C, s8 = _IC[_repo], _snap[:8]
    S = issue_snapshot(C, _repo, _snap, _inst[_grp[0]["Instance"]]["snap_ct"])
    _of = build_graph(_snap, repo_path=repo_path_of(_repo))
    _fc = {n["id"] for n in json.load(open(graph_summary_path(_of), encoding="utf-8"))["nodes"] if n["label"] in _FC}
    keep = np.array([f"{s8}:{k}" in _fc for k in C["fn_keys"]], dtype=bool)
    keys = np.array(C["fn_keys"])
    for row in _grp:
        inst = _inst[row["Instance"]]
        own = issue_own(C, _repo, inst)
        sims, ok = issue_rank(C, S, inst, own)
        rk = {x: i + 1 for i, x in enumerate(row["ast_ids"])}
        row["issue_sig"] = _ISSUE_SIG
        for var in ISSUE_VARIANTS:
            nfix = S["nfix"][var]
            elig = ok & (nfix > 0)
            _put(row, var, S["R"][var].T @ np.where(elig, sims / np.maximum(nfix, 1.0), 0.0), keep, keys, rk)
            cand = [j for j in np.argsort(-np.where(elig, sims, -1.0), kind="stable")[:V4_EVID_N] if elig[j] and sims[j] > 0]
            row[var.replace(ISSUE_SRC, "issue_cand", 1)] = [
                {"n": C["docs"][j]["n"], "cos": round(float(sims[j]), 6),
                 "fix": sorted(C["fn_keys"][k] for k in S["R"][var].indices[S["R"][var].indptr[j]:S["R"][var].indptr[j + 1]]),
                 "links": sorted(S["types"][var][j])} for j in cand]
_tmp = Path(str(V4_RESULTS) + ".tmp")
with open(_tmp, "w", encoding="utf-8") as f:
    for r in _rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
_tmp.replace(V4_RESULTS)

/kaggle/working/v4/scale_rows_v4.jsonl: 2294 dòng | cần tính ISSUE: 2214 | ISSUE_CUTOFF = 'instance' | dấu cách tính bf41ef497580


báo cáo astropy/astropy:   0%|          | 0/5624 [00:00<?, ?báo cáo/s]

[264d9677] disk cache hit → /kaggle/working/astropy_graph_264d9677.json
[9626265d] disk cache hit → /kaggle/working/astropy_graph_9626265d.json
[5e5764ed] disk cache hit → /kaggle/working/astropy_graph_5e5764ed.json
[a7141cd9] disk cache hit → /kaggle/working/astropy_graph_a7141cd9.json
[848c8fa2] disk cache hit → /kaggle/working/astropy_graph_848c8fa2.json
[4fc9f31a] disk cache hit → /kaggle/working/astropy_graph_4fc9f31a.json
[69e2fabd] disk cache hit → /kaggle/working/astropy_graph_69e2fabd.json
[0e1d299f] disk cache hit → /kaggle/working/astropy_graph_0e1d299f.json
[b750a0e6] disk cache hit → /kaggle/working/astropy_graph_b750a0e6.json
[b49ad06b] disk cache hit → /kaggle/working/astropy_graph_b49ad06b.json
[43ee5806] disk cache hit → /kaggle/working/astropy_graph_43ee5806.json
[38322105] disk cache hit → /kaggle/working/astropy_graph_38322105.json
[3a0cd2d8] disk cache hit → /kaggle/working/astropy_graph_3a0cd2d8.json
[b3fa7702] disk cache hit → /kaggle/working/astropy_graph_b3fa77

báo cáo django/django:   0%|          | 0/19388 [00:00<?, ?báo cáo/s]

[f9de1972] disk cache hit → /kaggle/working/django_graph_f9de1972.json
[9dcfecb7] disk cache hit → /kaggle/working/django_graph_9dcfecb7.json
[0368d63a] disk cache hit → /kaggle/working/django_graph_0368d63a.json
[af1fa5e7] disk cache hit → /kaggle/working/django_graph_af1fa5e7.json
[a8bb4935] disk cache hit → /kaggle/working/django_graph_a8bb4935.json
[511dfb33] disk cache hit → /kaggle/working/django_graph_511dfb33.json
[6f0b8c1c] disk cache hit → /kaggle/working/django_graph_6f0b8c1c.json
[02cd16a7] disk cache hit → /kaggle/working/django_graph_02cd16a7.json
[76dfa834] disk cache hit → /kaggle/working/django_graph_76dfa834.json
[838e432e] disk cache hit → /kaggle/working/django_graph_838e432e.json
[2aaabe20] disk cache hit → /kaggle/working/django_graph_2aaabe20.json
[df28ebd6] disk cache hit → /kaggle/working/django_graph_df28ebd6.json
[60dc957a] disk cache hit → /kaggle/working/django_graph_60dc957a.json
[44077985] disk cache hit → /kaggle/working/django_graph_44077985.json
[cbe4d

báo cáo matplotlib/matplotlib:   0%|          | 0/10395 [00:00<?, ?báo cáo/s]

... (16 outputs left out)

[0e899793] disk cache hit → /kaggle/working/sphinx_graph_0e899793.json
[b3e26a6c] disk cache hit → /kaggle/working/sphinx_graph_b3e26a6c.json
[55fc0978] disk cache hit → /kaggle/working/sphinx_graph_55fc0978.json
[66e55a02] disk cache hit → /kaggle/working/sphinx_graph_66e55a02.json
[327c1872] disk cache hit → /kaggle/working/sphinx_graph_327c1872.json
[bf260800] disk cache hit → /kaggle/working/sphinx_graph_bf260800.json
[f1af7dc8] disk cache hit → /kaggle/working/sphinx_graph_f1af7dc8.json
[2c2335bb] disk cache hit → /kaggle/working/sphinx_graph_2c2335bb.json
[487b8436] disk cache hit → /kaggle/working/sphinx_graph_487b8436.json
[07983a5a] disk cache hit → /kaggle/working/sphinx_graph_07983a5a.json
[4b452338] disk cache hit → /kaggle/working/sphinx_graph_4b452338.json
[b923165d] disk cache hit → /kaggle/working/sphinx_graph_b923165d.json
[f47bf147] disk cache hit → /kaggle/working/sphinx_graph_f47bf147.json
[d2c8cd3c] disk cache hit → /kaggle/working/sphinx_graph_d2c8cd3c.json
[f0fef

báo cáo sympy/sympy:   0%|          | 0/9607 [00:00<?, ?báo cáo/s]

[496e7761] disk cache hit → /kaggle/working/sympy_graph_496e7761.json
[360290c4] disk cache hit → /kaggle/working/sympy_graph_360290c4.json
[0410a6aa] disk cache hit → /kaggle/working/sympy_graph_0410a6aa.json
[a53c4ad9] disk cache hit → /kaggle/working/sympy_graph_a53c4ad9.json
[479939f8] disk cache hit → /kaggle/working/sympy_graph_479939f8.json
[0cd4f13a] disk cache hit → /kaggle/working/sympy_graph_0cd4f13a.json
[d250ff08] disk cache hit → /kaggle/working/sympy_graph_d250ff08.json
[ae3d62ab] disk cache hit → /kaggle/working/sympy_graph_ae3d62ab.json
[b3934e41] disk cache hit → /kaggle/working/sympy_graph_b3934e41.json
[9bf73cc2] disk cache hit → /kaggle/working/sympy_graph_9bf73cc2.json
[205da797] disk cache hit → /kaggle/working/sympy_graph_205da797.json
[8fc63c2d] disk cache hit → /kaggle/working/sympy_graph_8fc63c2d.json
[9cbea134] disk cache hit → /kaggle/working/sympy_graph_9cbea134.json
[678217d5] disk cache hit → /kaggle/working/sympy_graph_678217d5.json
[6399a809] disk cach

PosixPath('/kaggle/working/v4/scale_rows_v4.jsonl')

In [28]:
# ── Section 8 (v4): hist-msg, nguồn HIST theo phần lời của commit (mục 6.18.3 planning.md, mục 6b pipeline_design.md) ───
# HIST cung cấp phần lời của lịch sử: trước đây cái gì hỏng, vì sao code được sửa, lỗi tương tự được xử lý ra sao. Việc tìm
# code là của AST. Tài liệu = 1 commit c. text(c) = message đầy đủ của c (lấy bằng git log; file history chỉ giữ 2.000 ký tự
# đầu), cộng tiêu đề và mô tả của mọi ticket, issue hoặc PR liên kết với c:
#   Django: ticket có "Fixed #N" trong message (loại fixed).
#   GitHub: issue mà commit đóng, và PR của commit: "Merge pull request #N" hoặc "(#N)" ở cuối dòng đầu (_MERGE_RE,
#           _SQUASH_RE của issue-index). Nội dung PR lấy từ các dòng pr = true của <repo>_reports.jsonl.
#   Liên kết tới ticket và issue lấy bằng issue_links của issue-index, tức cùng các cặp mà issue-index ghi vào
#   V4_DIR/issues/<repo>_links.jsonl. Các cặp này không có PR, nên số PR được tìm lại trong message.
# Commit được xét: tổ tiên của snapshot, sửa ít nhất một hàm/lớp có trong graph snapshot, sửa không quá HIST_MAX_TOUCH hàm/lớp
# (như hist_cand). Rò rỉ: liên kết chỉ vào text(c) khi báo cáo tạo trước mốc cắt của instance ít nhất 1 ngày (issue_cutoff)
# và không phải báo cáo của chính instance (issue_own; repo GitHub bỏ thêm PR của instance). Liên kết bị bỏ thì commit vẫn
# được xét với message.
# TF-IDF như NL của Locus (hist-proc): tách định danh, bỏ stop word và từ khóa, Porter, w = (log f + 1)·log(N / n_t), hàng
# chuẩn hóa L2. IDF tính cho từng snapshot trên text(c) của các commit được xét, với các liên kết tạo trước snapshot ít nhất
# 1 ngày. Với từng instance, commit có liên kết mà bộ lọc của instance cho kết quả khác được tính lại vectơ (IDF giữ nguyên).
# Điểm của commit = RRF gốc với k = HIST_MSG_K (Cormack, Clarke, Büttcher, SIGIR 2009) của hai danh sách:
#   lời:     commit có cosine(issue, text(c)) > 0, xếp theo cosine giảm dần;
#   nối AST: commit sửa ít nhất một hàm/lớp trong ast_ids, xếp theo hạng AST tốt nhất trong các hàm/lớp mà nó sửa.
#   s(c) = 1/(k + hạng lời) + 1/(k + hạng nối AST); commit vắng mặt ở một danh sách không nhận số hạng đó. Hòa trong từng
#   danh sách và ở s(c): commit mới hơn trước, rồi sha.
# Danh sách hist_msg: score(f) = max s(c) trên các commit c đã sửa f (A1 của Locus), LOCUS_TOPN hàm/lớp đầu, cùng dạng với
# nguồn DOC (_put của hist-proc). Không có Fix và CE.
# Pool bằng chứng hist_msg_cand: V4_EVID_N commit có s(c) cao nhất. Mỗi phần tử: h = sha, t = thời điểm commit, s, cos,
# ast_r = hạng AST tốt nhất (None khi không sửa node nào trong ast_ids), touch = hàm/lớp commit sửa (chỉ cho phân tích, không
# đưa vào prompt), links = số của các báo cáo liên kết đã qua bộ lọc.
# Dòng có dấu khác HIST_MSG_CFG được tính lại. Dấu gồm cấu hình và cây cú pháp của cell này (như issue_sig của issue-index),
# nên sửa code thì mọi dòng được tính lại, còn sửa chú thích thì không. Cần chạy sau hist-proc (_nl_terms, _counts, _tfidf,
# _qvec, _put, _created, _inst) và issue-index (file liên kết, issue_own, issue_cutoff), trước v4-fusion.
import ast
import hashlib
from collections import defaultdict
from itertools import groupby
import scipy.sparse as sp

HIST_MSG_SRC = "hist_msg"
HIST_MSG_K   = 60            # k của RRF gốc như bài, bằng V4_RRF_K của v4-fusion (quyết định 57)
_hm_code = (globals().get("In") or [""])[-1]   # mã của cell đang chạy (IPython ghi vào In trước khi chạy), như _issue_sig
try:
    _hm_code = ast.dump(ast.parse(_hm_code))     # cây cú pháp: sửa chú thích không làm đổi dấu
except SyntaxError:
    pass
HIST_MSG_CFG = json.dumps({"k": HIST_MSG_K, "evid_n": V4_EVID_N, "max_touch": HIST_MAX_TOUCH, "top": LOCUS_TOPN,
                           "cutoff": ISSUE_CUTOFF, "code": hashlib.sha1(_hm_code.encode("utf-8")).hexdigest()[:12]},
                          sort_keys=True)

def msg_corpus(repo):
    """Phần dùng chung cho mọi snapshot của 1 repo: commit có touch, ma trận đếm từ của message và của báo cáo được liên kết
    (chung từ điển), ma trận liên kết commit × báo cáo, ma trận commit × hàm/lớp, và chỉ mục báo cáo cho issue_own."""
    full = {h: msg for h, _, _, msg, _ in _hist_units(repo)}                 # message đầy đủ, một lệnh git log
    units = [u for u in HIST[repo]["units"] if u["touch"] and len(u["touch"]) <= HIST_MAX_TOUCH]
    reps = [json.loads(l) for l in open(_issues_path(repo), encoding="utf-8") if l.strip()]
    rep_of = {r["n"]: r for r in reps}
    ui = {u["h"]: i for i, u in enumerate(units)}
    lk = set()
    for (n, i), ty in issue_links(repo, reps).items():       # cùng hàm và cùng bộ lọc với file <repo>_links.jsonl
        h = HIST[repo]["units"][i]["h"]
        if ty == "fixed" and h in ui and n in rep_of and not rep_of[n]["pr"]:
            lk.add((ui[h], n))
    if repo != "django/django":
        for i, u in enumerate(units):
            m = _MERGE_RE.match(full[u["h"]]) or _SQUASH_RE.search(full[u["h"]].split("\n", 1)[0])
            if m and rep_of.get(int(m.group(1)), {}).get("pr"):
                lk.add((i, int(m.group(1))))
    ns = sorted({n for _, n in lk})
    ri = {n: j for j, n in enumerate(ns)}
    X, vocab = _counts(tqdm([full[u["h"]] for u in units] + [rep_of[n]["title"] + "\n" + rep_of[n]["body"] for n in ns],
                            desc=f"lời commit {repo}", unit="văn bản"), _nl_terms)
    X = X.tocsr()
    lr, lc = zip(*sorted((i, ri[n]) for i, n in lk)) if lk else ((), ())
    fn_vocab, tr, tc = {}, [], []
    for i, u in enumerate(units):
        for k in u["touch"]:
            tr.append(i); tc.append(fn_vocab.setdefault(k, len(fn_vocab)))
    docs = [r for r in reps if not r["pr"]]
    by_title = defaultdict(list)
    for j, r in enumerate(docs):
        by_title[_norm_title(r["title"])].append(j)
    h = np.array([u["h"] for u in units])
    return {"h": h, "t": np.array([u["t"] for u in units], dtype=np.float64), "sha_r": np.argsort(np.argsort(h)),
            "touch": [u["touch"] for u in units], "Xm": X[:len(units)], "Xr": X[len(units):], "vocab": vocab,
            "L": sp.csr_matrix((np.ones(len(lr)), (lr, lc)), shape=(len(units), len(ns))),
            "rep_n": np.array(ns, dtype=np.int64), "rep_ct": np.array([rep_of[n]["created"] for n in ns], dtype=np.float64),
            "T": sp.csr_matrix((np.ones(len(tr)), (tr, tc)), shape=(len(units), len(fn_vocab))),
            "fn_keys": sorted(fn_vocab, key=fn_vocab.get),
            # các khóa issue_own và issue_cutoff của issue-index đọc
            "docs": docs, "pos": {r["n"]: j for j, r in enumerate(docs)}, "by_title": by_title,
            "pr_body": {r["n"]: r["body"] for r in reps if r["pr"]},
            "created": np.array([r["created"] for r in docs], dtype=np.float64)}

def _msg_weigh(X, idf):
    """Trọng số Locus (2) với idf cho trước, hàng chuẩn hóa L2 (phần sau của _tfidf)."""
    W = X.tocsr().copy(); W.eliminate_zeros()
    W.data = np.log(W.data) + 1.0
    W = (W @ sp.diags(idf)).tocsr()
    norm = np.sqrt(np.asarray(W.power(2).sum(axis=1)).ravel()); norm[norm == 0] = 1.0
    return sp.diags(1.0 / norm) @ W

def _msg_text(C, rows, ok):
    """Ma trận đếm của text(c) cho các hàng rows, chỉ cộng báo cáo liên kết có ok = True."""
    if not C["Xr"].shape[0]:                                   # repo không có liên kết nào
        return C["Xm"][rows]
    return C["Xm"][rows] + C["L"][rows] @ sp.diags(ok.astype(np.float64)) @ C["Xr"]

def msg_snapshot(C, repo, snapshot, snap_ct, fc):
    """Phần của 1 snapshot: commit được xét (tổ tiên của snapshot, sửa ít nhất một hàm/lớp có trong graph) và TF-IDF với
    liên kết tạo trước snapshot ít nhất 1 ngày."""
    anc = set(_git(repo_path_of(repo), "rev-list", snapshot).split())
    s8 = snapshot[:8]
    rows = np.flatnonzero([h in anc and any(f"{s8}:{k}" in fc for k in tch) for h, tch in zip(C["h"], C["touch"])])
    ok = C["rep_ct"] <= snap_ct - _DAY
    W, idf = _tfidf(_msg_text(C, rows, ok))
    T = C["T"][rows].tocsr()
    return {"rows": rows, "ok": ok, "W": W, "idf": idf, "L": C["L"][rows].tocsr(), "T": T,
            "t": C["t"][rows], "sha_r": C["sha_r"][rows],
            "keep": np.array([f"{s8}:{k}" in fc for k in C["fn_keys"]], dtype=bool)}

def msg_rank(C, S, repo, inst, ast_ids):
    """Điểm s(c) của các commit được xét cho 1 instance. Trả (s, cos, hạng AST tốt nhất, mặt nạ liên kết qua bộ lọc)."""
    own = issue_own(C, repo, inst)
    cut, _src = issue_cutoff(C, inst, own)
    own_n = {C["docs"][j]["n"] for j in own} | ({int(inst["id"].rsplit("-", 1)[1])} if repo != "django/django" else set())
    ok = (C["rep_ct"] <= cut - _DAY) & ~np.isin(C["rep_n"], sorted(own_n))
    q = _qvec(_nl_terms(inst["text"]), C["vocab"], S["idf"])
    cos = np.asarray(S["W"] @ q).ravel()
    ch = ok != S["ok"]
    if ch.any():
        aff = np.flatnonzero(S["L"][:, np.flatnonzero(ch)].getnnz(axis=1) > 0)
        if aff.size:
            cos[aff] = np.asarray(_msg_weigh(_msg_text(C, S["rows"][aff], ok), S["idf"]) @ q).ravel()
    # Hạng AST tốt nhất của từng commit: min trên các hàm/lớp commit sửa (hàng của T luôn khác rỗng vì touch khác rỗng).
    big = len(ast_ids) + 1
    rk = {x: i + 1 for i, x in enumerate(ast_ids)}
    fr = np.array([rk.get(k, big) for k in C["fn_keys"]], dtype=np.float64)
    T = S["T"]
    best = np.minimum.reduceat(fr[T.indices], T.indptr[:-1]) if T.nnz else np.full(T.shape[0], big, dtype=np.float64)
    s = np.zeros(len(cos))
    for v, has in ((-cos, cos > 0), (best, best < big)):
        ix = np.flatnonzero(has)
        ix = ix[np.lexsort((S["sha_r"][ix], -S["t"][ix], v[ix]))]
        s[ix] += 1.0 / (HIST_MSG_K + np.arange(1, ix.size + 1))
    return s, cos, best, ok

_rows = [json.loads(l) for l in open(V4_RESULTS, encoding="utf-8") if l.strip()]
_todo = [r for r in _rows if "skip" not in r and r.get("hist_msg_cfg") != HIST_MSG_CFG]
print(f"{V4_RESULTS}: {len(_rows)} dòng | cần tính HIST theo phần lời: {len(_todo)}")
_MC = {}
for (_repo, _snap), _grp in groupby(sorted(_todo, key=lambda r: (r["Repo"], _inst[r["Instance"]]["snap_ct"], _inst[r["Instance"]]["snapshot"])),
                                    key=lambda r: (r["Repo"], _inst[r["Instance"]]["snapshot"])):
    _grp = list(_grp)
    if _repo not in _MC:
        _MC[_repo] = msg_corpus(_repo)
    C = _MC[_repo]
    _of = build_graph(_snap, repo_path=repo_path_of(_repo))
    _fc = {n["id"] for n in json.load(open(graph_summary_path(_of), encoding="utf-8"))["nodes"] if n["label"] in _FC}
    S = msg_snapshot(C, _repo, _snap, _inst[_grp[0]["Instance"]]["snap_ct"], _fc)
    _lk = [set(C["rep_n"][S["L"].indices[S["L"].indptr[j]:S["L"].indptr[j + 1]]]) for j in range(len(S["rows"]))]
    keys = np.array(C["fn_keys"])
    for row in _grp:
        inst = _inst[row["Instance"]]
        s, cos, best, ok = msg_rank(C, S, _repo, inst, row["ast_ids"])
        sc = ((sp.diags(s) @ S["T"]).max(axis=0).toarray().ravel() if len(s)    # score(f) = max s(c), A1 của Locus
              else np.zeros(len(keys)))
        _put(row, HIST_MSG_SRC, sc, S["keep"], keys, {x: i + 1 for i, x in enumerate(row["ast_ids"])})
        okn = set(C["rep_n"][ok])
        top = [j for j in np.lexsort((S["sha_r"], -S["t"], -s)) if s[j] > 0][:V4_EVID_N]
        row["hist_msg_cand"] = [{"h": C["h"][S["rows"][j]], "t": int(S["t"][j]), "s": round(float(s[j]), 6),
                                 "cos": round(float(cos[j]), 6),
                                 "ast_r": int(best[j]) if best[j] <= len(row["ast_ids"]) else None,
                                 "touch": C["touch"][S["rows"][j]], "links": sorted(int(n) for n in _lk[j] & okn)}
                                for j in top]
        row["hist_msg_cfg"] = HIST_MSG_CFG
    del S
_tmp = Path(str(V4_RESULTS) + ".tmp")
with open(_tmp, "w", encoding="utf-8") as f:
    for r in _rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
_tmp.replace(V4_RESULTS)

# Mô tả pool: tỷ lệ commit có liên kết, và tỷ lệ báo cáo xuất hiện ở cả pool HIST lẫn pool ISSUE (mục 6.18.9 planning.md).
_ok = [r for r in _rows if "skip" not in r and "hist_msg_cand" in r]
_el = [x for r in _ok for x in r["hist_msg_cand"]]
_both = [bool({n for x in r["hist_msg_cand"] for n in x["links"]} & {x["n"] for x in r.get("issue_cand", [])}) for r in _ok]
print(f"hist-msg: {len(_todo)} dòng mới tính | pool TB {len(_el) / max(1, len(_ok)):.1f} commit | commit có liên kết qua bộ lọc "
      f"{100 * np.mean([bool(x['links']) for x in _el]) if _el else 0:.1f}% | nối AST {100 * np.mean([x['ast_r'] is not None for x in _el]) if _el else 0:.1f}% | "
      f"instance có báo cáo ở cả pool HIST và pool ISSUE {100 * np.mean(_both) if _both else 0:.1f}% → {V4_RESULTS}")

/kaggle/working/v4/scale_rows_v4.jsonl: 2294 dòng | cần tính HIST theo phần lời: 2214


lời commit astropy/astropy:   0%|          | 0/8621 [00:00<?, ?văn bản/s]

[264d9677] disk cache hit → /kaggle/working/astropy_graph_264d9677.json
[9626265d] disk cache hit → /kaggle/working/astropy_graph_9626265d.json
[5e5764ed] disk cache hit → /kaggle/working/astropy_graph_5e5764ed.json
[a7141cd9] disk cache hit → /kaggle/working/astropy_graph_a7141cd9.json
[848c8fa2] disk cache hit → /kaggle/working/astropy_graph_848c8fa2.json
[4fc9f31a] disk cache hit → /kaggle/working/astropy_graph_4fc9f31a.json
[69e2fabd] disk cache hit → /kaggle/working/astropy_graph_69e2fabd.json
[0e1d299f] disk cache hit → /kaggle/working/astropy_graph_0e1d299f.json
[b750a0e6] disk cache hit → /kaggle/working/astropy_graph_b750a0e6.json
[b49ad06b] disk cache hit → /kaggle/working/astropy_graph_b49ad06b.json
[43ee5806] disk cache hit → /kaggle/working/astropy_graph_43ee5806.json
[38322105] disk cache hit → /kaggle/working/astropy_graph_38322105.json
[3a0cd2d8] disk cache hit → /kaggle/working/astropy_graph_3a0cd2d8.json
[b3fa7702] disk cache hit → /kaggle/working/astropy_graph_b3fa77

lời commit django/django:   0%|          | 0/16888 [00:00<?, ?văn bản/s]

[f9de1972] disk cache hit → /kaggle/working/django_graph_f9de1972.json
[9dcfecb7] disk cache hit → /kaggle/working/django_graph_9dcfecb7.json
[0368d63a] disk cache hit → /kaggle/working/django_graph_0368d63a.json
[af1fa5e7] disk cache hit → /kaggle/working/django_graph_af1fa5e7.json
[a8bb4935] disk cache hit → /kaggle/working/django_graph_a8bb4935.json
[511dfb33] disk cache hit → /kaggle/working/django_graph_511dfb33.json
[6f0b8c1c] disk cache hit → /kaggle/working/django_graph_6f0b8c1c.json
[02cd16a7] disk cache hit → /kaggle/working/django_graph_02cd16a7.json
[76dfa834] disk cache hit → /kaggle/working/django_graph_76dfa834.json
[838e432e] disk cache hit → /kaggle/working/django_graph_838e432e.json
[2aaabe20] disk cache hit → /kaggle/working/django_graph_2aaabe20.json
[df28ebd6] disk cache hit → /kaggle/working/django_graph_df28ebd6.json
[60dc957a] disk cache hit → /kaggle/working/django_graph_60dc957a.json
[44077985] disk cache hit → /kaggle/working/django_graph_44077985.json
[cbe4d

lời commit matplotlib/matplotlib:   0%|          | 0/13924 [00:00<?, ?văn bản/s]

... (15 outputs left out)

lời commit sphinx-doc/sphinx:   0%|          | 0/2758 [00:00<?, ?văn bản/s]

[0e899793] disk cache hit → /kaggle/working/sphinx_graph_0e899793.json
[b3e26a6c] disk cache hit → /kaggle/working/sphinx_graph_b3e26a6c.json
[55fc0978] disk cache hit → /kaggle/working/sphinx_graph_55fc0978.json
[66e55a02] disk cache hit → /kaggle/working/sphinx_graph_66e55a02.json
[327c1872] disk cache hit → /kaggle/working/sphinx_graph_327c1872.json
[bf260800] disk cache hit → /kaggle/working/sphinx_graph_bf260800.json
[f1af7dc8] disk cache hit → /kaggle/working/sphinx_graph_f1af7dc8.json
[2c2335bb] disk cache hit → /kaggle/working/sphinx_graph_2c2335bb.json
[487b8436] disk cache hit → /kaggle/working/sphinx_graph_487b8436.json
[07983a5a] disk cache hit → /kaggle/working/sphinx_graph_07983a5a.json
[4b452338] disk cache hit → /kaggle/working/sphinx_graph_4b452338.json
[b923165d] disk cache hit → /kaggle/working/sphinx_graph_b923165d.json
[f47bf147] disk cache hit → /kaggle/working/sphinx_graph_f47bf147.json
[d2c8cd3c] disk cache hit → /kaggle/working/sphinx_graph_d2c8cd3c.json
[f0fef

lời commit sympy/sympy:   0%|          | 0/13998 [00:00<?, ?văn bản/s]

[496e7761] disk cache hit → /kaggle/working/sympy_graph_496e7761.json
[360290c4] disk cache hit → /kaggle/working/sympy_graph_360290c4.json
[0410a6aa] disk cache hit → /kaggle/working/sympy_graph_0410a6aa.json
[a53c4ad9] disk cache hit → /kaggle/working/sympy_graph_a53c4ad9.json
[479939f8] disk cache hit → /kaggle/working/sympy_graph_479939f8.json
[0cd4f13a] disk cache hit → /kaggle/working/sympy_graph_0cd4f13a.json
[d250ff08] disk cache hit → /kaggle/working/sympy_graph_d250ff08.json
[ae3d62ab] disk cache hit → /kaggle/working/sympy_graph_ae3d62ab.json
[b3934e41] disk cache hit → /kaggle/working/sympy_graph_b3934e41.json
[9bf73cc2] disk cache hit → /kaggle/working/sympy_graph_9bf73cc2.json
[205da797] disk cache hit → /kaggle/working/sympy_graph_205da797.json
[8fc63c2d] disk cache hit → /kaggle/working/sympy_graph_8fc63c2d.json
[9cbea134] disk cache hit → /kaggle/working/sympy_graph_9cbea134.json
[678217d5] disk cache hit → /kaggle/working/sympy_graph_678217d5.json
[6399a809] disk cach

In [29]:
# ── Section 8 (v4): graph-proc, xếp hạng node sau hop theo KGCompass (arXiv 2503.21710), mục 8b pipeline_design.md ───────
# S(f) = β^l(f) · (α·cos_norm(e_issue, e_f) + (1 − α)·lev(t_issue, t_f))                        KGCompass, phương trình (1)
#   l(f)     = tổng trọng số trên đường ngắn nhất (Dijkstra) từ issue tới f, cạnh vô hướng; chỉ giữ l ≤ KGC_BUDGET
#   cos_norm = (cos + 1) / 2 như bài; cos giữa embedding của query và của node, cùng model và cùng cách lấy query với AST
#   lev      = 1 − d / max(|a|, |b|), d = khoảng cách Levenshtein theo ký tự, như apoc.text.levenshteinSimilarity mà
#              KGCompass gọi; a = dòng đầu của issue + " " + phần còn lại (như KGCompass), b = source của node
# Cạnh, trọng số theo code KGCompass (KGC_W): issue → điểm xuất phát 0,25; Defines 0,25; Documents 0,5; Calls và Inherits
# 1,0, chỉ đi được khi đích cạnh nằm trong 500 node đầu của AST (KGCompass chỉ nối lời gọi tới 500 method giống issue
# nhất). Không dùng Imports, để graph hop trùng graph của depth-v4. Module chỉ là node trung gian. Như KGCompass nối mọi
# method với file chứa nó, mỗi method của lớp còn được nối thẳng với module của nó bằng cạnh chứa 0,25.
# Hai danh sách, mỗi danh sách một loại điểm xuất phát:
#   graph   (nguồn GRAPH): KGC_SEEDS node đầu của AST, l = 0,25;
#   doc_kgc (nguồn DOC):   đích cạnh Documents của các chunk trong doc_cand (V4_EVID_N chunk), l = 0,25 + 0,5. Danh sách này
#                          thay cách bỏ phiếu của doc_votes (nhân 0,8, lan một bước nhân 0,5), vốn chưa có bài báo làm cơ sở.
# Ứng viên = Function/Class có embedding và l ≤ KGC_BUDGET. Xếp theo S giảm dần; hòa thì l nhỏ trước (như KGCompass), rồi id.
# Mỗi dòng có thêm doc_ids_<src>, doc_score_<src>, doc_l_<src>, doc_r_<src> (hạng AST, 0 = ngoài), doc_gold_<src> (vị trí
# của gold trong danh sách), <src>_gold_l (l của gold, None khi ngoài ngân sách) và <src>_ncand (số ứng viên). Dòng tính
# với cấu hình khác KGC_CFG được tính lại. Cần chạy sau v4-run (ast_ids, doc_cand) và trước v4-fusion.
import heapq
from collections import defaultdict
from itertools import groupby
from rapidfuzz.distance import Levenshtein as _Lev
from rapidfuzz.process import cdist as _lev_cdist

KGC_EDGES = ("Defines", "Calls", "Inherits")
KGC_CFG   = json.dumps({"beta": KGC_BETA, "alpha": KGC_ALPHA, "budget": KGC_BUDGET, "w": KGC_W, "seeds": KGC_SEEDS,
                        "evid_n": V4_EVID_N, "src_max": KGC_SRC_MAX, "top": LIST_K}, sort_keys=True)

def _kgc_graph(path):
    """Node Function/Class/Module của snapshot (id bỏ tiền tố), danh sách kề vô hướng (hàng xóm, loại cạnh, đích cạnh) và
    model embedding của graph. Method của lớp có thêm cạnh chứa (Defines) từ module của nó, như KGCompass nối mọi method với
    file chứa nó; graph chỉ có Defines từ lớp tới method."""
    g = read_graph_json(path)
    nodes = {_key(n["id"]): n for n in g["nodes"] if n["label"] in ("Function", "Class", "Module")}
    adj = defaultdict(list)
    cls_meth = set()
    for e in g["edges"]:
        if e["type"] in KGC_EDGES:
            s, t = _key(e["source"]), _key(e["target"])
            if s in nodes and t in nodes and s != t:
                adj[s].append((t, e["type"], t))
                adj[t].append((s, e["type"], t))
                if e["type"] == "Defines" and nodes[s]["label"] == "Class" and nodes[t]["label"] == "Function":
                    cls_meth.add(t)
    for t in sorted(cls_meth):
        m = nodes[t].get("file")                                   # id của Module là đường dẫn file
        if nodes.get(m, {}).get("label") == "Module":
            adj[m].append((t, "Defines", t))
            adj[t].append((m, "Defines", t))
    return nodes, adj, g["meta"].get("embed_model", _EMBED_MODEL_ID)

def _kgc_dist(starts, adj, top):
    """Dijkstra từ issue. starts = {node xuất phát: l}. Calls và Inherits chỉ đi được khi đích cạnh thuộc top."""
    dist = dict(starts)
    pq = [(l, x) for x, l in starts.items()]
    heapq.heapify(pq)
    while pq:
        l, u = heapq.heappop(pq)
        if l > dist[u]:
            continue
        for v, t, tgt in adj.get(u, ()):
            if t != "Defines" and tgt not in top:
                continue
            lv = l + KGC_W[t]
            if lv <= KGC_BUDGET and lv < dist.get(v, np.inf):
                dist[v] = lv
                heapq.heappush(pq, (lv, v))
    return dist

def _kgc_cand(dist, nodes):
    """Ứng viên: Function/Class có embedding (KGCompass bỏ node không có embedding)."""
    return [x for x in dist if nodes[x]["label"] in _FC and nodes[x].get("embedding") is not None]

def _kgc_sim(ids, nodes, q_emb, q_txt, sim):
    """α·cos_norm + (1 − α)·lev cho các node chưa có trong sim."""
    todo = [x for x in ids if x not in sim]
    if todo:
        cos = (np.array([nodes[x]["embedding"] for x in todo]) @ q_emb + 1) / 2
        lev = _lev_cdist([q_txt], [(nodes[x].get("source") or "")[:KGC_SRC_MAX] for x in todo],
                         scorer=_Lev.normalized_similarity, dtype=np.float64, workers=-1)[0]
        for x, c, v in zip(todo, cos, lev):
            sim[x] = KGC_ALPHA * float(c) + (1 - KGC_ALPHA) * float(v)

def _kgc_put(row, src, dist, cand, sim):
    """Xếp ứng viên theo S và ghi danh sách vào dòng kết quả, cùng dạng với các nguồn khác."""
    sc = {x: KGC_BETA ** dist[x] * sim[x] for x in cand}
    order = sorted(cand, key=lambda x: (-sc[x], dist[x], x))[:LIST_K]
    rk = {x: i + 1 for i, x in enumerate(row["ast_ids"])}
    pos = {x: i + 1 for i, x in enumerate(order)}
    row[f"doc_ids_{src}"] = order
    row[f"doc_score_{src}"] = [round(sc[x], 6) for x in order]
    row[f"doc_l_{src}"] = [dist[x] for x in order]
    row[f"doc_r_{src}"] = [rk.get(x, 0) for x in order]
    row[f"doc_gold_{src}"] = [pos.get(g) for g in row["gold_ids"]]
    row[f"{src}_gold_l"] = [dist.get(g) for g in row["gold_ids"]]
    row[f"{src}_ncand"] = len(cand)

_inst = {i["id"]: i for i in SCALE_INSTANCES}
_rows = [json.loads(l) for l in open(V4_RESULTS, encoding="utf-8") if l.strip()]
_todo = [r for r in _rows if "skip" not in r and r.get("kgc_cfg") != KGC_CFG]
print(f"{V4_RESULTS}: {len(_rows)} dòng | cần tính GRAPH và DOC theo KGCompass: {len(_todo)}")
_snap_of = lambda r: (r["Repo"], _inst[r["Instance"]]["snapshot"])
for (_repo, _snap), _grp in groupby(tqdm(sorted(_todo, key=_snap_of), desc="graph-proc", unit="inst"), key=_snap_of):
    _nodes, _adj, _mid = _kgc_graph(build_graph(_snap, repo_path=repo_path_of(_repo)))
    _model = _get_model(_mid)
    for row in _grp:
        q = _inst[row["Instance"]]["text"]
        title, _, body = q.partition("\n")
        q_txt = title.strip() + " " + body                     # root_text của KGCompass: tiêu đề + " " + nội dung
        q_emb = _model.encode([_embed_query(q)], normalize_embeddings=True, convert_to_numpy=True)[0]
        top = set(row["ast_ids"])
        starts = {"graph": {x: KGC_W["root"] for x in row["ast_ids"][:KGC_SEEDS] if x in _nodes},
                  "doc_kgc": {t: KGC_W["root"] + KGC_W["Documents"]
                              for ch in row["doc_cand"] for t in ch["targets"] if t in _nodes}}
        sim = {}
        for src, st in starts.items():
            dist = _kgc_dist(st, _adj, top)
            cand = _kgc_cand(dist, _nodes)
            _kgc_sim(cand, _nodes, q_emb, q_txt, sim)
            _kgc_put(row, src, dist, cand, sim)
        row["kgc_cfg"] = KGC_CFG
    del _nodes, _adj
_tmp = Path(str(V4_RESULTS) + ".tmp")
with open(_tmp, "w", encoding="utf-8") as f:
    for r in _rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
_tmp.replace(V4_RESULTS)
_ok = [r for r in _rows if "skip" not in r]
print(f"graph-proc: {len(_todo)} dòng mới tính | số ứng viên trung vị: GRAPH "
      f"{np.median([r['graph_ncand'] for r in _ok]):.0f}, DOC {np.median([r['doc_kgc_ncand'] for r in _ok]):.0f} → {V4_RESULTS}")

/kaggle/working/v4/scale_rows_v4.jsonl: 2294 dòng | cần tính GRAPH và DOC theo KGCompass: 0


graph-proc: 0inst [00:00, ?inst/s]

graph-proc: 0 dòng mới tính | số ứng viên trung vị: GRAPH 3712, DOC 3832 → /kaggle/working/v4/scale_rows_v4.jsonl


In [30]:
# ── Section 8 (v4): v4-fusion, trộn các nguồn với AST cho các hệ thống của mục 6.1 (planning.md mục 6.6, 14.7) ───────────
# RRF gốc (Cormack, Clarke, Büttcher, SIGIR 2009), trọng số bằng nhau: score(d) = 1/(k + r_AST(d)) + Σ_s 1/(k + r_s(d));
# r_s(d) = vị trí của d trong danh sách s (từ 1); d không có trong s thì không nhận điểm từ s; k = 60 như bài. Không tham số
# nào được chọn trên dữ liệu của đề tài. Hòa điểm xếp theo hạng AST rồi theo id, để kết quả lặp lại được giữa các lượt.
# Danh sách trộn dài tới LIST_K.
# Giới hạn: bài không thử trộn một danh sách mạnh với các danh sách yếu hơn nhiều. Ở lượt RRF trọng số bằng nhau ngày
# 21/10/2026, DOC và ISSUE làm kết quả kém đi.
# Mỗi danh sách xếp theo bằng chứng của chính nó (mục 9.4 research_context.md): AST theo điểm hybrid (ast_ids), DOC và GRAPH
# theo điểm S của KGCompass (doc_kgc và graph, cell graph-proc), HIST theo điểm s(c) của phần lời commit (hist_msg, cell
# hist-msg, mục 6.18.3 planning.md), ISSUE theo SimiScore.
# Hệ thống phụ: HIST theo Locus (hist_locus, để so với các lượt trước) cùng hai cấu hình thành phần của nó (Locus RQ4: chỉ
# NL, NL+CE; mục 6.8), và ISSUE có thêm liên kết Refs (mục 14.3).
# Kết quả: V4_FUSED (tên → hàm nhận dòng kết quả, trả danh sách id) cho verify-v4 và depth-v4; V4_DIR/fusion.json ghi cấu
# hình.
V4_RRF_K     = 60
V4_SRC_LISTS = {"DOC": "doc_kgc", "HIST": "hist_msg", "ISSUE": "issue", "GRAPH": "graph", "HIST (Locus)": "hist_locus",
                "HIST (NL)": "hist_locus_nl", "HIST (NL+CE)": "hist_locus_nlce", "ISSUE (Refs)": "issue_refs"}
V4_SYSTEMS   = {"AST": [], "+DOC": ["DOC"], "+HIST": ["HIST"], "+ISSUE": ["ISSUE"],
                "+ALL": ["DOC", "HIST", "ISSUE"], "ALL−DOC": ["HIST", "ISSUE"], "ALL−HIST": ["DOC", "ISSUE"],
                "ALL−ISSUE": ["DOC", "HIST"], "+GRAPH": ["GRAPH"], "+ALL+GRAPH": ["DOC", "HIST", "ISSUE", "GRAPH"],
                "+HIST (Locus)": ["HIST (Locus)"], "+HIST (NL)": ["HIST (NL)"], "+HIST (NL+CE)": ["HIST (NL+CE)"],
                "+ISSUE (Refs)": ["ISSUE (Refs)"]}

def _src_order(row, name):
    """Danh sách của 1 nguồn, xếp theo bằng chứng của chính nguồn đó."""
    src = V4_SRC_LISTS[name]
    ids = row.get(f"doc_ids_{src}", [])
    if src == "vote_ast":                            # doc_ids_vote_ast đang xếp theo hạng AST → xếp lại theo điểm vote
        sc = row[f"doc_score_{src}"]
        ids = [ids[j] for j in sorted(range(len(ids)), key=lambda j: (-sc[j], j))]
    return ids

def _rr_matrix(row, sources):
    """Ứng viên (hợp các danh sách), ma trận 1/(k + hạng) (hàng 0 = AST, các hàng sau = nguồn) và thứ tự phá hòa."""
    lists = [row["ast_ids"]] + [_src_order(row, s) for s in sources]
    cand = list(dict.fromkeys(x for lst in lists for x in lst))
    ci = {x: i for i, x in enumerate(cand)}
    R = np.zeros((len(lists), len(cand)))
    for j, lst in enumerate(lists):
        for r, x in enumerate(lst, 1):
            R[j, ci[x]] += 1.0 / (V4_RRF_K + r)
    ra = {x: i for i, x in enumerate(row["ast_ids"])}
    tie = np.empty(len(cand), dtype=np.int64)
    tie[sorted(range(len(cand)), key=lambda i: (ra.get(cand[i], 10 ** 6), cand[i]))] = np.arange(len(cand))
    return cand, R, tie

def rrf_fuse(row, sources):
    """RRF trọng số bằng nhau của AST và các nguồn; trả LIST_K id đầu."""
    cand, R, tie = _rr_matrix(row, sources)
    s = R[0] + R[1:].sum(axis=0)
    return [cand[i] for i in np.lexsort((tie, -s))[:LIST_K]]

_fr = [json.loads(l) for l in open(V4_RESULTS, encoding="utf-8") if l.strip()]
_fr = [r for r in _fr if "skip" not in r and r.get("gold_ids")]
V4_FUSED = {name: (lambda r, s=tuple(srcs): rrf_fuse(r, s)) for name, srcs in V4_SYSTEMS.items()}

with open(V4_DIR / "fusion.json", "w", encoding="utf-8") as f:
    json.dump({"method": "RRF gốc, trọng số bằng nhau (Cormack, Clarke, Büttcher, SIGIR 2009)",
               "k": V4_RRF_K, "list_k": LIST_K, "sources": V4_SRC_LISTS, "systems": V4_SYSTEMS},
              f, ensure_ascii=False)
print(f"v4-fusion: RRF gốc, trọng số bằng nhau, k = {V4_RRF_K} | {len(V4_FUSED)} hệ thống | {len(_fr)} instance "
      f"→ {V4_DIR / 'fusion.json'}")
_miss = {s: sum(f"doc_ids_{V4_SRC_LISTS[s]}" not in r for r in _fr) for s in V4_SRC_LISTS}
if any(_miss.values()):
    print("⚠ dòng thiếu danh sách của nguồn (nguồn đó không góp điểm ở các dòng này):",
          {s: n for s, n in _miss.items() if n})

v4-fusion: RRF gốc, trọng số bằng nhau, k = 60 | 14 hệ thống | 2214 instance → /kaggle/working/v4/fusion.json


In [31]:
# ── Section 8 (v4): judge-v4, LLM-as-a-judge cho bằng chứng (mục 6.7, 6.12, 6.18 và 14.8 planning.md) ──────────────────
# Phần này làm 2 loại request bằng chứng văn bản cho mỗi instance: #ev-doc (chunk doc của doc_cand) và #ev-issue (báo cáo
# cũ của issue_cand). Pool của mỗi loại là V4_EVID_N phần tử đầu (mục 13.2 research_context.md). Nguồn HIST liên kết trực
# tiếp tới các node AST (hàm/lớp) qua cạnh Modifies để xếp hạng định vị code (đánh giá ở verify-v4 và depth-v4), không đưa
# vào LLM judge.
# Chấm lặp: JUDGE_V4_REP_N instance chọn ngẫu nhiên (seed cố định) được chấm bằng chứng lần nữa, cùng phần tử nhưng xáo thứ
# tự khác (loại <loại>-rep), để đo độ ổn định của judge (Utility judgments: LLM nhạy với vị trí). #suff chờ bước G.
# Prompt v6: đánh giá thông tin mà code không cho (hành vi mong đợi, cách dùng API, lý do thiết kế, lỗi tương tự trước đây).
# Để khắc phục lỗi usefulness > relevance, thiên vị độ dài (length bias) và trôi điểm cuối danh sách (sequential drift) của
# mô hình nhỏ khi không có reason:
#   1. Đảo thứ tự bước và trường JSON thành: relevance (0..3) → info_types → usefulness (0..3), kèm quy tắc cứng
#      usefulness <= relevance (nếu relevance = 1 thì usefulness <= 1; nếu relevance = 0 thì info_types = ["unrelated"] và
#      usefulness = 0).
#   2. Đặt trần cắt văn bản (_clip) cho issue, patch và thân báo cáo cũ để loại bỏ các đuôi ngoại lai hàng chục nghìn token
#      và chống thiên vị độ dài.
#   3. Rào định nghĩa api_usage, similar_failure và yêu cầu giữ cùng độ khắt khe từ phần tử 1 đến phần tử N.
# Khóa của request = <instance>#<loại>#<hash của request>, nên nhãn cũ không bị dùng nhầm khi nội dung đổi. Nhãn ghi vào
# JUDGE_V4_DIR/labels_<model>.jsonl. Schema được đọc theo từng request. Chưa đặt GOOGLE_API_KEY (judge-config) thì cell
# chỉ tạo request.
import hashlib
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

JUDGE_V4_DIR = Path(WORK_DIR) / "judge_v4"
JUDGE_V4_DIR.mkdir(parents=True, exist_ok=True)
JUDGE_V4_MODE = "batch"                      # "batch" (giảm 50%, cần billing) hoặc "sync"
JUDGE_BATCH_INSTANCES = 100                  # số instance mỗi batch (giới hạn token đang xếp hàng của Tier 1)
SYNC_WORKERS = 4
EV_INFO_TYPES = ["expected_behavior", "api_usage", "similar_failure", "bug_cause", "code_location", "design_rationale",
                 "past_change", "no_new_information", "unrelated"]
EV_EXCLUSIVE = {"no_new_information", "unrelated"}
JUDGE_V4_REP_N    = 130                      # số instance được chấm lặp bằng chứng để đo độ ổn định của judge
JUDGE_V4_REP_SEED = 20261022
JUDGE_ISSUE_MAX   = 8000                     # trần ký tự (~2.000 token) cho văn bản issue gốc
JUDGE_PATCH_MAX   = 10000                    # trần ký tự (~2.500 token) cho bản sửa thật (patch)
JUDGE_EL_MAX      = 4000                     # trần ký tự (~1.000 token) cho mỗi phần tử bằng chứng (chống thiên vị độ dài)

def _clip(s, lim):
    """Cắt văn bản quá dài: giữ 70% đầu và 30% cuối để bảo toàn cả mô tả mở đầu lẫn kết luận/traceback cuối."""
    s = s or ""
    if len(s) <= lim:
        return s
    head = int(lim * 0.7)
    tail = lim - head
    return f"{s[:head]}\n...[truncated {len(s) - lim} chars]...\n{s[-tail:]}"

JUDGE_PROMPT_EV = """You are assessing evidence retrieved for a GitHub issue. The evidence is meant to give a developer information that the code alone does not give: what the program is supposed to do, how an API is meant to be used, why the code was written or changed in a certain way, how similar problems happened or were fixed before. Finding the code to change is done by a separate component and is NOT the purpose of this evidence.

Evidence can be:
- passages of the project documentation,
- past bug reports that were fixed earlier.
Every piece of evidence existed before the fix was written.

You also see the actual fix of the issue. Use the fix only as a reference to check whether the information in an element is correct and needed for this issue. An element does not need to mention or point to the changed code to be useful.
- Do NOT give a high score only because an element mentions the same file, function, class, exception type, or keywords as the fix or the issue.
- Do NOT reward length: a long bug report or general documentation page that only shares broad topic background must get usefulness <= 1, whereas a short passage that pinpoints the exact contract, cause, or prior fix can get usefulness 2 or 3.
- Judge each element independently in strict order from 1 to N, applying the exact same strictness to the last elements (e.g. D6..D10, I6..I10) as to the first elements (D1, I1).

## Issue
{issue}

## Actual fix
```diff
{patch}
```

## Evidence
{elements}

## Step 1: relevance to this issue and fix
3 = about the exact behavior, API contract, or code that the fix changes
2 = about closely related behavior, or code on the same execution path as the fix
1 = same feature, module, or area only (general reference or a different problem in the same area)
0 = unrelated to this issue

## Step 2: information types provided FOR THIS ISSUE
List every type of information that the element actually provides for solving THIS issue:
expected_behavior = what the program or the involved code is supposed to do in the specific situation described by the issue
api_usage = how an API, data structure, or function involved in the issue is meant to be used in the specific case of the issue (general syntax or parameter overview of a class is NOT api_usage for this issue)
similar_failure = symptoms AND mechanism/cause of a failure with the same root problem as the issue (merely raising the same exception type in another situation is NOT similar_failure)
bug_cause = which code or program state causes the problem in the issue
code_location = which file, class, or function is involved in the problem
design_rationale = why the involved code is implemented or was changed in a certain way
past_change = how the involved code changed before, or how a similar problem was fixed before
no_new_information = the element is about the issue (relevance >= 1) but only repeats what the issue already says (use this type alone)
unrelated = the element provides nothing related to the issue (use this type alone; required when relevance is 0)

## Step 3: usefulness for a developer who has the issue but not the fix
3 = gives information that the fix directly depends on: the exact expected behavior or API contract that the fix must satisfy, the cause of the problem, or how this same problem was fixed before (requires relevance >= 2 and at least one of: expected_behavior, api_usage, bug_cause, past_change)
2 = gives concrete information of at least one type from Step 2 that applies to the specific situation of the issue and that a developer needs to read to understand the issue or decide on the fix (requires relevance >= 2)
1 = background on the same feature or area that does not address the specific situation of the issue, or only identifies code_location
0 = no help, or the information is wrong or outdated for this issue
Strict rules for usefulness:
- usefulness CANNOT exceed relevance (usefulness <= relevance). If relevance is 0, usefulness MUST be 0. If relevance is 1, usefulness is at most 1.
- If info_types is no_new_information or unrelated, usefulness MUST be 0.
- If the only information an element gives is which code is involved (code_location), usefulness is at most 1.

Return a JSON array with exactly one object per element, ordered from 1 to N:
{{"id": "<element id>", "relevance": <0-3>, "info_types": ["<type>", ...], "usefulness": <0-3>}}"""

# Thứ tự trường theo các bước của prompt: điểm liên quan, loại thông tin, điểm hữu ích.
JUDGE_SCHEMA_EV = {"type": "array",
                   "items": {"type": "object",
                             "properties": {"id": {"type": "string"},
                                            "relevance": {"type": "integer", "minimum": 0, "maximum": 3},
                                            "info_types": {"type": "array", "items": {"type": "string", "enum": EV_INFO_TYPES}},
                                            "usefulness": {"type": "integer", "minimum": 0, "maximum": 3}},
                             "required": ["id", "relevance", "info_types", "usefulness"]}}

def _request(prompt, schema):
    """Request Gemini trả JSON theo schema."""
    return {"contents": [{"role": "user", "parts": [{"text": prompt}]}],
            "generationConfig": {"responseMimeType": "application/json", "responseJsonSchema": schema,
                                 "thinkingConfig": {"thinkingLevel": THINKING_LEVEL}}}

def _items(members, iid, prefix, salt=""):
    """Xáo thứ tự phần tử (seed theo instance) và đặt tên alias. salt đổi thứ tự xáo cho bản chấm lặp."""
    items = [{"node": k, "systems": sorted(v)} for k, v in members.items()]
    random.Random(iid + prefix + salt).shuffle(items)
    for j, it in enumerate(items, 1):
        it["alias"] = f"{prefix}{j}"
    return items

def _req_hash(req):
    return hashlib.sha1(json.dumps(req, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()[:12]

def parse_evidence(text, aliases):
    """Nhãn của request bằng chứng: {alias: {"relevance", "usefulness", "info_types", "reason"}} nếu đủ mọi
    alias, hai điểm trong 0..3, loại thông tin khác rỗng và thuộc danh sách; chuẩn hóa khi nhãn loại trừ đi kèm nhãn khác."""
    try:
        arr = json.loads(text)
    except Exception:
        m = re.search(r"\[.*\]", text or "", re.S)
        try:
            arr = json.loads(m.group(0)) if m else None
        except Exception:
            arr = None
    if not isinstance(arr, list):
        return None
    out = {}
    for it in arr:
        if not isinstance(it, dict):
            continue
        a, rel, use, ty = str(it.get("id", "")).strip(), it.get("relevance"), it.get("usefulness"), it.get("info_types")
        if a not in aliases or a in out or not all(isinstance(s, (int, float)) and int(s) in (0, 1, 2, 3) for s in (rel, use)):
            continue
        if not isinstance(ty, list) or not ty or not set(ty) <= set(EV_INFO_TYPES):
            continue
        rel, use, tys = int(rel), int(use), set(ty)
        if tys & EV_EXCLUSIVE and len(tys) > 1:
            if rel == 0:
                tys, use = {"unrelated"}, 0
            elif use == 0 and "no_new_information" in tys:
                tys = {"no_new_information"}
            else:
                tys -= EV_EXCLUSIVE
        out[a] = {"relevance": rel, "usefulness": use, "info_types": sorted(tys),
                  "reason": str(it.get("reason", ""))[:300]}
    return out if set(out) == set(aliases) else None

def _parse_v4(key, text):
    aliases = {it["alias"] for it in POOL_V4[key]["items"]}
    return parse_evidence(text, aliases)

def _day(t):
    return datetime.fromtimestamp(t, timezone.utc).strftime("%Y-%m-%d")

# ── Request bằng chứng (chỉ DOC và ISSUE; HIST nối trực tiếp tới AST node) ────────────────────────────────────────────
_v4r = [json.loads(l) for l in open(V4_RESULTS, encoding="utf-8") if l.strip()]
_v4r = [r for r in _v4r if "skip" not in r and "doc_cand" in r]
_reports = {}

_rep = set(random.Random(JUDGE_V4_REP_SEED).sample(sorted(r["Instance"] for r in _v4r), min(JUDGE_V4_REP_N, len(_v4r))))

REQ_V4, POOL_V4 = {}, {}
for (_repo, _snap), _grp in groupby(sorted(_v4r, key=lambda r: (r["Repo"], _inst[r["Instance"]]["snap_ct"], _inst[r["Instance"]]["snapshot"])),
                                    key=lambda r: (r["Repo"], _inst[r["Instance"]]["snapshot"])):
    if _repo not in _reports:
        _reports[_repo] = {r["n"]: r for r in map(json.loads, open(_issues_path(_repo), encoding="utf-8"))}
    nbi = {n["id"]: n for n in read_graph_json(build_graph(_snap, repo_path=repo_path_of(_repo)))["nodes"]}
    for r in _grp:
        iid, inst = r["Instance"], _inst[r["Instance"]]
        base = {"Instance": iid, "Repo": _repo, "issue_names_gold": int(r["issue_names_gold"])}
        issue_txt = _clip(inst["text"], JUDGE_ISSUE_MAX)
        patch_txt = _clip(inst["patch"], JUDGE_PATCH_MAX)
        fmt = lambda els: JUDGE_PROMPT_EV.format(issue=issue_txt, patch=patch_txt, elements="\n\n".join(els))

        def _mk(kind, members, prefix, el_text, salt=""):
            """Dựng 1 request: xáo phần tử, ghi text từng phần tử và số token ước lượng của nó (it["tok"], không vào hash)."""
            its = _items(members, iid, prefix, salt)
            if not its:
                return
            es = [el_text(it["node"], it["alias"]) for it in its]
            for it, e in zip(its, es):
                it["tok"] = len(e) // 4
            req = _request(fmt(es), JUDGE_SCHEMA_EV)
            key = f"{iid}#{kind}#{_req_hash(req)}"
            REQ_V4[key] = req
            POOL_V4[key] = {**base, "kind": kind, "items": its}

        # 1. Chunk doc
        ev = {x["id"]: x for x in r["doc_cand"][:V4_EVID_N]}
        doc_el = lambda node, a: (f"[{a}] Documentation passage\nFile: {nbi[node]['file']}\n"
                                  f"Section: {nbi[node].get('heading') or ''}\nText:\n{_clip(nbi[node].get('text') or '', JUDGE_EL_MAX)}")
        # 2. Báo cáo cũ
        evi = {str(x["n"]): x for x in r.get("issue_cand", [])[:V4_EVID_N]}

        def issue_el(node, a):
            b = _reports[_repo][int(node)]
            return f"[{a}] Past bug report #{node}, created {_day(b['created'])}: {b['title']}\n{_clip(b['body'], JUDGE_EL_MAX)}"
        for kind, cand, prefix, el_text in (("ev-doc", ev, "D", doc_el), ("ev-issue", evi, "I", issue_el)):
            members = {k: {kind[3:].upper()} for k in cand}
            _mk(kind, members, prefix, el_text)
            if iid in _rep:                          # chấm lặp: cùng phần tử, thứ tự xáo khác
                _mk(f"{kind}-rep", members, prefix, el_text, salt="rep")

with open(JUDGE_V4_DIR / "requests.jsonl", "w", encoding="utf-8") as f:
    for k, v in REQ_V4.items():
        f.write(json.dumps({"key": k, "request": v}, ensure_ascii=False) + "\n")
with open(JUDGE_V4_DIR / "pool.jsonl", "w", encoding="utf-8") as f:
    for k, v in POOL_V4.items():
        f.write(json.dumps({"key": k, **v}, ensure_ascii=False) + "\n")
for kind in ("ev-doc", "ev-issue", "ev-doc-rep", "ev-issue-rep"):
    ks = [k for k, v in POOL_V4.items() if v["kind"] == kind]
    if ks:
        n = pd.Series([len(POOL_V4[k]["items"]) for k in ks])
        ch = pd.Series([len(REQ_V4[k]["contents"][0]["parts"][0]["text"]) for k in ks])
        print(f"{kind}: {len(ks)} request | phần tử mỗi request TB {n.mean():.1f}, max {n.max()} | độ dài prompt TB "
              f"{ch.mean() / 4:.0f} token, p90 {ch.quantile(0.9) / 4:.0f}, max {ch.max() / 4:.0f} token (ước lượng)")

# ── Gọi Gemini (batch hoặc sync), schema theo từng request ─────────────────────────────────────────────────────────────
_client = None
def _gclient():
    global _client
    if _client is None:
        from google import genai
        assert GOOGLE_API_KEY, "Chưa đặt GOOGLE_API_KEY trong cell judge-config"
        _client = genai.Client(api_key=GOOGLE_API_KEY)
    return _client

def _labels_path_v4(model):
    return JUDGE_V4_DIR / f"labels_{model}.jsonl"

def load_labels_v4(model):
    out = {}
    p = _labels_path_v4(model)
    if p.exists():
        for line in open(p, encoding="utf-8"):
            if line.strip():
                o = json.loads(line)
                if o.get("labels"):
                    out[o["key"]] = o["labels"]
    return out

def _write_v4(model, key, labels, raw, usage=None):
    with open(_labels_path_v4(model), "a", encoding="utf-8") as f:
        f.write(json.dumps({"key": key, "model": model, "labels": labels, "raw": (raw or "")[:4000],
                            "usage": usage}, ensure_ascii=False) + "\n")

def _sync_one_v4(model, key):
    req = REQ_V4[key]
    cfg = {"response_mime_type": "application/json",
           "response_json_schema": req["generationConfig"]["responseJsonSchema"],
           "thinking_config": {"thinking_level": THINKING_LEVEL}}
    for attempt in range(6):
        try:
            r = _gclient().models.generate_content(model=model, contents=req["contents"], config=cfg)
            u = getattr(r, "usage_metadata", None)
            usage = {"in": getattr(u, "prompt_token_count", None), "out": getattr(u, "candidates_token_count", None),
                     "think": getattr(u, "thoughts_token_count", None)} if u else None
            return key, _parse_v4(key, r.text), r.text, usage
        except Exception as e:
            msg = str(e)
            if "response_json_schema" in msg and "response_json_schema" in cfg:
                print("  ⚠ SDK không nhận response_json_schema → dùng response_schema")
                cfg["response_schema"] = cfg.pop("response_json_schema")
                continue
            if "thinking" in msg.lower() and "thinking_config" in cfg:
                print(f"  ⚠ {model} không nhận thinking_level → gọi lại không đặt thinking"); cfg.pop("thinking_config")
                continue
            if any(t in msg for t in ("429", "RESOURCE_EXHAUSTED", "500", "503", "UNAVAILABLE", "DEADLINE")):
                time.sleep(min(90, 5 * 2 ** attempt))
                continue
            return key, None, f"ERROR: {msg[:500]}", None
    return key, None, "ERROR: hết số lần thử", None

def run_sync_v4(model, keys):
    done = load_labels_v4(model)
    todo = [k for k in keys if k not in done]
    print(f"[{model}] sync: {len(keys) - len(todo)} đã có, chấm {len(todo)}")
    bad = 0
    with ThreadPoolExecutor(SYNC_WORKERS) as ex:
        for key, labels, raw, usage in tqdm(ex.map(lambda k: _sync_one_v4(model, k), todo), total=len(todo),
                                            desc=f"judge {model}"):
            _write_v4(model, key, labels, raw, usage)
            bad += labels is None
    if bad:
        print(f"  ⚠ {bad} request không parse được hoặc thiếu phần tử; chạy lại cell sẽ thử lại")

def _resp_text(resp):
    parts = (((resp or {}).get("candidates") or [{}])[0].get("content") or {}).get("parts") or []
    return "".join(p.get("text", "") for p in parts if not p.get("thought"))

def collect_batch_v4(model, job_name):
    """Lấy kết quả của 1 batch job (tên job ghi trong JUDGE_V4_DIR/batch_jobs.txt)."""
    job = _gclient().batches.get(name=job_name)
    if job.state.name != "JOB_STATE_SUCCEEDED":
        print(f"  batch {job_name}: {job.state.name}")
        return False
    data = _gclient().files.download(file=job.dest.file_name).decode("utf-8")
    done = load_labels_v4(model)
    n = 0
    for line in data.splitlines():
        if not line.strip():
            continue
        o = json.loads(line)
        key = o.get("key")
        if key not in POOL_V4 or key in done:
            continue
        text = _resp_text(o.get("response"))
        labels = _parse_v4(key, text)
        usage = (o.get("response") or {}).get("usageMetadata")
        _write_v4(model, key, labels, text or json.dumps(o.get("error", ""))[:500], usage)
        n += labels is not None
    print(f"  batch {job_name}: nhận {n} kết quả hợp lệ")
    return True

def _batch_parts(keys):
    """Chia request theo instance (key = <instance>#<loại>#<hash>): mỗi phần gồm mọi request của JUDGE_BATCH_INSTANCES instance."""
    by_inst = {}
    for k in keys:
        by_inst.setdefault(k.split("#", 1)[0], []).append(k)
    insts = list(by_inst)
    return [[k for i in insts[j:j + JUDGE_BATCH_INSTANCES] for k in by_inst[i]]
            for j in range(0, len(insts), JUDGE_BATCH_INSTANCES)]

def _jobs_of_v4(model):
    out, p = {}, JUDGE_V4_DIR / "batch_jobs.txt"
    if p.exists():
        for line in open(p, encoding="utf-8"):
            f = line.rstrip("\n").split("\t")
            if len(f) == 3 and f[0] == model:
                out[f[2]] = f[1]
    return out

def _wait(job_name, poll):
    while True:
        state = _gclient().batches.get(name=job_name).state.name
        if state in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED", "JOB_STATE_EXPIRED"):
            return state
        time.sleep(poll)

def run_batch_v4(model, keys, poll=60):
    parts = _batch_parts(keys)
    print(f"[{model}] batch: {len(keys)} request, {len(parts)} phần, mỗi phần tối đa {JUDGE_BATCH_INSTANCES} instance")
    for i, part in enumerate(parts, 1):
        done = load_labels_v4(model)
        todo = [k for k in part if k not in done]
        if not todo:
            print(f"  phần {i}/{len(parts)}: đã có nhãn đủ")
            continue
        kinds = "-".join(sorted({k.split("#")[1].replace("ev-", "").replace("-rep", "r") for k in todo}))   # tên batch ngắn
        name = f"{JUDGE_V4_DIR.name}_{model}_p{i:03d}of{len(parts):03d}_{kinds}_{_req_hash(sorted(todo))[:8]}"
        job = _jobs_of_v4(model).get(name)
        state = None
        if job:                                   # đã gửi ở lần chạy trước: chờ và lấy kết quả, không gửi lại
            print(f"  phần {i}/{len(parts)}: đã gửi trước đó ({job}), chờ kết quả")
            state = _wait(job, poll)
        if state != "JOB_STATE_SUCCEEDED":
            if job:
                print(f"  phần {i}/{len(parts)}: batch cũ kết thúc {state} → gửi lại {len(todo)} request còn thiếu")
            src = JUDGE_V4_DIR / f"batch_in_{name}.jsonl"
            with open(src, "w", encoding="utf-8") as f:
                for k in todo:
                    f.write(json.dumps({"key": k, "request": REQ_V4[k]}, ensure_ascii=False) + "\n")
            tok = sum(len(REQ_V4[k]["contents"][0]["parts"][0]["text"]) for k in todo) / 4
            up = _gclient().files.upload(file=str(src), config={"display_name": name, "mime_type": "jsonl"})
            job = _gclient().batches.create(model=model, src=up.name, config={"display_name": name}).name
            with open(JUDGE_V4_DIR / "batch_jobs.txt", "a", encoding="utf-8") as f:
                f.write(f"{model}\t{job}\t{name}\n")
            print(f"  phần {i}/{len(parts)}: đã gửi {job} ({len(todo)} request, khoảng {tok / 1e6:.1f} triệu token); chờ kết quả")
            state = _wait(job, poll)
        print(f"  phần {i}/{len(parts)}: batch kết thúc {state}")
        if state == "JOB_STATE_SUCCEEDED":
            collect_batch_v4(model, job)
    done = load_labels_v4(model)
    left = [k for k in keys if k not in done]
    if left:
        print(f"  chấm lại từng cái {len(left)} request còn thiếu")
        run_sync_v4(model, left)

if GOOGLE_API_KEY:
    (run_batch_v4 if JUDGE_V4_MODE == "batch" else run_sync_v4)(JUDGE_MODEL, list(REQ_V4))
    _done_v4 = load_labels_v4(JUDGE_MODEL)
    print(f"{JUDGE_MODEL}: {sum(k in _done_v4 for k in REQ_V4)}/{len(REQ_V4)} request đã có nhãn")
else:
    print("Chưa đặt GOOGLE_API_KEY trong judge-config: chỉ tạo request, chưa gửi judge.")

[264d9677] disk cache hit → /kaggle/working/astropy_graph_264d9677.json
[9626265d] disk cache hit → /kaggle/working/astropy_graph_9626265d.json
[5e5764ed] disk cache hit → /kaggle/working/astropy_graph_5e5764ed.json
[a7141cd9] disk cache hit → /kaggle/working/astropy_graph_a7141cd9.json
[848c8fa2] disk cache hit → /kaggle/working/astropy_graph_848c8fa2.json
[4fc9f31a] disk cache hit → /kaggle/working/astropy_graph_4fc9f31a.json
[69e2fabd] disk cache hit → /kaggle/working/astropy_graph_69e2fabd.json
[0e1d299f] disk cache hit → /kaggle/working/astropy_graph_0e1d299f.json
[b750a0e6] disk cache hit → /kaggle/working/astropy_graph_b750a0e6.json
[b49ad06b] disk cache hit → /kaggle/working/astropy_graph_b49ad06b.json
[43ee5806] disk cache hit → /kaggle/working/astropy_graph_43ee5806.json
[38322105] disk cache hit → /kaggle/working/astropy_graph_38322105.json
[3a0cd2d8] disk cache hit → /kaggle/working/astropy_graph_3a0cd2d8.json
[b3fa7702] disk cache hit → /kaggle/working/astropy_graph_b3fa77

In [32]:
# ── Section 8 (v4): verify-v4, đánh giá phần AST theo bộ độ đo đã chốt (mục 13 research_context.md, 6.12 planning.md) ─
# Hai mức: hàm/lớp là mức chính; file là mức phụ. Xếp hạng file suy từ xếp hạng hàm/lớp: điểm của file = điểm cao nhất
# của hàm/lớp trong file (SweRank), tức là thứ tự xuất hiện đầu tiên của file trong danh sách. Gold mức file = các file
# chứa gold hàm/lớp. Id node trong V4_RESULTS đã bỏ tiền tố snapshot, có dạng {đường dẫn}:{tên lồng nhau}, nên file lấy
# được từ id.
# Kiểm định: McNemar chính xác (nhị thức trên cặp bất đồng) cho độ đo đúng/sai; kiểm định hoán vị ghép cặp (đổi dấu hiệu
# của chênh lệch từng instance, 10.000 hoán vị) cho MRR và MAP (Smucker và cộng sự 2007; Urbano và cộng sự 2019);
# Holm trong từng độ đo, riêng cho họ Q1 đến Q7 và họ G1, G2 của GRAPH. Phần cuối cell (bước I, mục 14.1 planning.md)
# đọc nhãn của judge-v4: Bảng 3 (bằng chứng), Bảng 4 (loại thông tin) và độ ổn định của judge. Bảng 5 (#suff) chờ bước G.
# Khối "Ảnh hưởng của độ phủ doc" (mục 10.2 pipeline_design.md) đặt độ phủ doc cạnh mức đóng góp của DOC, ở mức repo và mức
# instance.
# Hệ thống: lấy từ V4_FUSED (tên → hàm(row) → danh sách id) do cell v4-fusion tạo.
from scipy.stats import binomtest

V4_PERM_B    = 10000                 # số hoán vị (giá trị chọn của đề tài)
V4_PERM_SEED = 20261021
V4_QUESTIONS = [("Q1", "+DOC", "AST"), ("Q2", "+HIST", "AST"), ("Q3", "+ISSUE", "AST"), ("Q4", "+ALL", "AST"),
                ("Q5", "+ALL", "ALL−DOC"), ("Q6", "+ALL", "ALL−HIST"), ("Q7", "+ALL", "ALL−ISSUE"),
                ("G1", "+GRAPH", "AST"), ("G2", "+ALL+GRAPH", "+ALL")]   # G1, G2: danh sách GRAPH (graph-proc)
V4_FAMILIES = (r"^Q\d$", r"^G\d$")         # họ Holm: Q1 đến Q7, và G1, G2 (mục 8b pipeline_design.md)
V4_EXTRA_PAIRS = [("ALL−ISSUE", "AST")]   # DOC+HIST so với AST: kết luận hẹp hiện có
V4_LEVELS = {
    "hàm/lớp": {"bin": ["Top@1", "Top@5", "Top@10", "Acc@5", "Acc@10"], "cont": ["MRR", "MAP"]},
    "file":    {"bin": ["Top@1", "Top@5", "Top@10", "Acc@1", "Acc@3", "Acc@5"], "cont": ["MRR", "MAP"]},
}

_V4_SYS = dict(V4_FUSED)

def _file_of(node_id):
    return node_id.split(":", 1)[0]

def _file_order(order):
    """Thứ tự file = thứ tự xuất hiện đầu tiên của hàm/lớp thuộc file đó (điểm file = điểm cao nhất)."""
    seen, out = set(), []
    for x in order:
        f = _file_of(x)
        if f not in seen:
            seen.add(f)
            out.append(f)
    return out

def _level_metrics(order, gold, ks_acc):
    pos = {x: i + 1 for i, x in enumerate(order)}
    ranks = sorted(pos[g] for g in gold if g in pos)
    best = ranks[0] if ranks else None
    m = {f"Top@{k}": int(best is not None and best <= k) for k in (1, 5, 10)}
    m.update({f"Acc@{k}": int(len(ranks) == len(gold) and ranks[-1] <= k) for k in ks_acc})
    m["MRR"] = 1.0 / best if best else 0.0
    m["MAP"] = sum((i + 1) / r for i, r in enumerate(ranks)) / len(gold)
    m["best"] = best or LIST_K + 1      # gold ngoài danh sách có cùng hạng ở mọi hệ thống
    return m

def _v4_row_metrics(r, name, fn):
    order = fn(r)
    gold = list(dict.fromkeys(r["gold_ids"]))
    base = {"Instance": r["Instance"], "Repo": r["Repo"].split("/")[1],
            "nêu tên gold": bool(r.get("issue_names_gold")), "gold_doc": r.get("gold_doc"), "doc_cov": r.get("doc_cov"),
            "Hệ thống": name}
    out = [{**base, "Mức": "hàm/lớp", **_level_metrics(order, gold, (5, 10))}]
    out.append({**base, "Mức": "file", **_level_metrics(_file_order(order), list(dict.fromkeys(map(_file_of, gold))), (1, 3, 5))})
    return out

def _perm_test(x, y, B=V4_PERM_B, seed=V4_PERM_SEED):
    """Kiểm định hoán vị ghép cặp, hai phía, thống kê = trung bình chênh lệch. p = (số hoán vị cực trị + 1) / (B + 1)."""
    d = np.asarray(x, float) - np.asarray(y, float)
    obs = abs(d.mean())
    if not np.any(d):
        return 1.0
    rng, hits, done = np.random.default_rng(seed), 0, 0
    while done < B:
        n = min(2000, B - done)
        s = rng.integers(0, 2, size=(n, d.size), dtype=np.int8) * 2 - 1
        hits += int((np.abs((s * d).mean(axis=1)) >= obs - 1e-12).sum())
        done += n
    return (hits + 1) / (B + 1)

def _holm(pvals):
    """Hiệu chỉnh Holm cho một họ p-value (bỏ qua NaN)."""
    p = np.asarray(pvals, float)
    idx = [i for i in np.argsort(p) if not np.isnan(p[i])]
    adj, run = np.full(p.size, np.nan), 0.0
    for j, i in enumerate(idx):
        run = max(run, min(1.0, (len(idx) - j) * p[i]))
        adj[i] = run
    return adj

_v4rows = [json.loads(l) for l in open(V4_RESULTS, encoding="utf-8") if l.strip()]
_v4rows = [r for r in _v4rows if "skip" not in r and r.get("gold_ids")]
_V4M = pd.DataFrame([m for r in _v4rows for name, fn in _V4_SYS.items() for m in _v4_row_metrics(r, name, fn)])
_v4order = list(_V4_SYS)
_v4pairs = [(q, a, b) for q, a, b in V4_QUESTIONS if a in _V4_SYS and b in _V4_SYS] + \
           [(f"{a} so với {b}", a, b) for a, b in V4_EXTRA_PAIRS if a in _V4_SYS and b in _V4_SYS]

def _v4_table1(M, level):
    cols = V4_LEVELS[level]["bin"] + V4_LEVELS[level]["cont"]
    t = M[M["Mức"] == level].groupby("Hệ thống")[cols].mean().reindex(_v4order)
    t[V4_LEVELS[level]["bin"]] *= 100
    return t.round({c: 1 for c in V4_LEVELS[level]["bin"]} | {c: 3 for c in V4_LEVELS[level]["cont"]})

def _v4_tests(M, level, pairs=None):
    """Mỗi câu hỏi một hàng: chênh lệch trung bình, McNemar (b / c, p) cho độ đo đúng/sai, hoán vị cho MRR và MAP.
    Cột "Holm" = p sau hiệu chỉnh Holm trong từng độ đo, riêng cho từng họ của V4_FAMILIES (cặp phụ không tính vào họ).
    pairs thay cho _v4pairs khi chỉ cần một vài cặp."""
    L = M[M["Mức"] == level]
    rows = []
    for q, a, b in (_v4pairs if pairs is None else pairs):
        A = L[L["Hệ thống"] == a].set_index("Instance")
        Bm = L[L["Hệ thống"] == b].set_index("Instance").reindex(A.index)
        row = {"Câu hỏi": q, "So sánh": f"{a} so với {b}", "Số instance": len(A)}
        for k in V4_LEVELS[level]["bin"]:
            hb, hc = int(((A[k] == 1) & (Bm[k] == 0)).sum()), int(((A[k] == 0) & (Bm[k] == 1)).sum())
            row[f"Δ{k}"] = round(100 * (A[k].mean() - Bm[k].mean()), 1)
            row[f"{k} b/c"] = f"{hb}/{hc}"
            row[f"p {k}"] = binomtest(hb, hb + hc, 0.5).pvalue if hb + hc else 1.0
        for k in V4_LEVELS[level]["cont"]:
            row[f"Δ{k}"] = round(A[k].mean() - Bm[k].mean(), 4)
            row[f"p {k}"] = _perm_test(A[k].values, Bm[k].values)
        rows.append(row)
    t = pd.DataFrame(rows)
    for k in V4_LEVELS[level]["bin"] + V4_LEVELS[level]["cont"]:
        t[f"Holm {k}"] = np.nan
        for f in V4_FAMILIES:
            fam = t["Câu hỏi"].str.match(f)
            if fam.any():
                t.loc[fam, f"Holm {k}"] = _holm(t.loc[fam, f"p {k}"].values)
    return t.round(4)

def _v4_table2(M):
    """Bảng 2 (mức hàm/lớp): số instance tốt hơn / kém hơn / không đổi theo hạng gold tốt nhất, tách theo độ khó của AST."""
    W = M[M["Mức"] == "hàm/lớp"].pivot(index="Instance", columns="Hệ thống", values="best")
    diff = pd.cut(W["AST"], [0, 10, 100, np.inf], labels=["AST: gold trong top 10", "AST: hạng 11 đến 100",
                                                          "AST: ngoài hạng 100"])
    out = []
    for q, a, b in _v4pairs:
        d = np.sign(W[b] - W[a])
        for grp, idx in [("Tất cả", W.index)] + [(g, W.index[diff == g]) for g in diff.cat.categories]:
            dd = d.loc[idx]
            n = len(dd)
            f = lambda v: f"{v} ({100 * v / n:.1f}%)" if n else "0"
            out.append({"Câu hỏi": q, "So sánh": f"{a} so với {b}", "Nhóm": grp, "Số instance": n,
                        "Tốt hơn": f(int((dd > 0).sum())), "Kém hơn": f(int((dd < 0).sum())),
                        "Không đổi": f(int((dd == 0).sum()))})
    return pd.DataFrame(out)

print(f"verify-v4: {_V4M.Instance.nunique()} instance | hệ thống: {', '.join(_v4order)} | "
      f"câu hỏi có đủ hệ thống: {', '.join(q for q, _, _ in _v4pairs)}")
for scope, M in [("Gộp mọi repo", _V4M)] + [(f"Riêng {rp}", _V4M[_V4M.Repo == rp]) for rp in sorted(_V4M.Repo.unique())]:
    for level in V4_LEVELS:
        print(f"\nBảng 1. {scope}, mức {level} ({M.Instance.nunique()} instance)")
        display(_v4_table1(M, level))
for level in V4_LEVELS:
    print(f"\nKiểm định, gộp mọi repo, mức {level}")
    display(_v4_tests(_V4M, level))
print("\nBảng 2. Thay đổi theo từng instance, mức hàm/lớp, gộp mọi repo")
display(_v4_table2(_V4M))
for flag, name in [(True, "issue nêu sẵn tên gold"), (False, "issue không nêu tên gold")]:
    M = _V4M[_V4M["nêu tên gold"] == flag]
    if len(M):
        print(f"\nPhân tích bổ sung 2, nhóm {name} ({M.Instance.nunique()} instance)")
        for level in V4_LEVELS:
            display(_v4_table1(M, level))

# ── Ảnh hưởng của độ phủ doc tới hiệu quả (mục 10.2 pipeline_design.md, quyết định 60 research_context.md) ────────────────
# Biến độ phủ do scale_row ghi. doc_cov là tỷ lệ Function/Class của snapshot có ít nhất một cạnh Documents, như mục 1 của
# coverage_report.md. gold_doc là số gold có cạnh Documents. Kết quả chính là Δ của +DOC so với AST theo từng instance, ở mức
# hàm/lớp. Kết quả phụ là +ALL so với ALL−DOC. +HIST và +GRAPH so với AST là đối chứng âm, vì hai nguồn này không đọc doc
# (Lipsitch và cộng sự 2010). Nếu Δ của chúng cũng đi theo độ phủ, quan hệ của DOC có thể do đặc điểm của repo.
# Bảng C-A (mức repo) ghi độ phủ và Δ trung bình của từng repo. C1 là ρ của Spearman giữa hai cột đó (Cronen-Townsend và
# cộng sự 2002), với p tính bằng hoán vị nhãn repo. C1 cần ít nhất 3 repo.
# Bảng C-B (mức instance) tách nhóm gold_doc ≥ 1 và nhóm gold_doc = 0 (Kochhar và cộng sự 2014). C2 là hiệu Δ trung bình giữa
# hai nhóm, với p tính bằng hoán vị nhãn nhóm trong từng repo (Gelman và Stern 2006; Wang và cộng sự 2007).
# Bảng C-C mô tả cơ chế: trong nhóm gold_doc ≥ 1, tỷ lệ instance có gold trong danh sách DOC và có chunk của doc_cand nối tới
# gold. Họ C gồm C1 và C2 của +DOC so với AST, với Holm trong từng độ đo. Kết quả phụ và đối chứng chỉ có p thô.
V4_COV_MAIN  = ("+DOC", "AST")
V4_COV_PAIRS = [V4_COV_MAIN, ("+ALL", "ALL−DOC"), ("+HIST", "AST"), ("+GRAPH", "AST")]   # chính, phụ, hai đối chứng âm
_COV_MET = V4_LEVELS["hàm/lớp"]["bin"] + V4_LEVELS["hàm/lớp"]["cont"]
_COV_PCT = {k: 100 if k in V4_LEVELS["hàm/lớp"]["bin"] else 1 for k in _COV_MET}   # Top@k và Acc@k tính bằng điểm %

def _cov_delta(a, b):
    """Δ = a − b theo từng instance ở mức hàm/lớp, kèm Repo, gold_doc, doc_cov và cột nêu tên gold."""
    L = _V4M[_V4M["Mức"] == "hàm/lớp"]
    A = L[L["Hệ thống"] == a].set_index("Instance")
    B = L[L["Hệ thống"] == b].set_index("Instance").reindex(A.index)
    return (A[_COV_MET] - B[_COV_MET]).join(A[["Repo", "gold_doc", "doc_cov", "nêu tên gold"]])

def _spearman_perm(x, y, B=V4_PERM_B, seed=V4_PERM_SEED):
    """ρ của Spearman (hạng trung bình khi hòa) và p hai phía bằng hoán vị y, B lần."""
    rx, ry = pd.Series(x).rank().values, pd.Series(y).rank().values
    rx, ry = rx - rx.mean(), ry - ry.mean()
    den = np.sqrt((rx ** 2).sum() * (ry ** 2).sum())
    if den == 0:
        return np.nan, np.nan
    rho = float(rx @ ry) / den
    rng = np.random.default_rng(seed)
    P = np.stack([rng.permutation(ry) for _ in range(B)])
    return rho, (int((np.abs(P @ rx) / den >= abs(rho) - 1e-12).sum()) + 1) / (B + 1)

def _strat_perm_diff(D, grp, strata, B=V4_PERM_B, seed=V4_PERM_SEED):
    """Hiệu trung bình của từng cột D giữa nhóm grp và phần còn lại; p hai phía bằng hoán vị nhãn nhóm trong từng tầng."""
    D, grp, strata = np.asarray(D, float), np.asarray(grp, bool), np.asarray(strata)
    if grp.all() or not grp.any():
        return np.full(D.shape[1], np.nan), np.full(D.shape[1], np.nan)
    diff = lambda g: D[g].mean(axis=0) - D[~g].mean(axis=0)
    obs = diff(grp)
    rng, hits, g = np.random.default_rng(seed), np.zeros(D.shape[1]), grp.copy()
    blocks = [np.flatnonzero(strata == s) for s in np.unique(strata)]
    for _ in range(B):
        for ix in blocks:
            g[ix] = rng.permutation(grp[ix])
        hits += np.abs(diff(g)) >= np.abs(obs) - 1e-12
    return obs, (hits + 1) / (B + 1)

if not ("gold_doc" in _V4M and _V4M["gold_doc"].notna().all() and _V4M["doc_cov"].notna().all()):
    print("\nẢnh hưởng của độ phủ doc: V4_RESULTS thiếu gold_doc hoặc doc_cov, bỏ qua (v4-run cần tính lại)")
else:
    _cov_pairs = [p for p in V4_COV_PAIRS if p[0] in _V4_SYS and p[1] in _V4_SYS]
    _covC, _c1, _c2 = {}, [], []
    for a, b in _cov_pairs:
        D = _cov_delta(a, b)
        g = D.groupby("Repo")
        R = pd.DataFrame({"Số instance": g.size(), "Độ phủ doc (%)": 100 * g["doc_cov"].mean(),
                          "Có gold có doc (%)": 100 * g["gold_doc"].apply(lambda s: (s > 0).mean())})
        for k in _COV_MET:
            R[f"Δ{k}"] = _COV_PCT[k] * g[k].mean()
        R = R.sort_values("Độ phủ doc (%)")
        print(f"\nBảng C-A. {a} so với {b} theo repo, mức hàm/lớp (Δ của Top@k và Acc@k tính bằng điểm %)")
        display(R.round(3))
        row1 = {"So sánh": f"{a} so với {b}", "Số repo": len(R)}
        for k in _COV_MET:
            rho, p = _spearman_perm(R["Độ phủ doc (%)"], R[f"Δ{k}"]) if len(R) >= 3 else (np.nan, np.nan)
            row1[f"ρ {k}"], row1[f"p {k}"] = rho, p
            if (a, b) == V4_COV_MAIN and not np.isnan(p):
                _covC[("C1", k)] = p
        _c1.append(row1)
        grp = (D["gold_doc"] > 0).values
        obs, p = _strat_perm_diff(D[_COV_MET].values, grp, D["Repo"].values)
        row2 = {"So sánh": f"{a} so với {b}", "Có gold có doc": int(grp.sum()), "Không": int((~grp).sum())}
        for j, k in enumerate(_COV_MET):
            row2[f"Δ có {k}"] = _COV_PCT[k] * D.loc[grp, k].mean() if grp.any() else np.nan
            row2[f"Δ không {k}"] = _COV_PCT[k] * D.loc[~grp, k].mean() if (~grp).any() else np.nan
            row2[f"Hiệu {k}"], row2[f"p {k}"] = _COV_PCT[k] * obs[j], p[j]
            if (a, b) == V4_COV_MAIN and not np.isnan(p[j]):
                _covC[("C2", k)] = p[j]
        _c2.append(row2)
    if _V4M.Repo.nunique() < 3:
        print(f"\nC1 cần ít nhất 3 repo, hiện có {_V4M.Repo.nunique()} repo, nên không tính ρ")
    print("\nC1. ρ của Spearman giữa độ phủ doc của repo và Δ trung bình của repo (p bằng hoán vị nhãn repo)")
    display(pd.DataFrame(_c1).round(4))
    a, b = V4_COV_MAIN
    for flag, name in [(True, "có gold có cạnh Documents"), (False, "không có gold nào có cạnh Documents")]:
        ids = set(_cov_delta(a, b).query("gold_doc > 0" if flag else "gold_doc == 0").index)
        Mg = _V4M[_V4M.Instance.isin(ids) & _V4M["Hệ thống"].isin([a, b])]
        if len(Mg):
            print(f"\nBảng C-B. Nhóm {name} ({len(ids)} instance), mức hàm/lớp")
            display(_v4_table1(Mg, "hàm/lớp").dropna(how="all"))
            display(_v4_tests(Mg, "hàm/lớp", pairs=[(f"{a} so với {b}", a, b)]))
    print("\nC2. Δ trong hai nhóm gold_doc và hiệu giữa hai nhóm (p bằng hoán vị nhãn nhóm trong từng repo)")
    display(pd.DataFrame(_c2).round(4))
    D = _cov_delta(a, b)
    X = D.assign(**{"Gold có doc": D["gold_doc"] > 0}).groupby(["Gold có doc", "nêu tên gold"])
    X = pd.DataFrame({"Số instance": X.size(), **{f"Δ{k}": _COV_PCT[k] * X[k].mean() for k in ("Top@10", "MRR", "MAP")}})
    print(f"\nBảng C-B phụ. {a} so với {b} theo gold có doc và issue nêu tên gold (mô tả)")
    display(X.round(4))
    _dsrc = V4_SRC_LISTS["DOC"]
    _cc = []
    for r in _v4rows:
        if r.get("gold_doc", 0) > 0:
            pos = [x for x in r.get(f"doc_gold_{_dsrc}", []) if x is not None]
            gs = set(r["gold_ids"])
            _cc.append({"Repo": r["Repo"].split("/")[1], "Gold trong top 10 của DOC": int(any(x <= 10 for x in pos)),
                        "Gold trong danh sách DOC": int(bool(pos)),
                        "Chunk của doc_cand nối tới gold": int(any(gs & set(ch["targets"]) for ch in r["doc_cand"]))})
    if _cc:
        C = pd.DataFrame(_cc)
        C = pd.concat([C.groupby("Repo").mean() * 100, C.drop(columns="Repo").mean().to_frame("Gộp mọi repo").T * 100])
        print(f"\nBảng C-C. Nhóm có gold có cạnh Documents ({len(_cc)} instance), tỷ lệ instance (%)")
        display(C.round(1))
    if _covC:
        H = pd.DataFrame([{"Kiểm định": t, **{f"p {k}": _covC.get((t, k), np.nan) for k in _COV_MET}}
                          for t in ("C1", "C2") if any((t, k) in _covC for k in _COV_MET)])
        for k in _COV_MET:
            H[f"Holm {k}"] = _holm(H[f"p {k}"].values)
        print(f"\nHọ C ({a} so với {b}): C1 ở mức repo, C2 ở mức instance; Holm trong từng độ đo")
        display(H.round(4))

# ── Bước I: bằng chứng và judge (mục 6.12 planning.md; 13.2, 13.3 research_context.md) ────────────────────────────────────
# Nhãn đọc từ JUDGE_V4_DIR (judge-v4): pool.jsonl và labels_<JUDGE_MODEL>.jsonl. Nguồn HIST liên kết trực tiếp tới các node
# AST (hàm/lớp) và đã được đánh giá ở Bảng 1, Bảng 2 và depth-v4; bước này chỉ đánh giá 2 nguồn bằng chứng văn bản là DOC
# (ev-doc) và ISSUE (ev-issue). Một phần tử bằng chứng là hữu ích khi điểm hữu ích ≥ 2. Pool của mỗi loại là V4_EVID_N phần
# tử đầu theo điểm gốc của nguồn (doc_cand, issue_cand). Mỗi cách lọc giữ một phần của pool, tối đa 10 phần tử. Độ đo tính
# trên mọi instance có pool (mục 6.18.4 planning.md):
#   Precision, Recall trong pool và F1 dạng micro, từ tổng TP, FP, FN trên mọi instance (Bacchelli, Lanza, Robbes, ICSE
#   2010, mục 3.4). TP = phần tử được giữ và hữu ích, FP = được giữ và không hữu ích, FN = hữu ích nhưng không được giữ.
#   Recall chỉ tính trên phần tử hữu ích có trong pool, nên gọi là recall trong pool.
#   T11U = 2·TP − FP của từng instance, lấy trung bình (Robertson và Soboroff, TREC 2002 Filtering, mục 2.3; hệ số 2:1
#   lấy từ bài). Instance trả rỗng có T11U = 0.
#   Bảng 2×2: output rỗng hay không, pool có phần tử hữu ích hay không (số instance).
#   Token = tổng token ước lượng (ký tự / 4) của phần tử được giữ.
# So sánh giữa các nguồn chỉ là mô tả, vì mỗi nguồn có pool và loại phần tử riêng. So sánh có ý nghĩa chỉ trong cùng một
# nguồn: giữa các cách cắt, hoặc giữa các phiên bản judge.
# Cách lọc hiện có: top-k và hạn ngạch cố định hiện tại (5 chunk doc, 5 báo cáo cũ), là mốc so sánh của mục 13.2.
# Prompt quy định usefulness <= relevance và phần tử chỉ có loại code_location được tối đa 1 điểm hữu ích; Bảng 3 và Bảng 4
# áp lại hai quy tắc này khi đọc nhãn. File nhãn vẫn giữ nhãn thô, và độ ổn định của judge tính trên nhãn thô.
from sklearn.metrics import cohen_kappa_score

V4_EV_QUOTA = {"ev-doc": 5, "ev-issue": 5}
V4_EV_KS    = (1, 3, 5, 10)
V4_EV_SRC   = {"ev-doc": "DOC", "ev-issue": "ISSUE"}
_JD  = globals().get("JUDGE_V4_DIR", Path(WORK_DIR) / "judge_v4")

def _read_judge(d):
    """(pool, nhãn) của một thư mục judge: pool = {khóa: mục}, nhãn = {khóa: {alias: nhãn}} (bản hợp lệ ghi sau cùng)."""
    p, l = Path(d) / "pool.jsonl", Path(d) / f"labels_{JUDGE_MODEL}.jsonl"
    pool = {o["key"]: o for o in map(json.loads, open(p, encoding="utf-8"))} if p.exists() else {}
    lab = {}
    if l.exists():
        for line in open(l, encoding="utf-8"):
            if line.strip():
                o = json.loads(line)
                if o.get("labels"):
                    lab[o["key"]] = o["labels"]
    return pool, lab

def _cand_order(r, kind):
    """Thứ tự của pool theo điểm gốc của nguồn; phần tử định danh như trong judge-v4."""
    if kind == "ev-doc":
        return [x["id"] for x in r.get("doc_cand", [])]
    return [str(x["n"]) for x in r.get("issue_cand", [])]

def _ev_filter(E, keep):
    """Độ đo của một cách lọc trên một loại bằng chứng; keep(phần tử của 1 instance) → mặt nạ phần tử được giữ."""
    out = []
    for _, g in E.groupby("Instance"):
        kept = g[keep(g)]
        U, tp = int(g["useful"].sum()), int(kept["useful"].sum())
        out.append({"tp": tp, "fp": len(kept) - tp, "fn": U - tp, "empty": len(kept) == 0, "has": U > 0,
                    "tok": kept["tok"].sum()})
    D = pd.DataFrame(out)
    TP, FP, FN = int(D["tp"].sum()), int(D["fp"].sum()), int(D["fn"].sum())
    return {"Precision": TP / (TP + FP) if TP + FP else np.nan, "Recall trong pool": TP / (TP + FN) if TP + FN else np.nan,
            "F1": 2 * TP / (2 * TP + FP + FN) if TP + FP + FN else np.nan, "T11U": (2 * D["tp"] - D["fp"]).mean(),
            "Rỗng, pool có hữu ích": int((D["empty"] & D["has"]).sum()),
            "Rỗng, pool không có hữu ích": int((D["empty"] & ~D["has"]).sum()),
            "Khác rỗng, pool có hữu ích": int((~D["empty"] & D["has"]).sum()),
            "Khác rỗng, pool không có hữu ích": int((~D["empty"] & ~D["has"]).sum()),
            "Token TB": D["tok"].mean()}

_pool4, _lab4 = _read_judge(_JD)
_jl = {(o["Instance"], o["kind"]): {it["node"]: {**it, "alias_pos": j, **_lab4[k][it["alias"]]}
                                    for j, it in enumerate(o["items"], 1) if it["alias"] in _lab4[k]}
       for k, o in _pool4.items() if k in _lab4}
_ev = []
for r in _v4rows:
    gs = set(r.get("gold_ids", []))
    ast10 = bool(gs & set(r.get("ast_ids", [])[:10]))
    lk_of = {"ev-doc": {x["id"]: bool(gs & set(x.get("targets", []))) for x in r.get("doc_cand", [])},
             "ev-issue": {str(x["n"]): bool(gs & set(x.get("fix", []))) for x in r.get("issue_cand", [])}}
    for kind in V4_EV_QUOTA:
        lab = _jl.get((r["Instance"], kind))
        if not lab:
            continue
        for pos, node in enumerate(_cand_order(r, kind)[:V4_EVID_N], 1):
            if node in lab:
                L = lab[node]
                u_raw = L["usefulness"]
                u = min(u_raw, L["relevance"])
                if L["info_types"] == ["code_location"]:
                    u = min(u, 1)
                _ev.append({"Instance": r["Instance"], "Repo": r["Repo"].split("/")[1], "kind": kind, "pos": pos,
                            "alias_pos": L.get("alias_pos", pos), "u_raw": u_raw, "usefulness": u,
                            "relevance": L["relevance"], "useful": int(u >= 2), "useful_raw": int(u_raw >= 2),
                            "types": L["info_types"], "tok": L.get("tok", 0), "capped": u < u_raw,
                            "links_gold": lk_of.get(kind, {}).get(node, False),
                            "names_gold": bool(r.get("issue_names_gold")),
                            "gold_doc": bool((r.get("gold_doc") or 0) > 0),
                            "doc_cov": float(r.get("doc_cov") or 0.0), "ast_top10": ast10})
_EV = pd.DataFrame(_ev)
if _EV.empty:
    print(f"\nChưa có nhãn bằng chứng của judge-v4 trong {_JD} → bỏ qua Bảng 3, Bảng 4 và độ ổn định của judge")
else:
    print("Số phần tử bị hạ điểm hữu ích theo quy tắc hậu kiểm (usefulness <= relevance và chỉ code_location <= 1): "
          + ", ".join(f"{V4_EV_SRC[k]} {int(_EV.loc[_EV['kind'] == k, 'capped'].sum())}/{int((_EV['kind'] == k).sum())}"
                      for k in V4_EV_QUOTA if (_EV["kind"] == k).any()))
    for kind in V4_EV_QUOTA:
        E = _EV[_EV["kind"] == kind]
        if E.empty:
            continue
        rows = {f"top-{k}": _ev_filter(E, lambda g, k=k: g["pos"] <= k) for k in V4_EV_KS}
        rows[f"hạn ngạch hiện tại ({V4_EV_QUOTA[kind]})"] = _ev_filter(E, lambda g, q=V4_EV_QUOTA[kind]: g["pos"] <= q)
        dist = E["usefulness"].value_counts(normalize=True).reindex(range(4), fill_value=0) * 100
        print(f"\nBảng 3. Bằng chứng {V4_EV_SRC[kind]}: {E.Instance.nunique()} instance, {len(E)} phần tử | pool có phần tử "
              f"hữu ích ở {100 * E.groupby('Instance')['useful'].max().mean():.1f}% instance | điểm hữu ích 0/1/2/3: "
              + " / ".join(f"{v:.1f}%" for v in dist))
        display(pd.DataFrame(rows).T.round(3))
    print("Bảng 3 so giữa các cách cắt trong cùng một nguồn. So sánh giữa các nguồn chỉ là mô tả.")
    _tt = globals().get("EV_INFO_TYPES") or sorted({t for ts in _EV["types"] for t in ts})
    for name, E in (("phần tử hữu ích", _EV[_EV["useful"] == 1]), ("mọi phần tử", _EV)):
        print(f"\nBảng 4. Tỷ lệ (%) {name} có từng loại thông tin, theo nguồn (một phần tử có thể có nhiều loại)")
        display(pd.DataFrame({V4_EV_SRC[k]: {t: 100 * g["types"].apply(lambda ts, t=t: t in ts).mean() for t in _tt}
                              for k, g in E.groupby("kind")}).T.round(1))
    _tx = [t for t in _tt if t not in globals().get("EV_EXCLUSIVE", {"no_new_information", "unrelated"})]
    print("\nBảng 4. Số phần tử có từng loại thông tin trên mỗi instance, theo nguồn (mọi phần tử, trừ hai nhãn loại trừ)")
    display(pd.DataFrame({V4_EV_SRC[k]: {t: g["types"].apply(lambda ts, t=t: t in ts).sum() / g["Instance"].nunique()
                                         for t in _tx} for k, g in _EV.groupby("kind")}).T.round(2))
    # Độ ổn định: cùng phần tử được chấm hai lần, thứ tự xáo khác (loại <loại>-rep của judge-v4). κ của Cohen (1960) cho nhãn
    # hữu ích nhị phân; κ trọng số bậc hai (Cohen 1968) cho điểm 0..3.
    _rel = []
    for (iid, kind), lab in _jl.items():
        if kind.endswith("-rep") and kind[:-4] in V4_EV_QUOTA and (iid, kind[:-4]) in _jl:
            first = _jl[(iid, kind[:-4])]
            for node, L in lab.items():
                if node in first:
                    A, ta, tb = first[node], set(first[node]["info_types"]), set(L["info_types"])
                    _rel.append({"Instance": iid, "kind": kind[:-4], "u1": A["usefulness"], "u2": L["usefulness"],
                                 "r1": A["relevance"], "r2": L["relevance"], "t1": ta, "t2": tb,
                                 "jac": len(ta & tb) / len(ta | tb)})
    if _rel:
        Rl = pd.DataFrame(_rel)
        t6 = {}
        for kind, g in list(Rl.groupby("kind")) + [("gộp", Rl)]:
            t6[V4_EV_SRC.get(kind, kind)] = {
                "Instance": g["Instance"].nunique(), "Cặp phần tử": len(g),
                "Đồng thuận điểm hữu ích (%)": 100 * (g["u1"] == g["u2"]).mean(),
                "κ nhãn hữu ích (≥ 2)": cohen_kappa_score(g["u1"] >= 2, g["u2"] >= 2),
                "κ trọng số, điểm hữu ích": cohen_kappa_score(g["u1"], g["u2"], weights="quadratic"),
                "κ trọng số, điểm liên quan": cohen_kappa_score(g["r1"], g["r2"], weights="quadratic"),
                "Jaccard loại thông tin": g["jac"].mean()}
        print("\nĐộ ổn định của judge: cùng phần tử chấm hai lần, thứ tự trong prompt xáo khác")
        display(pd.DataFrame(t6).T.round(3))

    # ── Phân tích sâu về kết quả judge (J1 đến J6) ────────────────────────────────────────────────────────────────────────
    # J1. Kiểm tra tính nhất quán nội tại và vi phạm quy tắc prompt trên nhãn thô
    _j1 = {}
    for kind, g in list(_EV.groupby("kind")) + [("gộp", _EV)]:
        _j1[V4_EV_SRC.get(kind, kind)] = {
            "Số phần tử": len(g),
            "u_raw > relevance (%)": 100 * (g["u_raw"] > g["relevance"]).mean(),
            "u_raw = 2, relevance = 1 (%)": 100 * ((g["u_raw"] == 2) & (g["relevance"] == 1)).mean(),
            "u_raw = 3, relevance <= 2 (%)": 100 * ((g["u_raw"] == 3) & (g["relevance"] <= 2)).mean(),
            "Chỉ code_location mà u_raw >= 2": int((g["types"].apply(lambda ts: ts == ["code_location"]) & (g["u_raw"] >= 2)).sum()),
            "no_new_info mà relevance = 0 (%)": 100 * ((g["types"].apply(lambda ts: "no_new_information" in ts)) & (g["relevance"] == 0)).mean(),
            "Tỷ lệ hữu ích thô (u_raw >= 2, %)": 100 * g["useful_raw"].mean(),
            "Tỷ lệ hữu ích sau hậu kiểm (u >= 2, %)": 100 * g["useful"].mean(),
        }
    print("\nPhân tích sâu J1. Kiểm tra tính nhất quán nội tại trên nhãn thô so với sau hậu kiểm")
    display(pd.DataFrame(_j1).T.round(2))

    # J2. Đường cong chất lượng theo hạng gốc (pos 1..10) so với vị trí hiển thị trong prompt (alias_pos 1..10)
    for kind in V4_EV_QUOTA:
        E = _EV[_EV["kind"] == kind]
        if E.empty:
            continue
        by_pos = E.groupby("pos").agg(n=("useful", "size"), u_clean_pct=("useful", lambda s: 100 * s.mean()),
                                      u_raw_pct=("useful_raw", lambda s: 100 * s.mean()), u_mean=("usefulness", "mean"),
                                      rel_mean=("relevance", "mean"), gold_pct=("links_gold", lambda s: 100 * s.mean()))
        by_al = E.groupby("alias_pos").agg(al_u_clean_pct=("useful", lambda s: 100 * s.mean()),
                                           al_u_raw_pct=("useful_raw", lambda s: 100 * s.mean()),
                                           al_gold_pct=("links_gold", lambda s: 100 * s.mean()))
        J2 = by_pos.join(by_al).rename(columns={
            "n": "Số phần tử", "u_clean_pct": "Hạng gốc: u >= 2 (%)", "u_raw_pct": "Hạng gốc: u_raw >= 2 (%)",
            "u_mean": "Hạng gốc: u TB", "rel_mean": "Hạng gốc: rel TB", "gold_pct": "Hạng gốc: nối gold (%)",
            "al_u_clean_pct": "Vị trí prompt: u >= 2 (%)", "al_u_raw_pct": "Vị trí prompt: u_raw >= 2 (%)",
            "al_gold_pct": "Vị trí prompt: nối gold (%)"})
        print(f"\nPhân tích sâu J2 ({V4_EV_SRC[kind]}). Chất lượng theo hạng gốc (pos 1..10) so với vị trí hiển thị trong prompt (alias_pos 1..10)")
        display(J2.round(2))

    # J3. Đối chiếu nhãn Judge với liên kết cấu trúc tới Gold (links_gold) và độ dài phần tử (tok)
    _j3 = []
    for kind in V4_EV_QUOTA:
        E = _EV[_EV["kind"] == kind]
        for flag, lb in ((True, "Có nối tới gold"), (False, "Không nối tới gold")):
            g = E[E["links_gold"] == flag]
            if len(g):
                rho = g["tok"].corr(g["usefulness"], method="spearman")
                _j3.append({"Nguồn": V4_EV_SRC[kind], "Nhóm phần tử": lb, "Số phần tử": len(g),
                            "u = 0 (%)": 100 * (g["usefulness"] == 0).mean(), "u = 1 (%)": 100 * (g["usefulness"] == 1).mean(),
                            "u = 2 (%)": 100 * (g["usefulness"] == 2).mean(), "u = 3 (%)": 100 * (g["usefulness"] == 3).mean(),
                            "Hữu ích (u >= 2, %)": 100 * g["useful"].mean(), "Token TB": g["tok"].mean(),
                            "Spearman ρ(tok, u)": rho})
    print("\nPhân tích sâu J3. Phân bố điểm hữu ích và tương quan độ dài theo trạng thái nối tới gold")
    display(pd.DataFrame(_j3).round(2))

    # J4. Hiệu quả bằng chứng theo từng Repo (sắp theo độ phủ doc) và theo các phân nhóm độ khó
    for kind in V4_EV_QUOTA:
        E = _EV[_EV["kind"] == kind]
        if E.empty:
            continue
        q = V4_EV_QUOTA[kind]
        rp_rows = []
        for rp, g in E.groupby("Repo"):
            g5 = g[g["pos"] <= q]
            rp_rows.append({"Repo": rp, "Số instance": g["Instance"].nunique(), "Độ phủ doc (%)": 100 * g["doc_cov"].mean(),
                            "Pool có u >= 2 (%)": 100 * g.groupby("Instance")["useful"].max().mean(),
                            f"Top-{q} có u >= 2 (%)": 100 * g5.groupby("Instance")["useful"].max().mean(),
                            f"Precision@{q} (%)": 100 * g5["useful"].mean(), "Tỷ lệ nối gold (%)": 100 * g["links_gold"].mean()})
        print(f"\nPhân tích sâu J4a ({V4_EV_SRC[kind]}). Kết quả bằng chứng theo từng repo")
        display(pd.DataFrame(rp_rows).sort_values("Độ phủ doc (%)").set_index("Repo").round(1))
        sg_rows = []
        for col, v, name in (("names_gold", True, "Issue nêu sẵn tên gold"), ("names_gold", False, "Issue không nêu tên gold"),
                             ("gold_doc", True, "Gold có cạnh Documents"), ("gold_doc", False, "Gold không có cạnh Documents"),
                             ("ast_top10", True, "AST: gold trong top 10"), ("ast_top10", False, "AST: gold ngoài top 10")):
            g = E[E[col] == v]
            if len(g):
                g5 = g[g["pos"] <= q]
                sg_rows.append({"Phân nhóm": name, "Số instance": g["Instance"].nunique(),
                                "Pool có u >= 2 (%)": 100 * g.groupby("Instance")["useful"].max().mean(),
                                f"Top-{q} có u >= 2 (%)": 100 * g5.groupby("Instance")["useful"].max().mean(),
                                f"Precision@{q} (%)": 100 * g5["useful"].mean(), "Tỷ lệ nối gold (%)": 100 * g["links_gold"].mean()})
        print(f"\nPhân tích sâu J4b ({V4_EV_SRC[kind]}). Kết quả bằng chứng theo phân nhóm độ khó của instance")
        display(pd.DataFrame(sg_rows).set_index("Phân nhóm").round(1))

    # J5. Tính bổ sung lẫn nhau giữa DOC và ISSUE ở mức instance (DOC ∪ ISSUE)
    if {"ev-doc", "ev-issue"} <= set(_EV["kind"].unique()):
        _j5 = []
        for scope_name, cut_fn in (("Trong toàn bộ pool (top-10)", lambda df: df),
                                   ("Trong hạn ngạch hiện tại (top-5)", lambda df: df[df["pos"] <= 5])):
            sub = cut_fn(_EV)
            piv = sub.groupby(["Instance", "kind"])["useful"].max().unstack(fill_value=0)
            ast_map = _EV.groupby("Instance")["ast_top10"].first().reindex(piv.index)
            for grp_name, idx in (("Mọi instance", piv.index),
                                  ("Nhóm khó (AST trượt top 10)", piv.index[~ast_map]),
                                  ("Nhóm dễ (AST trúng top 10)", piv.index[ast_map])):
                P = piv.loc[idx]
                d_ok, i_ok = P.get("ev-doc", 0) > 0, P.get("ev-issue", 0) > 0
                _j5.append({"Phạm vi": scope_name, "Nhóm": grp_name, "Số instance": len(P),
                            "Chỉ DOC hữu ích (%)": 100 * (d_ok & ~i_ok).mean(),
                            "Chỉ ISSUE hữu ích (%)": 100 * (~d_ok & i_ok).mean(),
                            "Cả hai cùng hữu ích (%)": 100 * (d_ok & i_ok).mean(),
                            "Hợp DOC ∪ ISSUE hữu ích (%)": 100 * (d_ok | i_ok).mean(),
                            "Cả hai đều rỗng (%)": 100 * (~d_ok & ~i_ok).mean()})
        print("\nPhân tích sâu J5. Tính bổ sung lẫn nhau giữa DOC và ISSUE ở mức instance")
        display(pd.DataFrame(_j5).round(1))

    # J6. Độ ổn định chấm lặp (#rep) theo từng loại thông tin (info_types)
    if _rel:
        _j6 = {}
        for kind, g in Rl.groupby("kind"):
            _j6[V4_EV_SRC.get(kind, kind)] = {
                t: cohen_kappa_score(g["t1"].apply(lambda s, t=t: t in s), g["t2"].apply(lambda s, t=t: t in s))
                for t in _tt}
        print("\nPhân tích sâu J6. Hệ số κ của Cohen cho từng loại thông tin giữa hai lần chấm lặp (#rep)")
        display(pd.DataFrame(_j6).T.round(3))

verify-v4: 2214 instance | hệ thống: AST, +DOC, +HIST, +ISSUE, +ALL, ALL−DOC, ALL−HIST, ALL−ISSUE, +GRAPH, +ALL+GRAPH, +HIST (Locus), +HIST (NL), +HIST (NL+CE), +ISSUE (Refs) | câu hỏi có đủ hệ thống: Q1, Q2, Q3, Q4, Q5, Q6, Q7, G1, G2, ALL−ISSUE so với AST

Bảng 1. Gộp mọi repo, mức hàm/lớp (2214 instance)


,Top@1,Top@5,Top@10,Acc@5,Acc@10,MRR,MAP
Hệ thống,,,,,,,
AST,22.4,44.4,54.2,23.1,30.3,0.331,0.246
+DOC,11.4,27.4,39.5,12.2,20.3,0.203,0.148
+HIST,22.4,44.4,54.4,23.4,30.5,0.331,0.250
+ISSUE,16.4,33.2,44.4,15.7,21.4,0.254,0.181
+ALL,18.6,41.2,53.7,20.1,29.0,0.296,0.221
ALL−DOC,19.5,41.1,53.0,20.1,28.8,0.301,0.221
ALL−HIST,14.5,35.1,46.2,16.3,23.9,0.249,0.179
ALL−ISSUE,15.9,40.6,53.9,20.1,30.0,0.278,0.210
+GRAPH,17.2,39.7,53.8,19.8,30.0,0.285,0.221



Bảng 1. Gộp mọi repo, mức file (2214 instance)


,Top@1,Top@5,Top@10,Acc@1,Acc@3,Acc@5,MRR,MAP
Hệ thống,,,,,,,,
AST,47.9,77.4,86.4,36.2,56.9,65.0,0.611,0.568
+DOC,34.5,64.1,83.3,25.5,40.7,50.9,0.485,0.445
+HIST,48.0,78.1,86.9,36.6,56.5,65.2,0.615,0.573
+ISSUE,34.7,66.4,80.0,25.7,43.2,52.5,0.493,0.450
+ALL,41.5,78.0,86.7,31.2,54.4,63.9,0.574,0.531
ALL−DOC,41.2,76.2,86.4,30.6,52.6,62.4,0.565,0.522
ALL−HIST,34.8,70.7,81.7,25.8,48.6,56.4,0.512,0.468
ALL−ISSUE,40.9,78.0,86.9,30.9,54.0,64.2,0.571,0.531
+GRAPH,39.9,76.3,86.9,30.3,51.7,63.2,0.553,0.516



Bảng 1. Riêng astropy, mức hàm/lớp (89 instance)


,Top@1,Top@5,Top@10,Acc@5,Acc@10,MRR,MAP
Hệ thống,,,,,,,
AST,19.1,51.7,57.3,21.3,28.1,0.338,0.235
+DOC,14.6,29.2,43.8,14.6,21.3,0.237,0.165
+HIST,19.1,49.4,62.9,25.8,32.6,0.327,0.242
+ISSUE,6.7,23.6,34.8,7.9,12.4,0.167,0.113
+ALL,11.2,37.1,53.9,13.5,24.7,0.244,0.176
ALL−DOC,13.5,37.1,53.9,14.6,27.0,0.252,0.178
ALL−HIST,12.4,29.2,39.3,11.2,18.0,0.216,0.149
ALL−ISSUE,16.9,43.8,56.2,18.0,31.5,0.294,0.219
+GRAPH,15.7,47.2,57.3,19.1,30.3,0.306,0.210


... (110 outputs left out)


Phân tích sâu J5. Tính bổ sung lẫn nhau giữa DOC và ISSUE ở mức instance


,Phạm vi,Nhóm,Số instance,Chỉ DOC hữu ích (%),Chỉ ISSUE hữu ích (%),Cả hai cùng hữu ích (%),Hợp DOC ∪ ISSUE hữu ích (%),Cả hai đều rỗng (%)
0,Trong toàn bộ pool (top-10),Mọi instance,2214,12.6,31.1,31.7,75.4,24.6
1,Trong toàn bộ pool (top-10),Nhóm khó (AST trượt top 10),1015,13.6,33.2,25.4,72.2,27.8
2,Trong toàn bộ pool (top-10),Nhóm dễ (AST trúng top 10),1199,11.8,29.4,36.9,78.1,21.9
3,Trong hạn ngạch hiện tại (top-5),Mọi instance,2214,15.4,29.0,22.6,66.9,33.1
4,Trong hạn ngạch hiện tại (top-5),Nhóm khó (AST trượt top 10),1015,15.6,30.1,17.1,62.9,37.1
5,Trong hạn ngạch hiện tại (top-5),Nhóm dễ (AST trúng top 10),1199,15.3,27.9,27.2,70.4,29.6



Phân tích sâu J6. Hệ số κ của Cohen cho từng loại thông tin giữa hai lần chấm lặp (#rep)


,expected_behavior,api_usage,similar_failure,bug_cause,code_location,design_rationale,past_change,no_new_information,unrelated
DOC,0.558,0.513,0.000,0.397,0.505,0.172,0.665,0.376,0.651
ISSUE,0.597,0.536,0.381,0.582,0.496,0.480,0.678,0.189,0.588


In [33]:
# ── Section 8 (v4): depth-v4, số node phải đọc và độ sâu tới gold, không dùng LLM ────────────────────────────────────────
# Độ đo theo Renieris và Reiss (ASE 2003), Cleve và Zeller (ICSE 2005): một người đọc lý tưởng bắt đầu từ k node đầu của
# hệ thống, đọc lần lượt k node theo hạng; chưa gặp gold thì đọc mọi hàm/lớp kề với các node đã đọc (lớp 1), rồi lớp 2...
# cho đến lớp có gold. Số node phải đọc = số node của các lớp trước + vị trí kỳ vọng của gold đầu tiên trong lớp cuối,
# (n + 1) / (g + 1) với n node và g gold trong lớp. Gold trong top k ở hạng r: số node phải đọc = r. Không tới được gold:
# số node phải đọc = số hàm/lớp của snapshot. Báo số tuyệt đối, không báo phần trăm (Parnin và Orso, ISSTA 2011).
# Graph: node Function/Class của snapshot; cạnh Calls, Inherits, Defines, đi cả hai chiều. Hub không cần ngưỡng riêng:
# đi qua hub tốn nhiều node phải đọc. Độ sâu của một node = số bước ngắn nhất tới gold gần nhất (tối đa DEPTH_MAX).
# Kiểm định hoán vị ghép cặp trên số node phải đọc của từng instance (_perm_test của verify-v4), Holm trong từng họ của V4_FAMILIES.
# Chạy sau verify-v4 (dùng _v4rows, _V4_SYS, _v4order, _v4pairs, _perm_test, _holm, V4_FAMILIES).
from collections import deque
from itertools import groupby

DEPTH_KS    = (1, 5, 10)
DEPTH_MAX   = 3
DEPTH_EDGES = {"Calls", "Inherits", "Defines"}

def _code_graph(path):
    """Danh sách kề vô hướng giữa các node Function/Class của một graph snapshot."""
    g = read_graph_json(path)
    fc = {n["id"] for n in g["nodes"] if n["label"] in ("Function", "Class")}
    adj = {x: set() for x in fc}
    for e in g["edges"]:
        if e["type"] in DEPTH_EDGES and e["source"] in fc and e["target"] in fc and e["source"] != e["target"]:
            adj[e["source"]].add(e["target"])
            adj[e["target"]].add(e["source"])
    return adj

def _effort(seeds, gold, adj):
    """(số node phải đọc, lớp gặp gold); lớp None khi không tới được gold."""
    for i, s in enumerate(seeds, 1):
        if s in gold:
            return float(i), 0
    seen, frontier, read, layer = set(seeds), list(seeds), float(len(seeds)), 0
    while frontier:
        layer += 1
        nxt = {v for u in frontier for v in adj[u] if v not in seen}
        if not nxt:
            break
        g = len(nxt & gold)
        if g:
            return read + (len(nxt) + 1) / (g + 1), layer
        read += len(nxt)
        seen |= nxt
        frontier = nxt
    return float(len(adj)), None

def _gold_depth(gold, adj):
    """Độ sâu từ gold gần nhất, BFS nhiều nguồn tới DEPTH_MAX bước."""
    dist, q = {g: 0 for g in gold}, deque(gold)
    while q:
        u = q.popleft()
        if dist[u] < DEPTH_MAX:
            for v in adj[u]:
                if v not in dist:
                    dist[v] = dist[u] + 1
                    q.append(v)
    return dist

_dep_eff, _dep_lay, _dep_skip = [], [], 0
_dep_rows = sorted(_v4rows, key=lambda r: (r["Repo"], r["snapshot"]))
for (_repo, _snap8), _grp in groupby(_dep_rows, key=lambda r: (r["Repo"], r["snapshot"][:8])):
    _grp = list(_grp)
    _full = ((globals().get("_inst") or {}).get(_grp[0]["Instance"]) or {}).get("snapshot") or _grp[0]["snapshot"]
    _adj = _code_graph(build_graph(_full, repo_path=repo_path_of(_repo)))
    for r in _grp:
        gold = {f"{_snap8}:{g}" for g in r["gold_ids"]} & _adj.keys()
        if not gold:
            _dep_skip += 1
            continue
        dist = _gold_depth(gold, _adj)
        memo = {}
        for name, fn in _V4_SYS.items():
            order = [x for x in dict.fromkeys(f"{_snap8}:{x}" for x in fn(r)) if x in _adj]
            row = {"Instance": r["Instance"], "Repo": _repo.split("/")[1], "Hệ thống": name}
            for k in DEPTH_KS:
                seeds = tuple(order[:k])
                if seeds not in memo:
                    memo[seeds] = _effort(seeds, gold, _adj)
                row[f"@{k}"], row[f"lớp @{k}"] = memo[seeds]
            _dep_eff.append(row)
            top = order[:10]
            for d in range(DEPTH_MAX + 2):
                _dep_lay.append({"Instance": r["Instance"], "Hệ thống": name, "Độ sâu": d,
                                 "Tỷ lệ": sum((dist.get(x, DEPTH_MAX + 1) == d) for x in top) / max(1, len(top))})
    del _adj

_DE = pd.DataFrame(_dep_eff)
print(f"depth-v4: {_DE.Instance.nunique()} instance | bỏ {_dep_skip} instance không có gold nào trong graph | "
      f"cạnh {', '.join(sorted(DEPTH_EDGES))} | k = {', '.join(map(str, DEPTH_KS))}")
_t = {}
for name, g in _DE.groupby("Hệ thống"):
    _t[name] = {**{f"Trung vị @{k}": g[f"@{k}"].median() for k in DEPTH_KS},
                **{f"Trung bình @{k}": g[f"@{k}"].mean() for k in DEPTH_KS},
                "Không tới được gold @10 (%)": 100 * g["lớp @10"].isna().mean()}
print("\nSố node phải đọc trước khi gặp gold, gộp mọi repo (càng thấp càng tốt)")
display(pd.DataFrame(_t).T.reindex(_v4order).round(1))

_t = []
for q, a, b in _v4pairs:
    A = _DE[_DE["Hệ thống"] == a].set_index("Instance")
    Bm = _DE[_DE["Hệ thống"] == b].set_index("Instance").reindex(A.index)
    row = {"Câu hỏi": q, "So sánh": f"{a} so với {b}", "Số instance": len(A)}
    for k in DEPTH_KS:
        row[f"Δ trung bình @{k}"] = round(A[f"@{k}"].mean() - Bm[f"@{k}"].mean(), 1)
        row[f"p @{k}"] = _perm_test(A[f"@{k}"].values, Bm[f"@{k}"].values)
    _t.append(row)
_t = pd.DataFrame(_t)
for k in DEPTH_KS:
    _t[f"Holm @{k}"] = np.nan
    for _f in V4_FAMILIES:
        _fam = _t["Câu hỏi"].str.match(_f)
        if _fam.any():
            _t.loc[_fam, f"Holm @{k}"] = _holm(_t.loc[_fam, f"p @{k}"].values)
print("\nKiểm định số node phải đọc, gộp mọi repo (Δ âm: hệ thống đầu phải đọc ít hơn)")
display(_t.round(4))

_DL = pd.DataFrame(_dep_lay).groupby(["Hệ thống", "Độ sâu"])["Tỷ lệ"].mean().unstack() * 100
_DL.columns = [str(d) if d <= DEPTH_MAX else f"xa hơn {DEPTH_MAX}" for d in _DL.columns]
print("\nĐộ sâu tới gold của các node trong top 10, tỷ lệ (%) trung bình theo instance")
display(_DL.reindex(_v4order).round(1))

[0e1d299f] disk cache hit → /kaggle/working/astropy_graph_0e1d299f.json
[264d9677] disk cache hit → /kaggle/working/astropy_graph_264d9677.json
[38322105] disk cache hit → /kaggle/working/astropy_graph_38322105.json
[3a0cd2d8] disk cache hit → /kaggle/working/astropy_graph_3a0cd2d8.json
[43ee5806] disk cache hit → /kaggle/working/astropy_graph_43ee5806.json
[4fc9f31a] disk cache hit → /kaggle/working/astropy_graph_4fc9f31a.json
[5aa2d0be] disk cache hit → /kaggle/working/astropy_graph_5aa2d0be.json
[5e5764ed] disk cache hit → /kaggle/working/astropy_graph_5e5764ed.json
[69e2fabd] disk cache hit → /kaggle/working/astropy_graph_69e2fabd.json
[848c8fa2] disk cache hit → /kaggle/working/astropy_graph_848c8fa2.json
[9626265d] disk cache hit → /kaggle/working/astropy_graph_9626265d.json
[a3f4ae6c] disk cache hit → /kaggle/working/astropy_graph_a3f4ae6c.json
[a6c71237] disk cache hit → /kaggle/working/astropy_graph_a6c71237.json
[a7141cd9] disk cache hit → /kaggle/working/astropy_graph_a7141c

,Trung vị @1,Trung vị @5,Trung vị @10,Trung bình @1,Trung bình @5,Trung bình @10,Không tới được gold @10 (%)
AST,67.2,17.0,8.0,1233.1,626.8,519.7,0.3
+DOC,266.0,102.8,45.0,1512.4,1005.4,803.3,0.5
+HIST,77.0,19.0,8.0,1160.6,665.1,529.3,0.5
+ISSUE,272.5,87.0,55.0,1431.2,943.2,779.1,0.8
+ALL,159.0,30.4,8.0,1286.2,692.3,563.9,0.6
ALL−DOC,173.0,32.0,9.0,1292.7,730.4,562.7,0.8
ALL−HIST,289.6,52.2,38.2,1484.1,818.7,716.9,0.6
ALL−ISSUE,159.3,26.0,8.0,1283.3,683.0,542.1,0.5
+GRAPH,141.8,18.5,9.0,1451.7,687.6,516.7,0.3
+ALL+GRAPH,142.2,18.0,7.0,1205.0,638.2,522.1,0.5



Kiểm định số node phải đọc, gộp mọi repo (Δ âm: hệ thống đầu phải đọc ít hơn)


,Câu hỏi,So sánh,Số instance,Δ trung bình @1,p @1,Δ trung bình @5,p @5,Δ trung bình @10,p @10,Holm @1,Holm @5,Holm @10
0,Q1,+DOC so với AST,2214,279.3,0.0001,378.6,0.0001,283.6,0.0001,0.0007,0.0007,0.0007
1,Q2,+HIST so với AST,2214,-72.5,0.2001,38.3,0.2112,9.6,0.7404,0.8003,0.4224,1.0000
2,Q3,+ISSUE so với AST,2214,198.1,0.0038,316.4,0.0001,259.4,0.0001,0.0190,0.0007,0.0007
3,Q4,+ALL so với AST,2214,53.1,0.4284,65.5,0.0649,44.2,0.1623,1.0000,0.2596,0.6491
4,Q5,+ALL so với ALL−DOC,2214,-6.4,0.8859,-38.1,0.1402,1.3,0.9611,1.0000,0.4206,1.0000
5,Q6,+ALL so với ALL−HIST,2214,-197.9,0.0001,-126.4,0.0001,-153.0,0.0001,0.0007,0.0007,0.0007
6,Q7,+ALL so với ALL−ISSUE,2214,2.9,0.9444,9.2,0.7042,21.9,0.3100,1.0000,0.7042,0.9299
7,G1,+GRAPH so với AST,2214,218.6,0.0001,60.8,0.0161,-3.0,0.5476,0.0002,0.0322,0.5476
8,G2,+ALL+GRAPH so với +ALL,2214,-81.2,0.0401,-54.1,0.0202,-41.8,0.0572,0.0401,0.0322,0.1144
9,ALL−ISSUE so với AST,ALL−ISSUE so với AST,2214,50.2,0.4252,56.2,0.0616,22.4,0.4168,NaN,NaN,NaN



Độ sâu tới gold của các node trong top 10, tỷ lệ (%) trung bình theo instance


,0,1,2,3,xa hơn 3
Hệ thống,,,,,
AST,7.8,10.3,15.9,20.3,45.7
+DOC,5.6,8.8,16.8,20.9,47.9
+HIST,8.1,11.0,18.3,20.8,41.8
+ISSUE,5.7,7.5,19.1,24.0,43.9
+ALL,7.7,10.6,20.6,22.1,38.9
ALL−DOC,7.6,10.1,20.2,22.0,40.0
ALL−HIST,6.4,9.0,19.5,22.5,42.5
ALL−ISSUE,7.9,11.1,19.4,20.9,40.8
+GRAPH,7.8,10.0,16.3,19.9,46.0


In [34]:
# ── Section 8 (v4): manifest, ghi lại cấu hình và mã của lần chạy (mục 6.9 planning.md) ─────────────────────────────────
# Ghi V4_DIR/manifest.json: repo, số instance, model và mức thinking của judge, hash của prompt và schema bằng
# chứng, cách trộn, các tham số chính, sha1 của các file dữ liệu, và sha1 của mã từng cell đã chạy trong phiên
# (lấy từ In của IPython). Khóa của một cell là số thứ tự trong In cộng dòng đầu không phải magic. Cần số thứ tự vì có hai
# cell trùng dòng đầu (Doc Extractor và index tìm kiếm cùng bắt đầu bằng "import re"). Run All trong kernel mới cho cùng
# số thứ tự ở mọi lần chạy. Cell chạy nhiều lần thì có nhiều mục. Giá trị của GOOGLE_API_KEY và GITHUB_TOKEN được xóa trước khi
# băm, nên hash không phụ thuộc key. Notebook không được commit vì còn API key, nên hash của cell thay cho commit hash. So
# manifest giữa các lượt để biết code có đổi hay không.
import hashlib
import time
from collections import Counter

_SECRET_RE = re.compile(r"""^(\s*(?:GOOGLE_API_KEY|GITHUB_TOKEN)\s*=\s*)(["']).*?\2""", re.M)

def _sha(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]

def _file_sha(path):
    """sha1 của file, đọc từng khối 1 MB; None khi file không có."""
    p = Path(path)
    if not p.exists():
        return None
    h = hashlib.sha1()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()[:12]

def _cell_name(src):
    """Dòng đầu không phải magic hay lệnh shell (!pip) của cell."""
    return next((l.strip() for l in src.splitlines() if l.strip() and not l.lstrip().startswith(("%", "!"))), "")[:100]

_mf_cells = {}
for _i, _src in enumerate((globals().get("In") or [])[1:], 1):
    if _src.strip():
        _clean = _SECRET_RE.sub(r"\1\2\2", _src)                   # xóa giá trị key trước khi lấy tên và băm
        _mf_cells[f"{_i:03d} {_cell_name(_clean)}"] = _sha(_clean)   # số thứ tự trong In: cell trùng dòng đầu không đè nhau

_mf_files = {"scale_rows": _file_sha(V4_RESULTS), "fusion": _file_sha(V4_DIR / "fusion.json")}
for _repo in SCALE_REPOS:
    _n = _repo.split("/")[1]
    _mf_files[f"{_n} history"] = _file_sha(_hist_path(_repo)) if "_hist_path" in globals() else None
    _mf_files[f"{_n} reports"] = _file_sha(_issues_path(_repo)) if "_issues_path" in globals() else None
    _mf_files[f"{_n} links"] = _file_sha(ISSUE_DIR / f"{_n}_links.jsonl") if "ISSUE_DIR" in globals() else None

_mf_rows = [json.loads(l) for l in open(V4_RESULTS, encoding="utf-8") if l.strip()] if V4_RESULTS.exists() else []
_MANIFEST = {
    "time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "repos": SCALE_REPOS,
    "instances": dict(Counter(r["Repo"] for r in _mf_rows if "skip" not in r)),
    "skipped": dict(Counter(r["Repo"] for r in _mf_rows if "skip" in r)),
    "judge": {"model": JUDGE_MODEL, "thinking": THINKING_LEVEL, "dir": str(JUDGE_V4_DIR),
              "prompt_ev": _sha(JUDGE_PROMPT_EV), "schema_ev": _sha(json.dumps(JUDGE_SCHEMA_EV, sort_keys=True)),
              "rep_n": JUDGE_V4_REP_N, "batch_instances": JUDGE_BATCH_INSTANCES},
    "fusion": {"method": "RRF gốc, trọng số bằng nhau", "k": V4_RRF_K},
    "params": {"SNAPSHOT_BIN_DAYS": SNAPSHOT_BIN_DAYS, "V4_TOPK": V4_TOPK, "V4_EVID_N": V4_EVID_N,
               "ISSUE_CUTOFF": ISSUE_CUTOFF, "HIST_YEARS": HIST_YEARS, "HIST_MAX_TOUCH": HIST_MAX_TOUCH,
               "LOCUS_LAMBDA": LOCUS_LAMBDA, "LOCUS_BETA1": LOCUS_BETA1, "DEPTH_KS": list(DEPTH_KS),
               "DEPTH_EDGES": sorted(DEPTH_EDGES), "DEPTH_MAX": DEPTH_MAX,
               "KGC": json.loads(KGC_CFG) if "KGC_CFG" in globals() else None,
               "HIST_MSG": json.loads(HIST_MSG_CFG) if "HIST_MSG_CFG" in globals() else None},
    "files": _mf_files,
    "cells": _mf_cells,
}
with open(V4_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(_MANIFEST, f, ensure_ascii=False, indent=1)
print(f"manifest: {sum(_MANIFEST['instances'].values())} instance | {len(_mf_cells)} cell đã băm | "
      f"{sum(v is not None for v in _mf_files.values())}/{len(_mf_files)} file → {V4_DIR / 'manifest.json'}")

manifest: 2214 instance | 0 cell đã băm | 38/38 file → /kaggle/working/v4/manifest.json
